# GLM Likelihood + Native Qwen Stack — Close-Call Selection

This notebook is a reproduction of [0.694 | GLM-5.3 Likelihood + Qwen3.8-27B LoRA](https://www.kaggle.com/code/lightningv08/0-694-glm-5-3-likelihood-qwen3-8-27b-lora) by Vladislav Lavrentev (version 47, Apache 2.0), which adapts José Padilla's GLM-5.3 likelihood stage on top of the public weighted LoRA stack. The score in that title is the author's. This copy has no score of its own yet; it is the baseline that later experiments in this series are measured against.

## Experiment in this version

Notebook 23 of this series is the unchanged source. This one changes only which passages the GLM likelihood stage scores.
The stage has a fixed budget of about 130,000 tokens. The source spends it on the passages with the highest GLM probability, and many of those are already predicted as GLM with high confidence, so the likelihood evidence cannot change them.
Here the ranking is the mass of glm-5.3, glm-5.3-flash and claude-fable-5 times one minus the top probability: the budget goes to close calls inside the corner where the stack makes most of its mistakes. claude-fable-5 is included because it is the class most often confused with the two GLMs, and the likelihood classifier scores all 18 classes. The cap on scored passages goes from 20% to 30%; the token budget still binds first.
The likelihood classifier, its weight and the sanity guard are unchanged. Cells changed: the `v1_stage.py` writer (a `--select-mode` option, default = the source's behaviour) and the cell that calls it; both are marked `# ours`.

## What else is different from the source

- The cell right below this one is new. A Kaggle input can be mounted either as `/kaggle/input/<slug>` or as `/kaggle/input/<kind>/<owner>/<slug>`, depending on the notebook it is attached to, and the source mixes both forms. The cell finds which form exists here and rewrites the other form in every later cell before it runs. When all paths already exist as written, it changes nothing.
- Every other cell is the source's, unchanged, including its credits and the sections written by alexchilton.

## Inputs

All inputs are public and are listed in this notebook's metadata: the competition data, 23 datasets, 9 notebook outputs and 6 Qwen model variations. Dataset and notebook versions are the latest ones on 2026-11-14. The run needs GPU T4 x2 with internet off.

## How to run

Saving a version sees the three-row public sample: the notebook checks every input path, writes a format preview and loads no model. Submitting the version runs the full pipeline on the hidden test.


In [ ]:
# ours: make input paths independent of the mount form.
# The same input is mounted as /kaggle/input/<slug> in some notebooks and as
# /kaggle/input/<kind>/<owner>/<slug> in others. For each input, find the form that exists here
# and rewrite the other form in the source of every later cell (including %%writefile bodies).
import os, re

_NTL_INPUTS = [
    ('datasets', 'jamesliebel', 'glm53-iq3xxs-segs-a-mirror'),
    ('datasets', 'jamesliebel', 'glm53-iq3xxs-segs-b-mirror'),
    ('datasets', 'jamesliebel', 'glm53-flash-iq3xxs-segs-mirror'),
    ('datasets', 'dhruvtibarewal', 'nlm-l24-adapters'),
    ('datasets', 'lightningv08', 'ntl-olmo2-native24-final-inference-assets'),
    ('datasets', 'seyitkaangunes', 'ntl-extra-api-passages'),
    ('datasets', 'abdullah1afify', 'ntl-generations'),
    ('datasets', 'dophuchung', 'ntl-proxy-generations'),
    ('datasets', 'zbigniewkirchner', 'ntl-qwen3-8b-lora-claude-head'),
    ('datasets', 'josephadamski91', 'ntl-atlas-qwen3-8b-v1'),
    ('datasets', 'lucasboesen', 'ntl-free-proxy-generations'),
    ('datasets', 'nuridudak', 'ntl-q14-lora'),
    ('datasets', 'nuridudak', 'ntl-q14-lora-v4'),
    ('datasets', 'kirilltym', 'ntl-spectral-style-heads-paired-4404'),
    ('datasets', 'nuridudak', 'ntl-svc-prefit-pool-v4n'),
    ('datasets', 'dhruvtibarewal', 'nlm-enc-v5'),
    ('datasets', 'nuridudak', 'ntl-two-stage-encoders'),
    ('datasets', 'nishantdahal', 'ntl-two-stage-lora'),
    ('datasets', 'nishantdahal', 'ntl-handwriting-blend'),
    ('datasets', 'dhruvtibarewal', 'nlm-l24-adapters-v5'),
    ('datasets', 'prvsiyan', 'prvsiyans-extended-name-that-llm-dataset'),
    ('datasets', 'bbucxi', 'qwen35-9b-safetensors'),
    ('datasets', 'pjmathematician', 'ultrabert-380m-ntl'),
    ('notebooks', 'lucasboesen', 'name-that-llm-glm4-9b-trunc25'),
    ('notebooks', 'lucasboesen', 'name-that-llm-extract-encoders2'),
    ('notebooks', 'lucasboesen', 'name-that-llm-electra-large-weights'),
    ('notebooks', 'lucasboesen', 'name-that-llm-desklib-weights'),
    ('notebooks', 'lucasboesen', 'name-that-llm-train-lora24-s7'),
    ('notebooks', 'lucasboesen', 'name-that-llm-train-tokattn'),
    ('notebooks', 'lucasboesen', 'name-that-llm-train-tokattn2'),
    ('notebooks', 'vnton37nguyn', 'ntl-train-lora-qwen25-instyle'),
    ('notebooks', 'estopadilla', 'ntl-glm-likelihood-train'),
]
_NTL_MAP = {}
_forms = [('/kaggle/input/competitions/name-that-llm', '/kaggle/input/name-that-llm')]
_forms += [(f'/kaggle/input/{kind}/{owner}/{slug}', f'/kaggle/input/{slug}') for kind, owner, slug in _NTL_INPUTS]
for _pair in _forms:
    _present = [p for p in _pair if os.path.isdir(p)]
    if not _present:
        print('INPUT NOT ATTACHED:', _pair[0])
    elif len(_present) == 1:
        _NTL_MAP[_pair[1] if _present[0] == _pair[0] else _pair[0]] = _present[0]

_NTL_RX = re.compile('(?:' + '|'.join(re.escape(k) for k in sorted(_NTL_MAP, key=len, reverse=True)) + r')(?![\w.\-])') if _NTL_MAP else None

def _ntl_remap_inputs(lines):
    if _NTL_RX is None:
        return lines
    return [_NTL_RX.sub(lambda m: _NTL_MAP[m.group(0)], line) for line in lines]

_ip = get_ipython()
_ip.input_transformers_cleanup[:] = [t for t in _ip.input_transformers_cleanup if getattr(t, '__name__', '') != '_ntl_remap_inputs']
_ip.input_transformers_cleanup.append(_ntl_remap_inputs)
print(f'{len(_forms)} inputs checked; {len(_NTL_MAP)} path forms will be rewritten')


# GLM Likelihood + Native Qwen Embeddings

Adapted from [José Padilla’s scored V4](https://www.kaggle.com/code/estopadilla/0-689-real-glm-5-3-likelihood-on-2xt4?scriptVersionId=368783068) under Apache 2.0. The GLM-5.3 and Flash likelihood comparison reranks the highest GLM-mass passages with a fixed work allocation. The parent uses 12% trained Qwen3.8-27B GSQ, 36% native embedding and 0.50 marginal target mix. Likelihood features use the original fitted 160-token protocol. Original ensemble credits are retained below.


---
## Base notebook: description by its author (unchanged)

From here on, the next three sections are alexchilton's own text, kept as published: "we", "our account" and "our
changes" in them refer to alexchilton, not to us. Our additions are the section above, this note, the code cells
marked `# ---- ours, V1` and the final "Ours, V1" section.

# Name That LLM 0.680: All Public + What Failed

> **All public.** Every model, adapter, checkpoint and dataset this notebook reads is public on Kaggle, so anyone can run it and get the same pipeline.

This version runs lightningv08's *[0.680 | ModernBERT + LoRA + Sparse Ensemble](https://www.kaggle.com/code/lightningv08/0-680-modernbert-lora-sparse-ensemble)* (notebook title *Native Embedding LoRA + Multi-Model Ensemble*) as it was public on 2026-11-08, and on our account it scored **0.680** on the public leaderboard (2026-11-08).
Version 19 of this notebook (Areeb Ali's *[0.673 | I'M BETTER !](https://www.kaggle.com/code/areebbb/0-673-i-m-better)*) scored 0.673 when its code ran on our account; version 17 (Areeb Ali's *0.670 | THIS ACTUALLY WORKS*) scored 0.670; version 16 (ozertuu's *0.670+ | Residual Sibling Gating (M4 Apex)*) scored 0.669; versions 14 and 13 scored 0.668.

This is a different pipeline from version 19.
It is lightningv08's public weighted stack (*[0.666 | Qwen3.8-27B LoRA Weighted Ensemble](https://www.kaggle.com/code/lightningv08/0-666-qwen3-8-27b-lora-weighted-ensemble)*, Version 47) with one new expert: CHEN Han's five prompt-fold **Qwen3-Embedding-8B** classifiers from *[0.679](https://www.kaggle.com/code/concyclics/0-679)* (Version 1), at 40% of the geometric mixture.
The author's own description follows below our three sections, unchanged.

**Our changes:**
- the 3-row public-preview shortcut (fixed predictions, no weights loaded) also checks `KAGGLE_IS_COMPETITION_RERUN`, so the scoring rerun never takes it;
- 12 input roots are written as `/kaggle/input/<slug>`, the form this kernel mounts them in (the author's notebook and our private copy use `/kaggle/input/datasets/<owner>/<slug>`; version 20 failed its input check on exactly these 12); only those path strings differ from the scored copy;
- a first code cell asserts that every attached input and every literal `/kaggle/input` path in the notebook exists, so a missing input fails the commit instead of failing the scoring rerun.

**Credits.** lightningv08 (this version: the weighted stack, the sparse-text and complementary heads, the native-expert port), CHEN Han (concyclics: the native multi-depth Qwen3-Embedding-8B LoRA study and its trained fold states), Roman Tamrazov, mercurydust and Areeb Ali (the parent recipe, typography, the Open Pool readouts), Joseph Adamski (ATLAS), dhruvtibarewal (nlm-v27, the L24 adapters), Lucas Boesen, Nishant Dahal, nuridudak, pjmathematician, zbigniewkirchner, kirilltym, crtproduction (the open pool), and the authors of every public model and dataset attached.
The earlier versions of this notebook also build on ozertuu, omerfarukmerey, pablodiegoo, synthreaper, bgunshui, judith007 and avazaku.
All trained weights remain theirs.

- **Commit-only cheap path:** on a save/commit the test file is the 3-row public sample, so the notebook writes the preview and loads no weights (about 20 seconds); the hidden competition rerun always runs the full pipeline. Submit this notebook to get the real run.
- Our own experiments, what did not work, and how noisy a single public score is are in the next two sections.


## What we tried that did not work
Public leaderboard scores of our own submissions, each against the base it changed.
Each one was a single change to a public notebook run verbatim first, so the difference is the change.

| base (public score) | change | result |
|---|---|---|
| nlm-v27 (0.651) | blend in our own five-backbone model | 0.648 |
| nlm-v27 (0.651) | swap the q8v5 member for a Qwen3.5-27B NF4 member (0.642 on its own) | 0.638 |
| nlm-v27 (0.651) | turn the Qwen3.5-9B token-attention member back on | 0.640 |
| our 5-backbone two-stage LoRA blend (nested CV 0.6865 offline) | run on the board, plain heads | 0.559-0.560 |
| same | prompt clusters + centred heads | 0.567 |
| same | plain + 1.5 x centred on clusters ("twin" rule) | 0.588 |
| same | twin rule + Hungarian assignment per cluster | 0.579 |
| lightningv08 LoRA + Adaptive Centering (0.541) | Hungarian assignment inside each test prompt cluster | 0.524 |
| saddamaditya Apex Stylometric Centering (0.542) | the same Hungarian assignment | 0.525 |
| zbigniewkirchner v6 (0.552) | within-Gemini gate (stylegram LinearSVC, extra Gemini passages) | 0.552 |
| obstacledeveloper adaptive centered stylegram (0.556) | extra seed-11 LoRA24 member | 0.553 |
| obstacledeveloper stylegram (0.559) | 4 intra-family gates (claude / gpt / gemini / glm) | 0.559 |
| nishantdahal two-stage LoRA (0.592) | our two-stage Qwen2.5-7B LoRA as a third member | 0.592 |
| our 0.597 | lightningv08 OLMo2 native24 member, log-prob weight 0.03 | 0.594 |
| our 0.597 | swap lora24 seed 7 for an in-style seed-11 LoRA24 | 0.597 |
| version 8, omerfarukmerey 0.663 (0.662) | judith007's typography evidence 1.5 -> 1.2 and `--sinkhorn` 0.45 -> 0.60 (their 0.664 notebook, run verbatim) | 0.659, 0.659 |
| version 8, omerfarukmerey 0.663 (0.662) | SVC pool -> the 198k open pool, `--svc_w 12` (zbigniewkirchner's 0.664 notebook, run verbatim) | no score: failed in the scoring run |
| zbigniewkirchner One Swap (0.664) | kirilltym's LoRA + Trained27 independent votes (their 0.664 notebook, run verbatim) | no score: failed in the scoring run |

What we take from it:
- Our own blend scored 0.6865 in nested CV but 0.56 on the board.
  The offline folds used true 18-passage prompt groups, so centring and Hungarian assignment looked strong offline.
  On the board, the assignment tricks lost every time (-0.009 to -0.017), which points to few passages per prompt in the hidden test.
- Adding more LoRA backbones or members to a strong ensemble moved the score by at most 0.004 in either direction.
- The two changes that did help were small: a 4th LoRA member on v6 (0.552 -> 0.556) and the prvsiyan 0.569 blend as one extra log-prob member on nishantdahal (0.592 -> 0.597).


## How noisy is one public score?

Before reading any table of gains, including ours above: the same notebook, submitted again, does not score the same.
These are our own submissions of code we did not change, against each other and against the score on the author's page.

| pipeline | identical code, our submissions | author's page |
|---|---|---|
| dhruvtibarewal nlm-v27 | 0.651, 0.651, 0.643, 0.651 | 0.651 |
| Roman Tamrazov *Who am I?* | 0.645, 0.654 | 0.654 |
| mercurydust V31 | 0.660, 0.660 | 0.660 |
| omerfarukmerey 0.663 (version 8) | 0.662 | 0.663 |
| judith007 0.664 | 0.659, 0.659 | 0.664 |
| bgunshui NTL Apex E (version 9) | 0.663, 0.663 | 0.663 |
| synthreaper Sovereign Gold Apex (version 11) | 0.666, 0.666 | 0.665 |
| Areeb Ali NTL We Back | 0.666 | 0.666 |
| zbigniewkirchner Pairwise Verifier + Matched External Rows (version 12) | 0.667, 0.663 | 0.667 |
| omerfarukmerey Verifier + Matched Rows + Pool Prompts | 0.661 | 0.668 |
| pablodiegoo Starter Notebook (version 13) | 0.668, 0.668 | 0.668 |
| ozertuu Dual-SVC Blend + Residual Refinement V5 (version 14) | 0.668 | 0.672 |
| ozertuu Residual Sibling Gating M4 Apex (version 16) | 0.669 | 0.673 |
| ozertuu Dual-SVC BETTER! | 0.669 | 0.672 |
| Areeb Ali 0.672 Stack + Public DeBERTa-v3 Member (version 17) | 0.670 | 0.670 |
| Areeb Ali 0.672 + DeBERTa-v3 + Our ModernBERT (version 19) | 0.673, 0.673 | 0.673 |
| lightningv08 ModernBERT + LoRA + Sparse Ensemble (this notebook) | 0.680 | 0.680 |
| Lucas Boesen LoRA x Adaptive Centering | 0.549 | 0.547 |
| vnton37nguyn lora-instyle | 0.561 | 0.563 |
| josephadamski91 Teach the Model + ATLAS | 0.606 | 0.607 |

The spread is up to **0.009** on identical code (0.645 vs 0.654, 0.643 vs 0.651).
So a single submission that gains less than about 0.01 is not evidence that the change helped; most single-change results in the table above are inside that band.
A related trick from the forum (GoPain, in a comment on discussion 753177): scoring out-of-fold on 2 answers per prompt instead of all 18 tracked the board far better (correlation 0.91 against 0.13 over their 15 submissions).


In [ ]:
import matplotlib.pyplot as plt
runs = {'nlm-v27': [0.651, 0.651, 0.643, 0.651], 'Who am I?': [0.645, 0.654], 'V31': [0.660, 0.660], '0.663 (this)': [0.662],
        'LoRA x AC': [0.549], 'lora-instyle': [0.561], 'Teach + ATLAS': [0.606]}
page = {'nlm-v27': 0.651, 'Who am I?': 0.654, 'V31': 0.660, '0.663 (this)': 0.663, 'LoRA x AC': 0.547, 'lora-instyle': 0.563, 'Teach + ATLAS': 0.607}
fig, ax = plt.subplots(figsize=(8, 3.2))
for i, (k, v) in enumerate(runs.items()):
    ax.scatter([i] * len(v), [x - page[k] for x in v], s=60, color='tab:blue', label='our submission, identical code' if i == 0 else None, zorder=3)
ax.axhspan(-0.01, 0.01, color='tab:orange', alpha=0.15, label='+-0.01: one submission cannot tell these apart')
ax.axhline(0, color='k', lw=0.8)
ax.set_xticks(range(len(runs))); ax.set_xticklabels(runs, rotation=20)
ax.set_ylabel('score - author page score'); ax.set_title('Same code, different public score')
ax.set_ylim(-0.013, 0.013); ax.legend(fontsize=8, loc='upper right'); plt.tight_layout(); plt.show()


# Native Embedding LoRA + Multi-Model Ensemble

This ensemble adds five prompt-fold Qwen3-Embedding-8B classifiers to the multi-model, sparse-text and complementary attribution stack. The native expert uses all 36 transformer layers, learned normalisation at eight depths, and separate plain and prompt-centred heads. Its probability mass in the geometric mixture is **40%**. The Qwen3.8-27B GSQ3 task classifier contributes **3%** inside the remaining parent branch.

The native task weights, original fold calibration and model design are from [CHEN Han's .679 notebook, Version 1](https://www.kaggle.com/code/concyclics/0-679?scriptVersionId=365653660). The parent is the scored weighted ensemble [Version 47](https://www.kaggle.com/code/lightningv08/0-666-qwen3-8-27b-lora-weighted-ensemble?scriptVersionId=365000313). The source ensemble's private legacy models are not substituted as if identical: this notebook combines the recovered native expert with the public weighted stack described below.

The native backbone is divided across both T4 GPUs. Its original BF16 values are used in FP16, while the trained rank-16 LoRA, normalisation and classifier weights stay FP32. Long passages use covering 512-token windows with repeated task instructions and token-weighted document logits; no passage tail is discarded. Both heads use the author's fixed fold-specific scales. Centring uses this notebook's lexical prompt groups, capped at 18 passages, rather than unavailable private grouping models.

The parent retains its pre-fitted Open Pool text heads, V5 and two-stage frozen readouts, complementary stack and selective 10% ATLAS mixture. The native branch enters before marginal balancing and typography. Each fold is corrected once, then its probabilities are averaged. Class names and passage IDs are aligned explicitly at every model boundary.

All inference code is visible below. The shared artifact dataset contains model weights, calibration data and numeric states. Training and fixed train-feature extraction are performed beforehand.

Credits: CHEN Han for the native multi-depth LoRA study and trained states; Roman Tamrazov, mercurydust and Areeb Ali for the public parent recipe, typography and Open Pool readouts; Joseph Adamski for ATLAS; the model and adapter authors named in the corresponding modules.


## Exact public sample preview

The exact three public sample records produce a CPU-only format preview. Their IDs, byte lengths and text fingerprints must all match. Every other test input runs the full model pipeline.


In [ ]:
# ours: every input root and every literal /kaggle/input path in this notebook must exist.
# The commit takes the 3-row preview path and loads no weights, so this is the only commit-time check of the rerun's inputs.
import os
_ROOTS = [
 "/kaggle/input/competitions/name-that-llm/test.csv",
 "/kaggle/input/competitions/name-that-llm/train.csv",
 "/kaggle/input/nlm-l24-adapters",
 "/kaggle/input/datasets/lightningv08/ntl-olmo2-native24-final-inference-assets",
 "/kaggle/input/datasets/seyitkaangunes/ntl-extra-api-passages",
 "/kaggle/input/datasets/abdullah1afify/ntl-generations",
 "/kaggle/input/datasets/dophuchung/ntl-proxy-generations",
 "/kaggle/input/datasets/zbigniewkirchner/ntl-qwen3-8b-lora-claude-head",
 "/kaggle/input/ntl-atlas-qwen3-8b-v1",
 "/kaggle/input/datasets/lucasboesen/ntl-free-proxy-generations",
 "/kaggle/input/ntl-q14-lora",
 "/kaggle/input/ntl-q14-lora-v4",
 "/kaggle/input/ntl-spectral-style-heads-paired-4404",
 "/kaggle/input/ntl-svc-prefit-pool-v4n",
 "/kaggle/input/nlm-enc-v5",
 "/kaggle/input/ntl-two-stage-encoders",
 "/kaggle/input/datasets/nishantdahal/ntl-two-stage-lora",
 "/kaggle/input/ntl-handwriting-blend",
 "/kaggle/input/nlm-l24-adapters-v5",
 "/kaggle/input/datasets/prvsiyan/prvsiyans-extended-name-that-llm-dataset",
 "/kaggle/input/datasets/bbucxi/qwen35-9b-safetensors",
 "/kaggle/input/datasets/pjmathematician/ultrabert-380m-ntl",
 "/kaggle/input/notebooks/lucasboesen/name-that-llm-glm4-9b-trunc25",
 "/kaggle/input/notebooks/lucasboesen/name-that-llm-extract-encoders2",
 "/kaggle/input/notebooks/lucasboesen/name-that-llm-electra-large-weights",
 "/kaggle/input/notebooks/lucasboesen/name-that-llm-desklib-weights",
 "/kaggle/input/notebooks/lucasboesen/name-that-llm-train-lora24-s7",
 "/kaggle/input/notebooks/lucasboesen/name-that-llm-train-tokattn",
 "/kaggle/input/notebooks/lucasboesen/name-that-llm-train-tokattn2",
 "/kaggle/input/notebooks/vnton37nguyn/ntl-train-lora-qwen25-instyle",
 "/kaggle/input/models/qwen-lm/qwen-3",
 "/kaggle/input/models/qwen-lm/qwen-3-embedding",
 "/kaggle/input/models/qwen-lm/qwen2.5"
]
_PATHS = [
 "/kaggle/input/datasets/abdullah1afify/ntl-generations/ntl_generations.csv",
 "/kaggle/input/datasets/bbucxi/qwen35-9b-safetensors",
 "/kaggle/input/nlm-enc-v5/encoder/config.json",
 "/kaggle/input/nlm-enc-v5/encoder/model.safetensors",
 "/kaggle/input/nlm-enc-v5/encoder/tokenizer.json",
 "/kaggle/input/nlm-enc-v5/encoder/tokenizer_config.json",
 "/kaggle/input/nlm-enc-v5/head.safetensors",
 "/kaggle/input/nlm-enc-v5/meta.json",
 "/kaggle/input/nlm-l24-adapters",
 "/kaggle/input/nlm-l24-adapters-v5",
 "/kaggle/input/nlm-l24-adapters-v5/ll8v5",
 "/kaggle/input/nlm-l24-adapters-v5/ll8v5_f0",
 "/kaggle/input/nlm-l24-adapters-v5/ll8v5_f1",
 "/kaggle/input/nlm-l24-adapters-v5/ll8v5_f2",
 "/kaggle/input/nlm-l24-adapters-v5/ll8v5_f3",
 "/kaggle/input/nlm-l24-adapters-v5/ll8v5_f4",
 "/kaggle/input/nlm-l24-adapters-v5/q14v5",
 "/kaggle/input/nlm-l24-adapters-v5/q14v5_f0",
 "/kaggle/input/nlm-l24-adapters-v5/q14v5_f1",
 "/kaggle/input/nlm-l24-adapters-v5/q14v5_f2",
 "/kaggle/input/nlm-l24-adapters-v5/q14v5_f3",
 "/kaggle/input/nlm-l24-adapters-v5/q14v5_f4",
 "/kaggle/input/nlm-l24-adapters-v5/q8v5",
 "/kaggle/input/nlm-l24-adapters-v5/q8v5_f0",
 "/kaggle/input/nlm-l24-adapters-v5/q8v5_f1",
 "/kaggle/input/nlm-l24-adapters-v5/q8v5_f2",
 "/kaggle/input/nlm-l24-adapters-v5/q8v5_f3",
 "/kaggle/input/nlm-l24-adapters-v5/q8v5_f4",
 "/kaggle/input/datasets/dophuchung/ntl-proxy-generations/ntl_proxy_generations.csv",
 "/kaggle/input/ntl-atlas-qwen3-8b-v1/atlas_adapter/atlas_adapter",
 "/kaggle/input/ntl-atlas-qwen3-8b-v1/atlas_adapter/atlas_adapter/atlas_bundle.json",
 "/kaggle/input/datasets/lightningv08/ntl-olmo2-native24-final-inference-assets",
 "/kaggle/input/datasets/lightningv08/ntl-olmo2-native24-final-inference-assets/gsq120k-trained/gsq120k-inference/gaussian/q14v5",
 "/kaggle/input/datasets/lightningv08/ntl-olmo2-native24-final-inference-assets/gsq120k-trained/gsq120k-inference/gaussian/q14v5/metadata.json",
 "/kaggle/input/datasets/lightningv08/ntl-olmo2-native24-final-inference-assets/gsq120k-trained/gsq120k-inference/gaussian/q14v5/parameters.npz",
 "/kaggle/input/datasets/lightningv08/ntl-olmo2-native24-final-inference-assets/gsq120k-trained/gsq120k-inference/gaussian/q8v5",
 "/kaggle/input/datasets/lightningv08/ntl-olmo2-native24-final-inference-assets/gsq120k-trained/gsq120k-inference/gaussian/q8v5/metadata.json",
 "/kaggle/input/datasets/lightningv08/ntl-olmo2-native24-final-inference-assets/gsq120k-trained/gsq120k-inference/gaussian/q8v5/parameters.npz",
 "/kaggle/input/datasets/lightningv08/ntl-olmo2-native24-final-inference-assets/gsq120k-trained/gsq120k-inference/gsq/COMPLETE.json",
 "/kaggle/input/datasets/lightningv08/ntl-olmo2-native24-final-inference-assets/gsq120k-trained/gsq120k-inference/gsq/final.json",
 "/kaggle/input/datasets/lightningv08/ntl-olmo2-native24-final-inference-assets/gsq120k-trained/gsq120k-inference/gsq/final.safetensors",
 "/kaggle/input/datasets/lightningv08/ntl-olmo2-native24-final-inference-assets/gsq120k-trained/gsq120k-inference/svc4",
 "/kaggle/input/datasets/lightningv08/ntl-olmo2-native24-final-inference-assets/gsq120k-trained/gsq120k-inference/svc4/linear_svc.joblib",
 "/kaggle/input/datasets/lightningv08/ntl-olmo2-native24-final-inference-assets/gsq120k-trained/gsq120k-inference/svc4/metadata.json",
 "/kaggle/input/datasets/lightningv08/ntl-olmo2-native24-final-inference-assets/gsq120k-trained/gsq120k-inference/svc4/vectorizer-char.joblib",
 "/kaggle/input/datasets/lightningv08/ntl-olmo2-native24-final-inference-assets/gsq120k-trained/gsq120k-inference/svc4/vectorizer-function.joblib",
 "/kaggle/input/datasets/lightningv08/ntl-olmo2-native24-final-inference-assets/gsq120k-trained/gsq120k-inference/svc4/vectorizer-shape.joblib",
 "/kaggle/input/datasets/lightningv08/ntl-olmo2-native24-final-inference-assets/gsq120k-trained/gsq120k-inference/svc4/vectorizer-word.joblib",
 "/kaggle/input/datasets/lightningv08/ntl-olmo2-native24-final-inference-assets/gsq3-base/qwen38-27b-gsq3",
 "/kaggle/input/datasets/lightningv08/ntl-olmo2-native24-final-inference-assets/llama31-prefix24",
 "/kaggle/input/datasets/lightningv08/ntl-olmo2-native24-final-inference-assets/name-that-llm-artifacts",
 "/kaggle/input/datasets/lightningv08/ntl-olmo2-native24-final-inference-assets/name-that-llm-artifacts/QWEN38_ADDITION_MANIFEST.json",
 "/kaggle/input/datasets/lightningv08/ntl-olmo2-native24-final-inference-assets/name-that-llm-artifacts/ntl-olmo2-native24-final-inference-assets/asset_manifest.json",
 "/kaggle/input/datasets/lightningv08/ntl-olmo2-native24-final-inference-assets/name-that-llm-artifacts/ntl-smollm3-two-stage-final-inference-assets/final_inference_manifest.json",
 "/kaggle/input/datasets/lightningv08/ntl-olmo2-native24-final-inference-assets/name-that-llm-artifacts/qwen38-27b-lora-tail8/final.safetensors",
 "/kaggle/input/datasets/lightningv08/ntl-olmo2-native24-final-inference-assets/name-that-llm-artifacts/qwen38-offline-linux/accelerate-1.15.0-py3-none-any.whl",
 "/kaggle/input/datasets/lightningv08/ntl-olmo2-native24-final-inference-assets/name-that-llm-artifacts/qwen38-offline-linux/bitsandbytes-0.50.2-py3-none-manylinux_2_24_x86_64.whl",
 "/kaggle/input/datasets/lightningv08/ntl-olmo2-native24-final-inference-assets/name-that-llm-artifacts/qwen38-offline-linux/huggingface_hub-1.32.0-py3-none-any.whl",
 "/kaggle/input/datasets/lightningv08/ntl-olmo2-native24-final-inference-assets/name-that-llm-artifacts/qwen38-offline-linux/safetensors-0.8.0-cp310-abi3-manylinux_2_17_x86_64.manylinux2014_x86_64.whl",
 "/kaggle/input/datasets/lightningv08/ntl-olmo2-native24-final-inference-assets/name-that-llm-artifacts/qwen38-offline-linux/tokenizers-0.22.2-cp39-abi3-manylinux_2_17_x86_64.manylinux2014_x86_64.whl",
 "/kaggle/input/datasets/lightningv08/ntl-olmo2-native24-final-inference-assets/name-that-llm-artifacts/qwen38-offline-linux/transformers-5.14.1-py3-none-any.whl",
 "/kaggle/input/datasets/lightningv08/ntl-olmo2-native24-final-inference-assets/notebook-inputs/notebook-inputs/ntl-lora24-fivefold-training/lora24/final",
 "/kaggle/input/datasets/lightningv08/ntl-olmo2-native24-final-inference-assets/notebook-inputs/notebook-inputs/olmo2-7b-fp16-offline-weights/olmo2-fp16-base",
 "/kaggle/input/datasets/lightningv08/ntl-olmo2-native24-final-inference-assets/notebook-inputs/notebook-inputs/smollm3-3b-base-offline-weights/smollm3-base",
 "/kaggle/input/datasets/lightningv08/ntl-olmo2-native24-final-inference-assets/notebook-inputs/notebook-inputs/smollm3-3b-base-offline-weights/smollm3_staging_complete.json",
 "/kaggle/input/datasets/lightningv08/ntl-olmo2-native24-final-inference-assets/ntl-native-q8-multidepth-fivefold-0679",
 "/kaggle/input/datasets/lightningv08/ntl-olmo2-native24-final-inference-assets/ntl611-prefit",
 "/kaggle/input/datasets/lightningv08/ntl-olmo2-native24-final-inference-assets/ntl636-updated-style",
 "/kaggle/input/datasets/lightningv08/ntl-olmo2-native24-final-inference-assets/qwen38-mixed-tail8/name-that-llm-artifacts/qwen38-27b-mixed-tail8-text",
 "/kaggle/input/datasets/lightningv08/ntl-olmo2-native24-final-inference-assets/qwen38-mixed-tail8/name-that-llm-artifacts/qwen38-27b-mixed-tail8-text/manifest.json",
 "/kaggle/input/datasets/lucasboesen/ntl-free-proxy-generations/gen_passages.csv",
 "/kaggle/input/ntl-handwriting-blend",
 "/kaggle/input/ntl-handwriting-blend/deberta_v3_large",
 "/kaggle/input/ntl-handwriting-blend/lora_qwen25_14b",
 "/kaggle/input/ntl-handwriting-blend/lora_qwen3_14b",
 "/kaggle/input/datasets/nishantdahal/ntl-two-stage-lora/lora14b_v1",
 "/kaggle/input/datasets/nishantdahal/ntl-two-stage-lora/lora8b_v1",
 "/kaggle/input/datasets/nishantdahal/ntl-two-stage-lora/pool_v1.parquet",
 "/kaggle/input/datasets/pjmathematician/ultrabert-380m-ntl",
 "/kaggle/input/datasets/pjmathematician/ultrabert-380m-ntl/fold0",
 "/kaggle/input/datasets/pjmathematician/ultrabert-380m-ntl/fold1",
 "/kaggle/input/datasets/pjmathematician/ultrabert-380m-ntl/fold2",
 "/kaggle/input/datasets/pjmathematician/ultrabert-380m-ntl/fold3",
 "/kaggle/input/datasets/pjmathematician/ultrabert-380m-ntl/fold4",
 "/kaggle/input/datasets/prvsiyan/prvsiyans-extended-name-that-llm-dataset/passages_clean.csv",
 "/kaggle/input/datasets/seyitkaangunes/ntl-extra-api-passages/ntl_extra_passages_claude.csv",
 "/kaggle/input/datasets/zbigniewkirchner/ntl-qwen3-8b-lora-claude-head/final",
 "/kaggle/input/models/qwen-lm/qwen-3-embedding/transformers/8b/1",
 "/kaggle/input/models/qwen-lm/qwen-3/transformers/14b-base/1",
 "/kaggle/input/models/qwen-lm/qwen-3/transformers/4b-base/1",
 "/kaggle/input/models/qwen-lm/qwen-3/transformers/8b-base/1",
 "/kaggle/input/models/qwen-lm/qwen2.5/transformers/14b/1",
 "/kaggle/input/models/qwen-lm/qwen2.5/transformers/7b/1",
 "/kaggle/input/notebooks/lucasboesen/name-that-llm-desklib-weights/desklib-detector",
 "/kaggle/input/notebooks/lucasboesen/name-that-llm-electra-large-weights/electra-large-discriminator",
 "/kaggle/input/notebooks/lucasboesen/name-that-llm-extract-encoders2/deberta-v3-large",
 "/kaggle/input/notebooks/lucasboesen/name-that-llm-glm4-9b-trunc25/glm4_9b_trunc25",
 "/kaggle/input/notebooks/lucasboesen/name-that-llm-train-lora24-s7/lora_out/job_final",
 "/kaggle/input/notebooks/lucasboesen/name-that-llm-train-tokattn/tokattn_final.pt",
 "/kaggle/input/notebooks/lucasboesen/name-that-llm-train-tokattn2/tokattn_debl23_final.pt",
 "/kaggle/input/notebooks/vnton37nguyn/ntl-train-lora-qwen25-instyle/lora_out/job_final"
]
_missing = [r for r in _ROOTS + _PATHS if not os.path.exists(r)]
print('input roots checked:', len(_ROOTS), 'literal paths checked:', len(_PATHS), 'missing:', _missing)
assert not _missing, _missing

In [ ]:
import csv, json, os, sys, time
from hashlib import blake2s
from pathlib import Path
import pandas as pd

os.environ['NTL_STARTED_MONOTONIC'] = str(time.monotonic())
DATA = Path('/kaggle/input/competitions/name-that-llm/test.csv')
OUTPUT = Path('/kaggle/working/submission.parquet')
PUBLIC_FIXTURE = {'records': [{'id': '00353fb1c541', 'text_bytes': 321, 'fingerprint': '0c306cbd5d401399327e4cb46f766928'}, {'id': '005f5a9a6139', 'text_bytes': 764, 'fingerprint': 'c2853bf2b68bdd808a8a05cec2439b5d'}, {'id': '0062a53d5487', 'text_bytes': 1077, 'fingerprint': 'f8c583ac9cbc0806bee20f96c9124159'}], 'preview_predictions': ['claude-fable-5', 'claude-fable-5', 'claude-fable-5']}

def emit_known_preview(test_csv, fixture_path, output):
    with Path(test_csv).open(encoding='utf-8-sig', newline='') as stream:
        reader = csv.DictReader(stream)
        if reader.fieldnames != ['id', 'text']:
            return False
        rows = []
        for row in reader:
            rows.append(row)
            if len(rows) > 3:
                return False
    fixture = PUBLIC_FIXTURE
    identities = [{'id': r['id'], 'text_bytes': len(r['text'].encode('utf-8')),
                   'fingerprint': blake2s(r['text'].encode('utf-8'), digest_size=16).hexdigest()}
                  for r in rows]
    if identities != fixture['records']:
        return False
    import pandas as pd
    frame = pd.DataFrame({'id': [r['id'] for r in rows], 'model': fixture['preview_predictions']})
    assert len(frame) == 3 and frame.id.is_unique and frame.model.notna().all()
    frame.to_parquet(output, index=False)
    Path(output).with_suffix('.mode.json').write_text(json.dumps({
        'mode': 'exact_public_format_preview', 'rows': 3,
        'model_weights_loaded': False, 'hidden_runtime_verified': False,
        'all_other_inputs': 'full ensemble inference'}), encoding='utf-8')
    return True

assert DATA.is_file()
preview = os.getenv('KAGGLE_IS_COMPETITION_RERUN') is None and emit_known_preview(DATA,None,OUTPUT)   # ours: the scoring rerun never takes the preview path
RUNTIME = Path('/kaggle/temp/ntl-gsq-v27-runtime')
for relative in ['', 'legacy', 'legacy/inline_support/ntl_atlas', 'legacy/inline_support/ntl_rlcd', 'legacy/inline_support/ultra', 'gsq']:
    (RUNTIME/relative).mkdir(parents=True,exist_ok=True)


In [ ]:
# Inference uses the original competition test input.


In [ ]:
import json
Path('/kaggle/temp/v1').mkdir(parents=True, exist_ok=True)
Path('/kaggle/temp/v1/ll_model.json').write_text(json.dumps({'format': 'ntl-v1-ll-model-1', 'features': ['flash:n_tok', 'flash:lp_mean', 'flash:lp_std', 'flash:lp_q05', 'flash:lp_q10', 'flash:lp_q25', 'flash:lp_q50', 'flash:lp_q75', 'flash:lp_q90', 'flash:lp_min', 'flash:ent_mean', 'flash:ent_std', 'flash:fdg', 'flash:surprise_mean', 'flash:z_mean', 'flash:logrank_mean', 'flash:top1', 'flash:top5', 'flash:top20', 'flash:rank_ge100', 'flash:lp_mean_first', 'flash:lp_mean_second', 'flash:bpc', 'flash:tok_per_char', 'glm53:n_tok', 'glm53:lp_mean', 'glm53:lp_std', 'glm53:lp_q05', 'glm53:lp_q10', 'glm53:lp_q25', 'glm53:lp_q50', 'glm53:lp_q75', 'glm53:lp_q90', 'glm53:lp_min', 'glm53:ent_mean', 'glm53:ent_std', 'glm53:fdg', 'glm53:surprise_mean', 'glm53:z_mean', 'glm53:logrank_mean', 'glm53:top1', 'glm53:top5', 'glm53:top20', 'glm53:rank_ge100', 'glm53:lp_mean_first', 'glm53:lp_mean_second', 'glm53:bpc', 'glm53:tok_per_char', 'd:lp_mean', 'd:lp_q50', 'd:ent_mean', 'd:top1', 'd:logrank_mean', 'd:fdg'], 'diff_features': ['lp_mean', 'lp_q50', 'ent_mean', 'top1', 'logrank_mean', 'fdg'], 'mean': [121.38532789922193, -2.7803846505372083, 3.097825097087325, -9.037813317417966, -7.333253962996272, -4.376777090907936, -1.6158449170146885, -0.30363898588225063, -0.03409017864621625, -13.478645561447228, 1.6547750254399232, 1.241843176512571, -6.6471115795172695, 1.1256096250972856, -0.6509554042356471, 1.314347139907373, 0.45188089629737493, 0.7099387436903867, 0.8417969355641189, 0.06987658387522303, -2.9887337676830588, -2.573760688804061, 0.7122438537480235, 0.17556811209707204, 121.38532789922193, -2.812101332697135, 3.0687895181142895, -8.975851290090828, -7.287101179849398, -4.418885495672406, -1.7056639001032916, -0.3376782801735176, -0.0392805059912824, -13.361867175008598, 1.8394780051200452, 1.3843021665605892, -5.501231789836153, 0.9726233275770891, -0.5569220074723701, 1.3681972075472397, 0.4471236599702501, 0.7007655834496941, 0.8339825162661187, 0.07560197108786391, -3.026507322164044, -2.5995706754525263, 0.7196566479854759, 0.17556811209707204, 0.03171668215992607, 0.08981898308860324, -0.18470297968012234, 0.004757236327124889, -0.05385006763986662, -1.145879789681117], 'scale': [38.592979005558206, 0.5988023434785398, 0.3880666202313587, 1.2651451190799916, 1.245037603437369, 1.1240408839035936, 0.6848026605381858, 0.25756256241253866, 0.06412675345221755, 1.8794767077860175, 0.36873250713136524, 0.15963580740408476, 1.9640795929663162, 0.3465082108844926, 0.20448646067471712, 0.3620435087818863, 0.07959753720287464, 0.08339241874766047, 0.06710865822483786, 0.04139980210192376, 0.6895990966172538, 0.6348438311759198, 0.22707099654607205, 0.03392095256315838, 38.592979005558206, 0.6132050679231041, 0.4198763557915873, 1.38380805397124, 1.293859968160405, 1.0976572767828576, 0.7001444609437095, 0.27757737066080584, 0.06946829639443497, 1.9848421277855721, 0.40314974548105265, 0.16199908419501954, 2.030416672130051, 0.3736486993572914, 0.2893104289558502, 0.3808710393692196, 0.07780374596108659, 0.08320120953109782, 0.06873486075940839, 0.04453472095126764, 0.683807700183431, 0.717116655469136, 0.227680935297412, 0.03392095256315838, 0.32824260283744167, 0.380842200659128, 0.16093052162576027, 0.04337485325968618, 0.19200436145002478, 1.905887391985024], 'coef': [[-0.06940070984026342, 0.10218803384155796, -0.10449556855818597, 0.050833713834707965, 0.029471256014503972, 0.2064236683807975, 0.04152579751342489, 0.10011151220408283, 0.06306650783398199, -0.06936550215146765, -0.20170334599159198, 0.0954098067129377, 0.25076826774931626, 0.038048582718493366, 0.0879875745425837, -0.01064770820488679, 0.026355787373272443, 0.1555551371749171, -0.0030870961342364103, 0.12333655514417023, -0.00909000653998989, 0.20242430308752002, 0.050668049306142354, 0.1735736815897637, -0.06940070984026342, 0.06039048607922554, -0.01748078949745143, -0.096050689650006, 0.07428775540638542, 0.10537131205429756, 0.09679734622260205, 0.07487116240977784, 0.141187047357883, -0.15242982747242814, -0.1454710602515456, -0.0627652679699293, 0.1253205177584775, 0.05784810393914934, 0.038311684463502936, 0.028481601375793166, -0.007121847913253451, 0.04922507355464619, -0.01645753290921352, -0.04835848239392935, 0.1206508905697725, -0.010202341557182824, 0.09099733283110376, 0.1735736815897637, 0.07360008058927336, -0.10328463890179257, -0.09773136500216231, 0.06114054601854305, -0.07657508739304277, 0.12491607302423098], [-0.27308200885772593, 0.07526189078809645, -0.131437915642093, 0.04632654763218192, 0.06021011896099205, -0.08068057411102672, 0.07112555467757038, 0.14749293795227192, 0.021988054343681666, -0.05325697085563757, -0.14677512788642086, 0.12930422217682308, -0.028357798855977733, 0.02612857077307902, 0.08910527771044773, -0.09499200887006208, 0.10488026378691331, 0.1461102402041511, 0.06374997892549883, -0.11962192435079316, -0.2357636362704654, 0.39499817742560067, -0.08919275563465683, 0.057262121592808225, -0.27308200885772593, 0.03829509716544983, 0.000957115127064535, -0.17390586339384212, -0.11758339155240273, 0.15184750074263226, 0.04957822465742139, -0.11987325773366131, 0.04934309145576599, -0.14173876858057127, -0.08502829250285676, -0.06351754015466908, 0.07625524754583801, 0.028894485224554557, 0.03255504948781556, -0.027649221572406308, 0.12957942590859553, 0.1861574962571462, 0.14119925310670525, -0.02424730826538737, 0.05606734163902229, 0.012923715468210133, -0.07666212003081248, 0.057262121592808225, 0.06575700025985277, 0.03674763372839612, -0.12329312182938595, -0.03996679887700924, -0.12427036687776219, -0.11046135300243268], [-0.038003372519655874, 0.054385446379199294, -0.10748455153802167, 0.0823558304435, -0.1358783126794774, 0.03432575849173638, 0.12596508791732736, -0.1574865162396906, -0.08668389060268653, 0.00520515123528877, -0.01671192963065932, -0.057084161253563, 0.061060089240904204, -0.07619992889599002, 0.013524216064757264, -0.09223787726621702, -0.03953662735360317, 0.20781529943759902, 0.08410025645093487, -0.0905404756524612, -0.05193249499098788, 0.16358925858517875, 0.05652905888071671, 0.03210405075524166, -0.038003372519655874, 0.04511501669548815, 0.07185279441937975, -0.06706614857985749, -0.0053482165985838685, 0.11338734399096616, 0.06286517391175177, 0.0257639416542483, -0.1722455695857165, -0.17286526353947448, -0.013612154424763729, 0.200912282350288, -0.04944622127359967, -0.05935259596370297, 0.04337316670376969, -0.031186571147161116, 0.0018567452275227779, 0.034407075949474474, 0.2657267286041039, -0.06101832524897019, 0.03920539915498692, 0.041360976986450824, 0.033737315352691845, 0.03210405075524166, 0.014932174627891384, 0.11092920891387523, -0.004191219372867686, -0.07588451956125648, -0.11206028237266398, 0.11560142965269285], [0.1332680928665223, 0.06737936014359873, 0.11751460397045753, 0.04200945835327355, 0.0041718039273808075, 0.16755111759632665, 0.17592958523611485, -0.09032607866848898, 0.024213697063074302, 0.04108580930437467, -0.09864893900712433, 0.2643550964092003, 0.1537267381333343, -0.01146249360148421, 0.19786744019489266, -0.037473191969560345, -0.12052856326691441, 0.18674130452721513, 0.08082209891694721, 0.01699125804310564, -0.19152765386592768, 0.3268084527757364, 0.07029665010273466, 0.14515547279640426, 0.1332680928665223, 0.03961144039534334, 0.05428064980859709, 0.08667627143208023, -0.0753148867651708, 0.020261648377351216, 0.0741688009558052, -0.00822924253844778, -0.18396913163569492, -0.0981451447770254, -0.06176695853222269, 0.15376912376276228, -0.017640482127336904, 0.0016362899544342248, -0.04425782711057732, -0.019296463061803933, -0.07414325364320061, 0.2542743816072348, 0.12937914360677347, 0.19845575921096412, 0.027296110609397718, 0.03384119153802916, 0.015007043973892464, 0.14515547279640426, 0.04891803385395669, 0.17999101138054116, -0.07129621453178196, -0.08818828514674121, -0.03238187880974811, 0.1772135540101047], [-0.05871006424487397, 0.03934862307041067, 0.03223638687048614, 0.1104527141730722, 0.07969179255215743, -0.03595630334327365, -0.038890287992255225, -0.03953068069735346, 0.04405072942206329, 0.02493228504571338, -0.13715980997812535, -0.04107359440390849, 0.14311524721671817, 0.07795842076794328, 0.09683736979335766, 0.026031254957282583, 0.010854263059164506, 0.18876735415611642, 0.15458218123598627, 0.21307213375045253, -0.0784846515059209, 0.16995946739876885, 0.05205149360022238, -0.08547137782605171, -0.05871006424487397, 0.05080816948302928, 0.004177649584047623, -0.10493103171854404, 0.11621843436634965, 0.1927734359279376, -0.07648830427493346, -0.01825451349945741, -0.13234210560728146, 0.031126293774556116, -0.08658176496522903, 0.07174118635716459, -0.017039872201999345, 0.010035066351341647, -0.0006470842124612798, 0.08974680675641947, 0.045023005180355624, 0.016797215477211528, 0.11115705267464769, -0.03600938238829063, -0.12259408683417045, 0.20986734728069797, 0.04753118040252267, -0.08547137782605171, -0.023134654813332416, 0.07068725533117698, -0.0973703679299773, -0.06084137815973197, -0.12894244953440576, 0.1656382105460042], [-0.0020672197228700077, -0.0461165239127655, 0.1190674711482114, -0.10358001420519292, -0.1710210871871977, -0.012622979946681021, 0.14781740691139011, -0.08008406923725495, -0.03036067588561329, 0.11106265999088251, 0.09062302204311332, -0.041304254181566434, -0.10099936050108077, -0.016741223865727742, -0.034349923955211435, 0.05905198150311783, -0.12841996359976052, -0.006816378618359575, -0.15544053812106495, 0.04395702413769389, 0.19769618239277506, -0.2966171553640023, -0.09867288676123652, -0.11428283941347525, -0.0020672197228700077, -0.028922562223976163, 0.09180243632444114, -0.19916767112936065, -0.07902514571765026, 0.017546630281493163, 0.17590638271304262, 0.06485206450715884, 0.16419783152572304, 0.03444962795313122, 0.09135670336136886, -0.09816087593685417, 0.1751056249335141, -0.05110407184048911, 0.12303238062128433, 0.052859276528582924, -0.05254263208860943, 0.07074642797960247, -0.044199813092150354, 0.16157993587443203, 0.0893334711830491, -0.13203177334291355, -0.1035982418123032, -0.11428283941347525, -0.030097314527685336, -0.05759373813207704, -0.021218955558267125, -0.1414160226514413, 0.00649370140735074, -0.2906300579147391], [-0.07260377957217772, -0.05650091927625106, 0.11217872045161677, -0.10265186387461087, -0.030915847305801925, -0.2688439911388818, -0.003907658325077459, -0.05001205460029712, 0.060312404551677384, 0.06266031169322407, 0.09971196839578211, 0.08160903468685606, -0.27001952073488306, -0.008467797108701791, -0.09734634724477692, 0.013162805409444407, 0.052138146754254384, -0.014946408561363556, -0.12341813067976565, -0.042860953186471464, -0.07357594240725203, -0.03165599777423161, -0.1602248307510553, -0.10095736961101835, -0.07260377957217772, -0.02163750358911075, -0.01929858577707289, 0.24015792333068633, -0.09788579091417861, -0.23046090270867703, 0.023412257702746745, 0.11543316621332404, 0.049778513451586, 0.009665109566784406, 0.08487500684730732, 0.013526254441546554, -0.08999764467904896, -0.056066328040701593, -0.1373668051983376, 0.04966633188442922, 0.09726291381816217, -0.24047004971641048, -0.06864374367471361, -0.012859911560588662, 0.2732503174571404, -0.29580505465698537, -0.08519464580238523, -0.10095736961101835, -0.06265078279143543, -0.05006781636397164, 0.015843524669079664, -0.07878645586932338, -0.073701238262309, -0.1823859655250203], [-0.034804455874000655, -0.13371156410191423, 0.017847144226947873, 0.17768912678933516, -0.022165239768046168, -0.16644343630050384, -0.044102756193569466, 0.013863600716955986, 0.24629215520638517, 0.07231345751727339, 0.04182379184976722, 0.15507498413789927, -0.3345616290500546, 0.1865612538960915, -0.2082643834703499, 0.04324391656423562, 0.03242335524204204, 0.010836317031011511, -0.03819435673678314, 0.07900218368240543, -0.09040481443098966, -0.1624718393968381, -0.07321782308211877, -0.23472111366422257, -0.034804455874000655, -0.03495985001766463, 0.030134906703507693, 0.07018608619455759, 0.0025186860241051758, -0.29562812211320266, -0.02191690560347659, 0.06514698250336015, 0.3015638485369947, 0.1715013310858658, 0.0009161777165890526, 0.10557613417286106, -0.21504601912283094, 0.056385049453255426, 0.16137042789049133, 0.054304112027858085, 0.0014028178126831606, -0.155975837868277, -0.17666143601792492, 0.16024834418772357, -0.04039328112926558, -0.028327997404016418, -0.13528622002192886, -0.23472111366422257, -0.17861557342932427, -0.03901008001059355, 0.09353374773898236, 0.05698404855907676, -0.026180052711949623, -0.1156797860313033], [-0.11186816598726078, -0.05329813715113244, -0.026357848194808287, -0.13452445891484247, -0.1502738920623752, -0.1538622998800485, 0.06137069703541959, -0.04662756848333584, 0.13304983903181675, 0.23067198332017794, 0.03659530272608114, 0.3251717441881989, -0.4527533963412981, 0.05316229493869135, -0.0008804180243000996, 0.05671431577803973, 0.11089742594473516, -0.27179530315610256, -0.24064720277873577, 0.16173609258471142, -0.001553204136518404, -0.1004336726757443, -0.10188943586681477, -0.15717637848766927, -0.11186816598726078, 0.012153089427818755, 0.03944969255781264, 0.06588677982917522, -0.004640243423334967, 0.008052550930425112, 0.08316106841542738, 0.013582132267034855, 0.14309706207503697, 0.07963221100994722, -0.008671033916837913, 0.18384629929203342, -0.3287556474519791, -0.010589119989541738, -0.12149895748537101, 0.10191016617352065, 0.08165595058703834, -0.27643645600152655, -0.2650302265728774, -0.025858578612462265, -0.016554804589201163, 0.0370914295934426, -0.1573916091021613, -0.15717637848766927, -0.11993380846025095, -0.042531906338767224, 0.10557104189072898, 0.05703819069639486, -0.09521440498689737, -0.11634095471180392], [0.004413955757619904, -0.06037617646141136, 0.14325875244835803, -0.15297162613061346, 0.1160806326260341, 0.09377882796517237, -0.094141557100761, 0.03303492152856712, -0.057867345187059285, -0.0670852829085587, 0.026553479544434096, -0.05261766720252924, 0.19008209230817044, 0.07607976966142976, 0.23116480413385385, 0.12689407697143298, 0.008741011735136842, -0.19586931539340646, -0.03541851167821635, -0.09809725594426419, -0.01829050181165831, -0.09384371451424485, 0.1301503746003017, 0.12985935195845968, 0.004413955757619904, 0.1661654589626142, -0.08348260848970304, 0.047118910764631254, 0.24100838721258958, 0.27878572213202313, -0.036913750139817104, 0.24426924949973491, 0.08089170561177429, 0.09539125580938197, -0.14576438306905146, 0.15807669468501043, 0.4489297159284768, -0.11542560616252082, 0.11690360005778767, -0.020709679890642842, 0.11535081301503311, 0.3068360883686124, -0.041077669568682064, 0.09713467664806659, -0.07358356292747148, 0.3553893521937178, -0.011303068440765854, 0.12985935195845968, -0.42056362066254066, -0.10141583839687285, 0.42599753190011125, -0.1908700946117736, 0.28035245537555203, -0.2823766102909395], [0.021453169103888243, 0.16234743944269892, -0.1302779945722615, 0.11286520819120713, 0.07484239976969263, 0.24974169587108316, 0.04120396235249916, 0.10089230223401902, 0.06859747523699285, -0.008071636886205621, -0.09591117747604029, 0.16341371401285, 0.4992453620858775, -0.17849059941608028, 0.03343847415696943, -0.08180794520128852, -0.0031940730360464384, 0.3090934644830715, 0.10378554742860795, 0.19637191878830662, 0.11513258483667944, 0.1821003633570029, 0.026566266933552705, 0.10755928638533549, 0.021453169103888243, 0.010320417870782698, 0.057312421354772665, 0.043111412271658614, -0.028491979743118224, 0.03141442169961865, 0.04840085123746539, 0.05663663313799844, 0.04187111782321304, 0.08132620327207232, 0.03372335607171432, -0.02729773983036252, 0.16086610191063794, -0.05332306787845864, 0.02734393981914687, -0.09883534219423853, -0.15586741449825975, -0.11778483493273277, -0.14557708234740233, -0.20460057871941342, -0.03505348461496612, 0.05247789601111855, 0.14474956333856984, 0.10755928638533549, 0.27688512663784237, -0.014890694480156163, -0.3042376974555391, 0.27372607586325903, 0.04179844624576484, 0.3431117782750726], [0.16531381941560475, -0.037083191011983214, 0.08628187415422098, 0.06672280122226679, 0.009961414945704886, 0.022122823048570186, -0.09424987740449534, -0.10944651427674883, -0.07425002303462881, 0.09384590019293126, -0.05775747585715538, -0.2191655375270085, 0.03568633197301756, 0.12554554031773654, 0.028070847042794663, 0.010508805391497168, 0.16825550859336064, -0.023258507886188474, 0.03114888125289252, 0.10055401138175822, -0.04382198296452898, -0.01944743519394531, 0.16085213344036056, 0.06863140167272308, 0.16531381941560475, -0.06974165059519931, 0.010092590753569367, -0.06482533905446734, 0.060083439138573176, -0.013484040834640523, -0.10079061664341647, -0.002112522709807648, -0.14152619221828577, -0.11156783513542529, -0.0005088300858398403, -0.0214009366719473, 0.003122302909437815, 0.11500392851309801, 0.008653651129456796, -0.02782995485971409, 0.09772892766337775, 0.00816625791373644, 0.13630292819901615, -0.0893342116377528, -0.14866734997260456, 0.020684287797757727, 0.18682127922662367, 0.06863140167272308, 0.06263791394209435, 0.01582131692749479, -0.13106229909474543, 0.1334655222743604, 0.0750205281863382, 0.033449626017445055], [0.0327426911422868, -0.04158173829273587, -0.0842499564926695, -0.03485549275907791, 0.15259511906557532, 0.0026720322740353174, -0.11571740845821374, 0.025777418456819034, 0.03437208302655565, -0.060596845033285635, 0.16233150309227115, -0.1708277724640674, 0.06479298470071711, -0.10088551638295175, -0.17493089209627466, -0.04183365154052118, -0.05932880526406804, -0.23055066976175392, 0.04793742214277917, -0.16035621344528844, -0.0003967313105844369, -0.07399582084124565, -0.014802338795916956, -0.059823778666229864, 0.0327426911422868, -0.05826686262858003, -0.06996626147851845, 0.057765807229963, -0.08416247701500662, 0.06620642346381321, -0.17626967386173917, -0.025771456912727333, -0.0651770545503517, 0.1080104181490452, 0.13794019776214062, -0.10341144214874591, 0.01495136106322667, -0.05320778634526337, -0.1840271183456907, -0.023875112234029144, -0.06293876825437329, -0.10162734564886795, 0.1218254529800166, 0.016473464092182612, 0.002544434305153032, -0.09972677766653266, -0.08571584465964088, -0.059823778666229864, 0.03299478198930387, 0.1159814918815069, 0.026386209771182906, 0.004021803857497531, -0.03152148800970293, 0.05084302810950505], [-0.02687962396141555, 0.0009116301333238196, 0.02137251267020715, -0.018768130929337174, 0.14089400255998918, 0.009034779550125749, -0.002942046423951715, 0.15992929930438637, -0.1758830730857085, -0.19686867914438141, -0.04523687150983157, -0.2142550250952029, -0.19469803543753653, 0.04656287579793585, -0.03863143549204249, 0.08116007398659333, -0.03087821123034445, -0.0833369297933882, -0.15736296848586084, -0.017299667694590695, 0.04153804430753591, -0.04035066587139467, 0.026997597002693777, -0.07446521807383034, -0.02687962396141555, -0.019379324256419885, -0.11365170239630687, 0.06896901512984155, -0.08324106603357657, -0.07091024622601741, -0.00764006003218497, 0.06911342295379835, -0.18852568924505395, 0.034830068731558826, -0.06680818400564222, -0.06960936121256224, -0.12633120552102525, 0.10388689239827939, 0.07105203357499246, 0.1073210042901282, -0.014464311865106797, -0.06668014359512082, -0.073221743593955, -0.03795944521891064, 0.007512785450648874, -0.04129367397920023, 0.01931256766959445, -0.07446521807383034, 0.03786646218296451, 0.00875539656186156, 0.06371319267311147, -0.030719456578526087, -0.05985272628307961, -0.0660571305643595], [0.04343487478479568, -0.049802433497836314, -0.03441816675546406, -0.0650279842424463, -0.04021324561186038, 0.04737555198640511, -0.15267730665005103, 0.012190179913465897, -0.02746304805730228, -0.07943611779906963, 0.09163021488590527, -0.1365730013594493, -0.06401170090112644, -0.011443379550243776, -0.061134677717274606, -0.049718086628978746, 0.0653865489804156, -0.11106721112922453, 0.034234678530664336, -0.13688873741533425, 0.07534179567254642, -0.17294882858946972, -0.010509300974618092, -0.001307186963315139, 0.04343487478479568, -0.07216031073451132, 0.024277375697254626, 0.09109284893203662, -0.031751063600452434, -0.062116290320087746, -0.11020105665139436, -0.07076301104486239, -0.09435213463843119, -0.034756388043358045, 0.13254722332013458, -0.23213382763342286, -0.021701626460231382, -0.024588098704016056, -0.18543646269887568, -0.10511010810407784, 0.021118592146499315, -0.16244956154755114, 0.11553663043634903, -0.13559084035140018, -0.14272146983652448, 0.013378828635627577, 0.03771889788517132, -0.001307186963315139, 0.043953022036614366, -0.07193836803202247, -0.12209828367772851, 0.08210984980839903, 0.11475408903390331, -0.04284656670545583], [-0.042270471812587036, 0.012323294315463139, -0.12264634106412328, 0.0008986683594727319, -0.16607113744034968, -0.1186677414578119, 0.04350468657997569, 0.10733132706113521, -0.13258774882778648, -0.01641008097695138, -0.03307278794210514, -0.024197066491384617, -0.25815972445783086, 0.013898067488641815, 0.019393089357308654, -0.020442632362700098, -0.2816308597829322, -0.15663047919477052, 0.015825782958223577, -0.06509318412300605, 0.0845045044854501, -0.07803661799737274, -0.11973104762076213, -0.0064434508188333135, -0.042270471812587036, 0.009732709980774766, -0.08662018695830567, -0.0423456794538851, 0.059272644636123015, -0.139638750973454, -0.007371811257331409, -0.2200936850053404, -0.22780692328103566, 0.05591215671429494, -0.021926001435476673, -0.058732217824832775, -0.29380878259243387, 0.007684530464662662, 0.05108669635595874, -0.06774370611061567, -0.217451630401289, 0.07035998706628152, -0.0910383613835188, -0.17904567239971889, 0.035782590319760416, -0.024582536351464394, -0.10437868527008226, -0.0064434508188333135, 0.004298864371779921, 0.09177937180883818, -0.020850924260284398, -0.12676864623756723, 0.09583372621737227, 0.046963952034570496], [0.18010702329337724, -0.026363117758564734, 0.09017295820308457, -0.053550991761398985, -0.05201231920725817, 0.12276444886498879, -0.04739969483879098, -0.09721658108281975, 0.058285629402613205, -0.08193880454483725, 0.09612515927886903, -0.01111976570125259, 0.053303989470287035, -0.05673220335498133, -0.07476149902096524, 0.0607718448406402, 0.21398478677712726, -0.09322405117823541, 0.13660584649092294, -0.05839380793393332, 0.018518806345961387, -0.07130072441462597, 0.08584849963868478, 0.11209256461272808, 0.18010702329337724, -0.06479038502244362, 0.06279925685661207, 0.2336399590561953, -0.05053646254587932, -0.10237069157839052, 0.01857323593722282, -0.2263695702773786, 0.03589166718184849, -0.012820479135414693, 0.03537465452472763, -0.08602362629616038, -0.030629417350701623, 0.06816164360838306, 0.10380832509948461, 0.0025047180731466315, 0.14662825238331176, -0.024365749921967078, 0.047926045562154115, 0.2727687251111099, -0.06717763236866238, -0.04460079366939273, 0.11056276017336389, 0.11209256461272808, 0.07294450978101734, -0.11937591296044749, 0.13162940004902263, 0.12967040357505522, 0.10962291276329142, 0.08756223389198395], [0.1489562460287349, -0.009311916649754275, 0.0014379186740366813, -0.024223506181497052, 0.10063254084033532, -0.11871337785101375, -0.11441418483655563, -0.029893436085713546, -0.1691327704380572, -0.008747637999471505, 0.18758302346283093, -0.24612075664483307, 0.25178006340144593, -0.1835222341838814, -0.10708951597577077, -0.04838597335806917, -0.13039999471275254, -0.017423862341288135, 0.000776130281205474, -0.14586895776646094, 0.262109702193875, -0.29877754999669237, 0.008280295981769895, 0.008410782161182257, 0.1489562460287349, -0.06273343699262127, -0.05663675458970078, -0.2563125911908638, 0.10459137712522798, -0.07103794484608855, -0.09527116328919237, -0.03820149542475304, 0.19812291574202523, 0.022479030617059925, 0.11940534358548377, -0.06439513938218014, 0.18584604673157668, -0.02587931498246373, -0.10425670015237729, -0.06455785793518883, -0.1530775850784871, 0.14881997505850697, -0.14714562600932854, -0.0517781683276542, -0.00489766841606561, -0.10044407687736394, 0.07309249428654611, 0.008410782161182257, 0.10020778441197661, -0.030583692916989935, 0.13067580002052093, 0.03528521704078535, 0.036824116011988636, 0.061478539184444944]], 'intercept': [-0.24156619910319668, -0.2706344949609409, 0.1599590154711437, 0.15208768105334894, 0.30625938485474286, 0.10409653774215712, 0.04202424664962271, -0.4606111789516119, -0.5934719275153107, -0.9130411778695865, -0.8603894527611426, 0.4153259959919418, 0.3842094509062752, 0.36802156260962093, 0.38676548930903965, 0.2783911068181156, 0.3899820806974939, 0.3525918790582875], 'classes': ['claude-fable-5', 'claude-opus-4-6', 'claude-opus-4-8', 'claude-opus-5', 'claude-sonnet-5', 'gemini-3.1-pro-preview', 'gemini-3.6-flash', 'gemini-3.7-flash', 'gemini-3.8-flash', 'glm-5.3', 'glm-5.3-flash', 'gpt-5.5', 'gpt-5.6-luna', 'gpt-5.6-sol', 'gpt-5.6-terra', 'gpt-6-astra', 'kimi-k3', 'qwen3.8-max'], 'C': 0.05, 'W': 1.0, 'entropy_ref': 0.5692304418233374, 'max_text_tokens': 160, 'tok_per_char': 0.19383936500423704, 'train_rows': 2699, 'source': 'experiments/bench/ll_trunc/L160'}))


In [ ]:
%%writefile /kaggle/temp/v1/segio.py
"""Read and write the binary files of ntl-segrun (see the header of tools/ntl-segrun/ntl-segrun.cpp)."""
import struct

import numpy as np


def write_texts(path, texts):
    with open(path, 'wb') as f:
        f.write(b'NTLT')
        f.write(struct.pack('<I', len(texts)))
        for t in texts:
            b = t.encode('utf-8')
            f.write(struct.pack('<I', len(b)))
            f.write(b)


def write_tokens(path, seqs):
    """seqs: list of (ids, n_prefix)."""
    with open(path, 'wb') as f:
        f.write(b'NTLK')
        f.write(struct.pack('<I', len(seqs)))
        for ids, n_prefix in seqs:
            ids = np.asarray(ids, dtype='<i4')
            f.write(struct.pack('<II', len(ids), n_prefix))
            f.write(ids.tobytes())


def read_tokens(path):
    out = []
    with open(path, 'rb') as f:
        assert f.read(4) == b'NTLK'
        (n,) = struct.unpack('<I', f.read(4))
        for _ in range(n):
            nt, npre = struct.unpack('<II', f.read(8))
            ids = np.frombuffer(f.read(4 * nt), dtype='<i4')
            out.append((ids, npre))
    return out


def read_scores(path):
    """List of float32 arrays [n_scored, 4]: logprob, entropy, E[logp^2], rank."""
    out = []
    with open(path, 'rb') as f:
        assert f.read(4) == b'NTLP'
        n, nf = struct.unpack('<II', f.read(8))
        for _ in range(n):
            (ns,) = struct.unpack('<I', f.read(4))
            out.append(np.frombuffer(f.read(4 * ns * nf), dtype='<f4').reshape(ns, nf))
    return out


def read_state(path):
    with open(path, 'rb') as f:
        assert f.read(4) == b'NTLS'
        dt, w, nt, nr = struct.unpack('<IIIQ', f.read(20))
        dtype = {0: '<f4', 1: '<f2'}.get(dt)
        if dtype is None:  # bf16
            raw = np.frombuffer(f.read(2 * w * nr), dtype='<u2').astype('<u4') << 16
            return raw.view('<f4').reshape(nr, w)
        return np.frombuffer(f.read(np.dtype(dtype).itemsize * w * nr), dtype=dtype).reshape(nr, w).astype('f4')


In [ ]:
%%writefile /kaggle/temp/v1/features.py
"""Per-text likelihood features from the per-token scores of ntl-segrun.

Per scored token: lp = log-prob of the true token, H = entropy of the next-token distribution,
m2 = E_p[logp^2] (so Var_p[logp] = m2 - H^2), rank = how many tokens beat the true one (0 = top-1).
"""
import numpy as np
import pandas as pd


def one(s, n_chars):
    lp, h, m2, rank = s[:, 0].astype(np.float64), s[:, 1].astype(np.float64), s[:, 2].astype(np.float64), s[:, 3]
    n = len(lp)
    var = np.maximum(m2 - h * h, 1e-9)
    half = max(n // 2, 1)
    q = np.quantile(lp, [0.05, 0.10, 0.25, 0.50, 0.75, 0.90]) if n else np.zeros(6)
    return {
        'n_tok': n,
        'lp_mean': lp.mean(),
        'lp_std': lp.std(),
        'lp_q05': q[0], 'lp_q10': q[1], 'lp_q25': q[2], 'lp_q50': q[3], 'lp_q75': q[4], 'lp_q90': q[5],
        'lp_min': lp.min(),
        'ent_mean': h.mean(),
        'ent_std': h.std(),
        # Fast-DetectGPT analytic curvature: how far the text's log-likelihood sits above the model's own expectation
        'fdg': (lp + h).sum() / np.sqrt(var.sum()),
        'surprise_mean': (-lp - h).mean(),          # observed surprisal minus expected surprisal
        'z_mean': ((lp + h) / np.sqrt(var)).mean(),
        'logrank_mean': np.log1p(rank).mean(),
        'top1': (rank == 0).mean(),
        'top5': (rank < 5).mean(),
        'top20': (rank < 20).mean(),
        'rank_ge100': (rank >= 100).mean(),
        'lp_mean_first': lp[:half].mean(),
        'lp_mean_second': lp[half:].mean() if n > half else lp.mean(),
        'bpc': -lp.sum() / max(n_chars, 1) / np.log(2),
        'tok_per_char': n / max(n_chars, 1),
    }


def text_features(scores, texts):
    return pd.DataFrame([one(s, len(t)) for s, t in zip(scores, texts)])


In [ ]:
%%writefile /kaggle/temp/v1/run_pipeline.py
"""Score text likelihood by streaming each model segment once per state chunk."""

import argparse
import glob
import json
import os
import re
import shutil
import struct
import subprocess
import sys
import threading
import time

import numpy as np
import pandas as pd

sys.path.insert(0, os.path.dirname(os.path.abspath(__file__)))
import features  # noqa: E402
import segio  # noqa: E402

CHAT_PREFIX = '[gMASK]<sop><|user|><|assistant|><think></think>'


def seg_index(path):
    m = re.search(r'seg(\d+)-of-(\d+)', os.path.basename(path))
    assert m, path
    return int(m.group(1)), int(m.group(2))


def gguf_kv_u32(path, key):
    """Read one uint32 metadata value from a GGUF header without the gguf package."""
    with open(path, 'rb') as f:
        assert f.read(4) == b'GGUF'
        _ver, _nt, nkv = struct.unpack('<IQQ', f.read(20))

        def rstr():
            (n,) = struct.unpack('<Q', f.read(8))
            return f.read(n).decode('utf-8', 'replace')

        sizes = {0: 1, 1: 1, 2: 2, 3: 2, 4: 4, 5: 4, 6: 4, 7: 1, 10: 8, 11: 8, 12: 8}

        def skip(t):
            if t == 8:
                rstr()
            elif t == 9:
                it, n = struct.unpack('<IQ', f.read(12))
                if it in sizes:
                    f.seek(sizes[it] * n, 1)
                else:
                    for _ in range(n):
                        skip(it)
            else:
                f.seek(sizes[t], 1)

        for _ in range(nkv):
            k = rstr()
            (t,) = struct.unpack('<I', f.read(4))
            if k == key:
                assert t == 4, (key, t)
                return struct.unpack('<I', f.read(4))[0]
            skip(t)
    return None


class EarlyFailure(RuntimeError):
    """ntl-segrun failed before writing any output (model load or context reservation, e.g. out of GPU memory)."""


_PREFETCHED = {}


def prefetch(path):
    """Read a file once in a daemon thread (page cache warm-up for the next mmap); each file at most once."""
    path = os.path.realpath(path)
    if path in _PREFETCHED:
        return
    info = _PREFETCHED[path] = {'s': None}

    def read():
        t0 = time.time()
        with open(path, 'rb', buffering=0) as f:
            try:
                os.posix_fadvise(f.fileno(), 0, 0, os.POSIX_FADV_SEQUENTIAL)
            except (AttributeError, OSError):
                pass
            buf = bytearray(16 << 20)
            while f.readinto(buf):
                pass
        info['s'] = time.time() - t0
    threading.Thread(target=read, daemon=True).start()


def run(cmd, log, on_ready=None):
    """Run one ntl-segrun call; on_ready() is called when it prints "context ready" (the model is loaded)."""
    t0 = time.time()
    p = subprocess.Popen(cmd, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, errors='replace')
    lines = []
    for line in p.stdout:
        lines.append(line)
        if on_ready is not None and 'context ready' in line:
            on_ready()
            on_ready = None
    p.wait()
    out = ''.join(lines)
    with open(log, 'a') as lf:
        lf.write('$ ' + ' '.join(cmd) + '\n' + out)
    dt = time.time() - t0
    if p.returncode != 0:
        print('\n'.join(out.splitlines()[-40:]), flush=True)
        err = EarlyFailure if 'context ready' not in out else RuntimeError
        raise err(f'failed ({p.returncode}): {" ".join(cmd)}')
    return dt


def main():
    ap = argparse.ArgumentParser()
    ap.add_argument('--bin', required=True)
    ap.add_argument('--segs', required=True, help='JSON list of the pinned segment paths')
    ap.add_argument('--csv', required=True)
    ap.add_argument('--text-col', default='text')
    ap.add_argument('--id-col', default='id')
    ap.add_argument('--out', required=True)
    ap.add_argument('--tag', required=True)
    ap.add_argument('--work', default='/tmp/ntl')
    ap.add_argument('--prefix', default=CHAT_PREFIX)
    ap.add_argument('--max-text-tokens', type=int, default=320)
    ap.add_argument('--seq-batch', type=int, default=8)
    ap.add_argument('--state-dtype', default='auto', help='auto | f32 | f16 | bf16')
    ap.add_argument('--threads', type=int, default=min(4, os.cpu_count() or 4))
    ap.add_argument('--limit', type=int, default=-1)
    ap.add_argument('--disk-frac', type=float, default=0.8, help='share of the free work disk the state may use')
    ap.add_argument('--pack', action='store_true', help='sort by length and pack padded decodes (ntl-segrun --pack)')
    ap.add_argument('--ubatch-div', type=int, default=1, help='n_ubatch = n_batch / P (pipeline parallel on 2 GPUs)')
    ap.add_argument('--prefetch', action='store_true', help='read the next segment while the current one computes')
    ap.add_argument('--prefetch-after', default='', help='with --prefetch: file to read during the last segment')
    a = ap.parse_args()
    main_run(a)


def main_run(a):

    t_start = time.time()

    os.makedirs(a.out, exist_ok=True)
    os.makedirs(a.work, exist_ok=True)
    log = os.path.join(a.out, f'{a.tag}_run.log')
    segrun = os.path.join(a.bin, 'ntl-segrun')

    segs = sorted(json.loads(a.segs), key=lambda p: seg_index(p)[0])
    assert segs, a.segs
    n_seg = seg_index(segs[0])[1]
    assert [seg_index(p)[0] for p in segs] == list(range(n_seg)), segs
    ranges = [(gguf_kv_u32(p, 'ntl.layer_start'), gguf_kv_u32(p, 'ntl.layer_end')) for p in segs]
    print(f'[{a.tag}] {n_seg} segments, layers {ranges}', flush=True)
    for (s0, e0), (s1, _e1) in zip(ranges, ranges[1:]):
        assert e0 == s1, ranges

    df = pd.read_csv(a.csv)
    if a.limit > 0:
        df = df.iloc[:a.limit]
    texts = df[a.text_col].astype(str).tolist()
    tx = os.path.join(a.work, f'{a.tag}_texts.bin')
    tk = os.path.join(a.work, f'{a.tag}_tok.bin')
    segio.write_texts(tx, texts)
    run([segrun, 'tokenize', '--model', segs[0], '--texts', tx, '--out', tk, '--prefix', a.prefix,
         '--max-text-tokens', str(a.max_text_tokens)], log)
    seqs = segio.read_tokens(tk)
    n_rows = np.array([len(ids) for ids, _ in seqs])
    # processing order: longest first when packing (similar lengths share a decode, little padding)
    order = np.argsort(-n_rows, kind='stable') if a.pack else np.arange(len(seqs))
    print(f'[{a.tag}] {len(seqs)} texts, {n_rows.sum()} rows, max {n_rows.max()}, prefix {seqs[0][1]}', flush=True)

    # state width: glm5-next hc*n_embd, glm-dsa n_embd (read from the first segment's metadata)
    n_embd = gguf_kv_u32(segs[0], 'glm5-next.embedding_length')
    if n_embd is not None:
        width = gguf_kv_u32(segs[0], 'glm5-next.hyper_connection.count') * n_embd
    else:
        width = gguf_kv_u32(segs[0], 'glm-dsa.embedding_length')
    free = shutil.disk_usage(a.work).free
    if a.state_dtype == 'auto':
        # exact f32 if the whole state fits in one chunk, else bf16 (half the disk, no overflow risk)
        a.state_dtype = 'f32' if n_rows.sum() * width * 4 <= free * a.disk_frac else 'bf16'
        print(f'[{a.tag}] state dtype auto -> {a.state_dtype}', flush=True)
    el = 2 if a.state_dtype in ('f16', 'bf16') else 4
    max_rows = int(free * a.disk_frac / (width * el))
    chunks, cur, cur_rows = [], [], 0
    for i in order:
        r = n_rows[i]
        if cur and cur_rows + r > max_rows:
            chunks.append(cur)
            cur, cur_rows = [], 0
        cur.append(i)
        cur_rows += r
    chunks.append(cur)
    print(f'[{a.tag}] state width {width}, {el} B/value, {free/2**30:.1f} GiB free on {a.work} -> '
          f'{len(chunks)} chunk(s)', flush=True)

    multi_gpu = False
    try:
        out = subprocess.run(['nvidia-smi', '-L'], capture_output=True, text=True).stdout
        multi_gpu = out.count('GPU ') > 1
    except FileNotFoundError:
        pass

    timing = {'tag': a.tag, 'n_texts': len(seqs), 'n_rows': int(n_rows.sum()), 'chunks': [], 'segments': segs,
              'ranges': ranges, 'seq_batch': a.seq_batch, 'state_dtype': a.state_dtype, 'prefix': a.prefix,
              'pack': a.pack, 'ubatch_div': a.ubatch_div}
    all_scores = [None] * len(seqs)
    st = os.path.join(a.work, f'{a.tag}_state.bin')
    done = 0
    for ci, idx in enumerate(chunks):
        ctk = os.path.join(a.work, f'{a.tag}_tok_c{ci}.bin')
        segio.write_tokens(ctk, [seqs[i] for i in idx])
        sc = os.path.join(a.work, f'{a.tag}_scores_c{ci}.bin')
        seg_t = []
        for k, p in enumerate(segs):
            while True:
                cmd = [segrun, 'run', '--model', p, '--tokens', ctk, '--threads', str(a.threads),
                       '--seq-batch', str(a.seq_batch), '--state-dtype', a.state_dtype]
                if multi_gpu:
                    cmd.append('--split-mid')
                if a.pack:
                    cmd.append('--pack')
                if a.ubatch_div > 1:
                    cmd += ['--ubatch-div', str(a.ubatch_div)]
                if k > 0:
                    cmd += ['--state-in', st]
                cmd += ['--out', sc if k == n_seg - 1 else st]
                nxt = (segs[k + 1] if k + 1 < n_seg else segs[0] if ci + 1 < len(chunks) else a.prefetch_after)
                ready = (lambda f=nxt: prefetch(f)) if a.prefetch and nxt else None
                try:
                    dt = run(cmd, log, on_ready=ready)
                    break
                except EarlyFailure:
                    # nothing written yet (the input state is intact): retry with half the texts per decode
                    if a.seq_batch == 1:
                        raise
                    a.seq_batch //= 2
                    timing['seq_batch_reduced_to'] = a.seq_batch
                    print(f'[{a.tag}] segment {k+1} failed before writing: retrying with seq-batch {a.seq_batch}', flush=True)
            seg_t.append(dt)
            print(f'[{a.tag}] chunk {ci+1}/{len(chunks)} segment {k+1}/{n_seg} {os.path.basename(p)}: {dt:.0f} s '
                  f'(total {time.time()-t_start:.0f} s)', flush=True)
        for i, s in zip(idx, segio.read_scores(sc)):
            all_scores[i] = s
        done += len(idx)
        timing['chunks'].append({'n_texts': len(idx), 'seg_seconds': seg_t})
        for f in (ctk, sc, st):
            if os.path.exists(f):
                os.remove(f)

    ok = [i for i, s in enumerate(all_scores) if s is not None]
    off = np.zeros(len(ok) + 1, dtype=np.int64)
    off[1:] = np.cumsum([len(all_scores[i]) for i in ok])
    np.savez_compressed(os.path.join(a.out, f'{a.tag}_scores.npz'),
                        ids=df[a.id_col].astype(str).values[ok],
                        scores=np.concatenate([all_scores[i] for i in ok]).astype(np.float32),
                        offsets=off,
                        n_prefix=np.array([seqs[i][1] for i in ok]),
                        tokens=np.concatenate([seqs[i][0][seqs[i][1]:] for i in ok]).astype(np.int32))
    feats = features.text_features([all_scores[i] for i in ok], [texts[i] for i in ok])
    feats.insert(0, a.id_col, df[a.id_col].astype(str).values[ok])
    feats.to_csv(os.path.join(a.out, f'{a.tag}_features.csv'), index=False)
    timing['total_s'] = time.time() - t_start
    if a.prefetch:
        timing['prefetch_s'] = {os.path.basename(k): v['s'] for k, v in _PREFETCHED.items()}
    timing['n_done'] = len(ok)
    with open(os.path.join(a.out, f'{a.tag}_timing.json'), 'w') as f:
        json.dump(timing, f, indent=1)
    print(f'[{a.tag}] done: {len(ok)}/{len(seqs)} texts in {timing["total_s"]:.0f} s', flush=True)


if __name__ == '__main__':
    main()


In [ ]:
%%writefile /kaggle/temp/v1/v1_stage.py
"""Rank by GLM probability and add two-model likelihood evidence on selected texts."""

import argparse
import glob
import json
import os
import subprocess
import sys
import time
from pathlib import Path

import numpy as np
import pandas as pd
from scipy.special import log_softmax

PREFIX = '[gMASK]<sop><|system|>Reasoning Effort: Max<|user|><|assistant|><think></think>'


def entropy(z):
    return float(-(np.exp(z) * z).sum(1).mean())


def match_temperature(z, h_ref):
    """T' with mean entropy of log_softmax(T' z) equal to h_ref (entropy decreases with T')."""
    lo, hi = 1e-3, 1e3
    if entropy(log_softmax(z * hi, 1)) > h_ref or entropy(log_softmax(z * lo, 1)) < h_ref:
        raise ValueError('entropy_ref out of reach')
    for _ in range(100):
        mid = np.sqrt(lo * hi)
        if entropy(log_softmax(z * mid, 1)) > h_ref:
            lo = mid
        else:
            hi = mid
    return float(np.sqrt(lo * hi))


def member_logp(model, feats, ids):
    """feats: {'flash': DataFrame, 'glm53': DataFrame} indexed by id -> (n, 18) log-probs, model['classes'] order."""
    blocks = [feats[t].add_prefix(f'{t}:') for t in ('flash', 'glm53')]
    F = pd.concat(blocks, axis=1).reindex(ids)
    for c in model['diff_features']:
        F[f'd:{c}'] = F[f'flash:{c}'] - F[f'glm53:{c}']
    X = F[model['features']].to_numpy(np.float64)
    assert np.isfinite(X).all(), 'non-finite likelihood features'
    z = ((X - np.asarray(model['mean'])) / np.asarray(model['scale'])) @ np.asarray(model['coef']).T + np.asarray(model['intercept'])
    return log_softmax(z, 1)


def main():
    ap = argparse.ArgumentParser()
    for k in ('--test-csv', '--final-logits', '--out', '--report', '--bin', '--pipeline', '--flash-segs', '--glm53-segs',
              '--model', '--work', '--outdir'):
        ap.add_argument(k, required=True)
    ap.add_argument('--pack', action='store_true')
    ap.add_argument('--ubatch-div', type=int, default=1)
    ap.add_argument('--seq-batch', type=int, default=8)
    ap.add_argument('--max-text-tokens', type=int, default=0,
                    help='0 = the cut the member was trained with (model max_text_tokens); any other value must match it')
    ap.add_argument('--select-classes', default='glm-5.3,glm-5.3-flash',
                    help='texts are scored in decreasing order of the S10 probability of these classes')
    ap.add_argument('--select-mode', choices=('mass', 'uncertain'), default='mass',
                    help='ours: uncertain = rank by class mass times (1 - top probability)')
    ap.add_argument('--max-cover', type=float, default=0.20, help='at most this fraction of the test is scored')
    ap.add_argument('--min-texts', type=int, default=50, help='fewer texts than this fit: keep S10')
    ap.add_argument('--prefetch', action='store_true', help='run_pipeline --prefetch (page-cache warm-up of segments)')
    ap.add_argument('--max-changed', type=float, default=0.25,
                    help='sanity guard: the blend changes ~11%% of the predictions on train OOF (W=1); far more means '
                         'something is broken (e.g. features unlike train): fail instead of submitting it')
    a = ap.parse_args()

    t0 = time.monotonic()
    rep = {'status': 'started'}
    out_report = Path(a.report)

    def save(**kw):
        rep.update(kw)
        out_report.write_text(json.dumps(rep, indent=1))
        print('V1', json.dumps({k: v for k, v in kw.items() if not isinstance(v, dict)}), flush=True)

    model = json.loads(Path(a.model).read_text())
    assert model['format'] == 'ntl-v1-ll-model-1'
    a.max_text_tokens = a.max_text_tokens or model['max_text_tokens']
    assert a.max_text_tokens == model['max_text_tokens'], 'the test must be cut like the train features'
    frame = pd.read_csv(a.test_csv, dtype={'id': str})
    frame['text'] = frame.text.fillna('').astype(str)
    ids = frame.id.tolist()
    with np.load(a.final_logits, allow_pickle=False) as f:
        assert f['ids'].astype(str).tolist() == ids, 'final-logits ids differ from the test order'
        labels = f['labels'].astype(str).tolist()
        z = f['logits'].astype(np.float64)
    assert sorted(labels) == sorted(model['classes']) and z.shape == (len(ids), 18) and np.isfinite(z).all()

    zl = log_softmax(z, 1)
    sel_cls = a.select_classes.split(',')
    assert all(c in labels for c in sel_cls), sel_cls
    mass = np.exp(zl[:, [labels.index(c) for c in sel_cls]]).sum(1)
    if a.select_mode == 'uncertain':   # ours: confident texts cannot flip, spend the token budget on close calls
        priority = mass * (1.0 - np.exp(zl).max(1))
    else:
        priority = mass
    order = np.argsort(-priority, kind='stable')

    # Bound work before starting, below the measured 160k-token pass.
    tpc = model.get('tok_per_char', 0.235)
    rows_i = np.minimum(frame.text.str.len().to_numpy() * tpc * 1.05, a.max_text_tokens) + 13
    cum = np.cumsum(rows_i[order])
    n_sel = min(int((cum <= 130000).sum()), int(a.max_cover * len(ids)))
    save(status='selected', select_mode=a.select_mode, select_classes=sel_cls, n_texts=len(ids), n_selected=n_sel,
         estimated_token_rows=int(cum[n_sel-1]) if n_sel else 0)
    if n_sel < a.min_texts:
        save(status='fallback_s10', reason='Insufficient selected texts')
        return
    sel = np.sort(order[:n_sel])
    sel_ids = [ids[i] for i in sel]

    Path(a.work).mkdir(parents=True, exist_ok=True)
    Path(a.outdir).mkdir(parents=True, exist_ok=True)
    times = {}
    for tag, segs in (('flash', a.flash_segs), ('glm53', a.glm53_segs)):
        query = Path(a.work) / f'v1_query_{tag}.csv'
        frame.iloc[sel][['id', 'text']].to_csv(query, index=False)
        cmd = [sys.executable, a.pipeline, '--bin', a.bin, '--segs', segs, '--csv', str(query), '--out', a.outdir,
               '--tag', tag, '--work', str(Path(a.work) / 'state'), '--seq-batch', str(a.seq_batch),
               '--state-dtype', 'auto', '--max-text-tokens', str(a.max_text_tokens), '--prefix', PREFIX]
        if a.pack:
            cmd.append('--pack')
        if a.ubatch_div > 1:
            cmd += ['--ubatch-div', str(a.ubatch_div)]
        if a.prefetch:
            cmd.append('--prefetch')
            if tag == 'flash':
                cmd += ['--prefetch-after', json.loads(a.glm53_segs)[0]]
        t = time.monotonic()
        with open(Path(a.outdir) / f'{tag}_pipeline.txt', 'w') as log:
            code = subprocess.call(cmd, stdout=log, stderr=subprocess.STDOUT)
        times[tag] = time.monotonic() - t
        if code != 0:
            raise RuntimeError(f'V1 {tag} pipeline failed with exit code {code}; see {tag}_pipeline.txt')
        save(status=f'{tag}_done', times=times)

    feats = {t: pd.read_csv(Path(a.outdir) / f'{t}_features.csv', dtype={'id': str}).set_index('id')
             for t in ('flash', 'glm53')}
    for t, f in feats.items():
        assert set(sel_ids) <= set(f.index), f'{t}: features missing for some texts'
        feats[t] = f.loc[sel_ids]
    ll = member_logp(model, feats, sel_ids)[:, [model['classes'].index(c) for c in labels]]

    tp = match_temperature(zl, model['entropy_ref'])
    w = model['W'] / tp
    before = zl.argmax(1)
    after = before.copy()
    after[sel] = (zl[sel] + w * ll).argmax(1)
    changed = float((before != after).mean())
    changed_sel = float((before[sel] != after[sel]).mean())
    agrees = float((ll.argmax(1) == before[sel]).mean())
    if changed > a.max_changed:
        save(status='error_sanity', changed=changed, changed_selected=changed_sel, temperature=tp, weight=w,
             member_argmax_agrees_s10=agrees)
        raise RuntimeError(f'V1 blend changes {changed:.1%} of the predictions (> {a.max_changed:.0%}): not submitted')
    sub = pd.DataFrame({'id': ids, 'model': np.asarray(labels)[after]})
    assert len(sub) == len(frame) and sub.id.is_unique and sub.model.isin(labels).all()
    sub.to_parquet(a.out, index=False)
    assert pd.read_parquet(a.out).equals(sub)
    count = lambda p: {labels[i]: int(n) for i, n in zip(*np.unique(p, return_counts=True))}
    save(status='applied', n_blended=len(sel_ids), entropy_s10=entropy(zl), temperature=tp, weight=w, changed=changed,
         changed_selected=changed_sel, member_argmax_agrees_s10=agrees,
         counts_s10=count(before), counts_v1=count(after), times=times, total_s=time.monotonic() - t0)


if __name__ == '__main__':
    main()


In [ ]:
# GLM segment paths come from the pinned Kaggle input versions.
import json, os, shutil, subprocess, sys
from pathlib import Path
V1_DIR = Path('/kaggle/temp/v1')
V1_OUT = Path('/kaggle/working/v1')
V1_OUT.mkdir(parents=True, exist_ok=True)
GLM_SEGMENTS = {'jamesliebel/glm53-iq3xxs-segs-a-mirror': [['glm53_iq3xxs-seg00-of-19-L0-6.gguf', 12753355264], ['glm53_iq3xxs-seg01-of-19-L6-10.gguf', 15651714688], ['glm53_iq3xxs-seg02-of-19-L10-14.gguf', 14583346912], ['glm53_iq3xxs-seg03-of-19-L14-18.gguf', 14583346912], ['glm53_iq3xxs-seg04-of-19-L18-22.gguf', 14583346912], ['glm53_iq3xxs-seg05-of-19-L22-26.gguf', 14583346912], ['glm53_iq3xxs-seg06-of-19-L26-30.gguf', 14583346912], ['glm53_iq3xxs-seg07-of-19-L30-34.gguf', 14583346912], ['glm53_iq3xxs-seg08-of-19-L34-38.gguf', 14583346912], ['glm53_iq3xxs-seg09-of-19-L38-42.gguf', 14583346912]], 'jamesliebel/glm53-iq3xxs-segs-b-mirror': [['glm53_iq3xxs-seg10-of-19-L42-46.gguf', 14583346912], ['glm53_iq3xxs-seg11-of-19-L46-50.gguf', 14583346912], ['glm53_iq3xxs-seg12-of-19-L50-54.gguf', 14583346912], ['glm53_iq3xxs-seg13-of-19-L54-58.gguf', 14583346912], ['glm53_iq3xxs-seg14-of-19-L58-62.gguf', 14583346912], ['glm53_iq3xxs-seg15-of-19-L62-66.gguf', 14583346912], ['glm53_iq3xxs-seg16-of-19-L66-70.gguf', 14583346912], ['glm53_iq3xxs-seg17-of-19-L70-74.gguf', 14583346912], ['glm53_iq3xxs-seg18-of-19-L74-78.gguf', 16345433920]], 'jamesliebel/glm53-flash-iq3xxs-segs-mirror': [['flash_iq3xxs-seg00-of-06-L0-8.gguf', 14879704352], ['flash_iq3xxs-seg01-of-06-L8-16.gguf', 22864262976], ['flash_iq3xxs-seg02-of-06-L16-24.gguf', 21821175680], ['flash_iq3xxs-seg03-of-06-L24-32.gguf', 21821175680], ['flash_iq3xxs-seg04-of-06-L32-40.gguf', 21821175680], ['flash_iq3xxs-seg05-of-06-L40-45.gguf', 14407253632]]}

def pinned_dataset(owner, slug):
    namespaced = Path('/kaggle/input/datasets') / owner / slug
    flat = Path('/kaggle/input') / slug
    if namespaced.is_dir():
        return namespaced
    assert flat.is_dir(), (namespaced, flat)
    return flat

V1_FLASH, V1_GLM53 = [], []
for ref, files in GLM_SEGMENTS.items():
    owner, slug = ref.split('/')
    root = pinned_dataset(owner, slug)
    for filename, size in files:
        p = root / filename
        assert p.is_file() and p.stat().st_size == size, p
        (V1_FLASH if filename.startswith('flash_') else V1_GLM53).append(str(p))
assert len(V1_FLASH) == 6 and len(V1_GLM53) == 19
V1_FLASH.sort()
V1_GLM53.sort()
artifacts = Path('/kaggle/input/notebooks/estopadilla/ntl-glm-likelihood-train/ntl-bin')
if not (artifacts/'ntl-segrun').is_file():
    artifacts = Path('/kaggle/input/ntl-glm-likelihood-train/ntl-bin')
for filename in ('ntl-segrun','lib/libcublas.so.13','lib/libcublasLt.so.13','lib/libcudart.so.13'):
    assert (artifacts/filename).is_file(), artifacts/filename
V1_BIN = Path('/kaggle/temp/v1-bin')
V1_BIN.mkdir(exist_ok=True)
shutil.copyfile(artifacts/'ntl-segrun',V1_BIN/'ntl-segrun')
(V1_BIN/'ntl-segrun').chmod(0o755)
V1_ENV = dict(os.environ,LD_LIBRARY_PATH=str(artifacts/'lib')+':'+os.environ.get('LD_LIBRARY_PATH',''))
sys.path.insert(0,str(V1_DIR))
import segio
from v1_stage import PREFIX
smoke_texts = pd.read_csv(DATA,dtype={'id':str}).text.head(3).fillna('').tolist()
smoke_root = V1_OUT
segio.write_texts(str(smoke_root/'smoke.bin'),smoke_texts)
smoke_results = {}
for tag, segments in (('flash',V1_FLASH),('glm53',V1_GLM53)):
    tokens = smoke_root/f'{tag}-smoke-tokens.bin'
    subprocess.run([str(V1_BIN/'ntl-segrun'),'tokenize','--model',segments[0],'--texts',str(smoke_root/'smoke.bin'),
                    '--out',str(tokens),'--max-text-tokens','160','--prefix',PREFIX],env=V1_ENV,check=True)
    smoke_results[tag] = str(tokens)
print('Verified 25 GLM segments, CUDA runtime and two tokenizers', flush=True)


## Fixed inputs

Every weight and fitted state uses an explicit path. The new `ntl666-prefit` cache contains the three core text heads fitted on the cleaned Open Pool and the matched external sample. The existing twelve-part cache continues to supply the complementary stylegram heads and family gates. Its old core-head files remain for historical reproducibility but are not read by the new core branch.

The frozen dense readouts, Gaussian states and LoRA checkpoints are unchanged. All three text heads and the optional small sibling heads are fitted before publication. Inference only transforms the test texts and applies their saved coefficients. The new cache is copied once to local temporary storage; the exact public sample preview skips cache preparation and all model forwards. All algorithms and transformations are visible in the cells below.

In [ ]:
CONFIG = {'RELEASE_READY': True, 'paths': {'test_csv': '/kaggle/input/competitions/name-that-llm/test.csv', 'new_two_stage_root': '/kaggle/input/ntl-handwriting-blend', 'qwen3_14b': '/kaggle/input/models/qwen-lm/qwen-3/transformers/14b-base/1', 'qwen25_14b': '/kaggle/input/models/qwen-lm/qwen2.5/transformers/14b/1', 'qwen_wheels': ['/kaggle/input/datasets/lightningv08/ntl-olmo2-native24-final-inference-assets/name-that-llm-artifacts/qwen38-offline-linux/accelerate-1.15.0-py3-none-any.whl', '/kaggle/input/datasets/lightningv08/ntl-olmo2-native24-final-inference-assets/name-that-llm-artifacts/qwen38-offline-linux/bitsandbytes-0.50.2-py3-none-manylinux_2_24_x86_64.whl', '/kaggle/input/datasets/lightningv08/ntl-olmo2-native24-final-inference-assets/name-that-llm-artifacts/qwen38-offline-linux/huggingface_hub-1.32.0-py3-none-any.whl', '/kaggle/input/datasets/lightningv08/ntl-olmo2-native24-final-inference-assets/name-that-llm-artifacts/qwen38-offline-linux/safetensors-0.8.0-cp310-abi3-manylinux_2_17_x86_64.manylinux2014_x86_64.whl', '/kaggle/input/datasets/lightningv08/ntl-olmo2-native24-final-inference-assets/name-that-llm-artifacts/qwen38-offline-linux/tokenizers-0.22.2-cp39-abi3-manylinux_2_17_x86_64.manylinux2014_x86_64.whl', '/kaggle/input/datasets/lightningv08/ntl-olmo2-native24-final-inference-assets/name-that-llm-artifacts/qwen38-offline-linux/transformers-5.14.1-py3-none-any.whl'], 'legacy_runtime': '/kaggle/temp/ntl-gsq-v27-runtime/legacy', 'legacy_prefit': '/kaggle/input/datasets/lightningv08/ntl-olmo2-native24-final-inference-assets/ntl611-prefit', 'complement_style': '/kaggle/input/datasets/lightningv08/ntl-olmo2-native24-final-inference-assets/ntl636-updated-style', 'v5_adapters': '/kaggle/input/nlm-l24-adapters-v5', 'gsq_base': '/kaggle/input/datasets/lightningv08/ntl-olmo2-native24-final-inference-assets/gsq3-base/qwen38-27b-gsq3', 'gsq_final': '/kaggle/input/datasets/lightningv08/ntl-olmo2-native24-final-inference-assets/gsq120k-trained/gsq120k-inference/gsq/final.safetensors', 'gsq_metadata': '/kaggle/input/datasets/lightningv08/ntl-olmo2-native24-final-inference-assets/gsq120k-trained/gsq120k-inference/gsq/final.json', 'gsq_complete': '/kaggle/input/datasets/lightningv08/ntl-olmo2-native24-final-inference-assets/gsq120k-trained/gsq120k-inference/gsq/COMPLETE.json', 'svc4_prefit': '/kaggle/input/datasets/lightningv08/ntl-olmo2-native24-final-inference-assets/gsq120k-trained/gsq120k-inference/svc4', 'gaussian_q14v5': '/kaggle/input/datasets/lightningv08/ntl-olmo2-native24-final-inference-assets/gsq120k-trained/gsq120k-inference/gaussian/q14v5', 'gaussian_q8v5': '/kaggle/input/datasets/lightningv08/ntl-olmo2-native24-final-inference-assets/gsq120k-trained/gsq120k-inference/gaussian/q8v5', 'train_csv': '/kaggle/input/competitions/name-that-llm/train.csv', 'ntl660_prefit': '/kaggle/temp/ntl660-prefit'}, 'gsq_head': 'plain', 'complementary_share': 0.15, 'gsq_share': 0.07, 'gsq_base_revision': 'b5ce0b76f60020a875dee4f6ec9d934cca4121e4', 'gsq_base_inventory': {'config.json': 5038, 'merges.txt': 3353259, 'model.safetensors.index.json': 198539, 'tokenizer.json': 12809320, 'tokenizer_config.json': 17928, 'vocab.json': 6722759, 'model-00001-of-00003.safetensors': 2272369784, 'model-00002-of-00003.safetensors': 4777704120, 'model-00003-of-00003.safetensors': 4777704376}, 'artifact_hub_version': 17, 'models': [{'name': 'q14v5', 'type': 'l24', 'path': '/kaggle/input/models/qwen-lm/qwen-3/transformers/14b-base/1', 'adapters': ['/kaggle/input/nlm-l24-adapters-v5/q14v5_f0', '/kaggle/input/nlm-l24-adapters-v5/q14v5_f1', '/kaggle/input/nlm-l24-adapters-v5/q14v5_f2', '/kaggle/input/nlm-l24-adapters-v5/q14v5_f3', '/kaggle/input/nlm-l24-adapters-v5/q14v5_f4', '/kaggle/input/nlm-l24-adapters-v5/q14v5'], 'keep': 27, 'layers': [2, 4, 6, 8, 12, 16, 20, 24], 'w': 3.0, 'bs': 8, 'max_len': 384, 'twin': 0.35, 'gauss': 0.7, 'adapter_source_ref': 'dhruvtibarewal/nlm-l24-adapters-v5', 'adapter_source_version': 1, 'adapter_relative_paths': ['q14v5_f0', 'q14v5_f1', 'q14v5_f2', 'q14v5_f3', 'q14v5_f4', 'q14v5'], 'base_source_ref': 'qwen-lm/qwen-3/Transformers/14b-base', 'base_source_version': 1, 'base_inventory_key': 'qwen-lm/qwen-3/Transformers/14b-base/1'}, {'name': 'q8v5', 'type': 'l24', 'path': '/kaggle/input/models/qwen-lm/qwen-3/transformers/8b-base/1', 'adapters': ['/kaggle/input/nlm-l24-adapters-v5/q8v5_f0', '/kaggle/input/nlm-l24-adapters-v5/q8v5_f1', '/kaggle/input/nlm-l24-adapters-v5/q8v5_f2', '/kaggle/input/nlm-l24-adapters-v5/q8v5_f3', '/kaggle/input/nlm-l24-adapters-v5/q8v5_f4', '/kaggle/input/nlm-l24-adapters-v5/q8v5'], 'keep': 24, 'layers': [2, 4, 6, 8, 12, 16, 20, 24], 'w': 2.0, 'bs': 16, 'max_len': 384, 'twin': 0.35, 'gauss': 0.44, 'adapter_source_ref': 'dhruvtibarewal/nlm-l24-adapters-v5', 'adapter_source_version': 1, 'adapter_relative_paths': ['q8v5_f0', 'q8v5_f1', 'q8v5_f2', 'q8v5_f3', 'q8v5_f4', 'q8v5'], 'base_source_ref': 'qwen-lm/qwen-3/Transformers/8b-base', 'base_source_version': 1, 'base_inventory_key': 'qwen-lm/qwen-3/Transformers/8b-base/1'}, {'name': 'll8v5', 'type': 'l24', 'path': '/kaggle/input/datasets/lightningv08/ntl-olmo2-native24-final-inference-assets/llama31-prefix24', 'adapters': ['/kaggle/input/nlm-l24-adapters-v5/ll8v5_f0', '/kaggle/input/nlm-l24-adapters-v5/ll8v5_f1', '/kaggle/input/nlm-l24-adapters-v5/ll8v5_f2', '/kaggle/input/nlm-l24-adapters-v5/ll8v5_f3', '/kaggle/input/nlm-l24-adapters-v5/ll8v5_f4', '/kaggle/input/nlm-l24-adapters-v5/ll8v5'], 'keep': 24, 'layers': [2, 4, 6, 8, 12, 16, 20, 24], 'w': 1.0, 'bs': 16, 'max_len': 384, 'twin': 0.35, 'adapter_source_ref': 'dhruvtibarewal/nlm-l24-adapters-v5', 'adapter_source_version': 1, 'adapter_relative_paths': ['ll8v5_f0', 'll8v5_f1', 'll8v5_f2', 'll8v5_f3', 'll8v5_f4', 'll8v5'], 'base_source_ref': 'lightningv08/ntl-olmo2-native24-final-inference-assets', 'base_source_version': 15, 'base_inventory_key': 'lightningv08/ntl-olmo2-native24-final-inference-assets/15:llama31-prefix24', 'base_component': 'llama31-prefix24', 'base_provenance': {'source': 'NousResearch/Meta-Llama-3.1-8B', 'revision': '1f47e50cdbe801ad8a5174156ec3a0655108fb9f', 'original_model': 'meta-llama/Llama-3.1-8B', 'original_revision': 'd04e592bb4f6aa9cfee91e2e20afa771667e1d4b', 'upstream_file_digests_match': True, 'whole_original_shard_sha_verified': False, 'included_decoder_layers': [0, 1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14, 15, 16, 17, 18, 19, 20, 21, 22, 23], 'precision': 'BF16', 'tokenizer_unchanged': True, 'mirror_shard_sha256': {'model-00001-of-00004.safetensors': 'f8b9704ab09cdeb097aa4a0a24bca96f906eec36bad63ab495bc21475058601b', 'model-00002-of-00004.safetensors': 'c28b25e7541751056ee126627e007f8d4288319733285e9f7b17b9ff6eb313f0', 'model-00003-of-00004.safetensors': 'd8e9504dd4e4a146d484c52a97584ec14dac92237c46b064934af67a85e7d383', 'model-00004-of-00004.safetensors': 'e4486f35c040f683f7d790354f66c169c109eb9fa0954a4a35d7c458a108405d'}, 'original_shard_sha256': {'model-00001-of-00004.safetensors': 'f8b9704ab09cdeb097aa4a0a24bca96f906eec36bad63ab495bc21475058601b', 'model-00002-of-00004.safetensors': 'c28b25e7541751056ee126627e007f8d4288319733285e9f7b17b9ff6eb313f0', 'model-00003-of-00004.safetensors': 'd8e9504dd4e4a146d484c52a97584ec14dac92237c46b064934af67a85e7d383', 'model-00004-of-00004.safetensors': 'e4486f35c040f683f7d790354f66c169c109eb9fa0954a4a35d7c458a108405d'}, 'prefix_file_sha256': {'embedding.safetensors': '58e7279a0bb1b39f44e14ad98f9248b78cda0d641ee40812f5a3db57b0e6ee2d', 'norm.safetensors': '20c82fab359cf0c84b582ab7ac53a8777d39f2224437f042113c6deba612e328', 'layer0.safetensors': 'bbc12a9b32b09b4df7ac4da6a98331bfdaee5c5e9e6c7c5c77b484ab7f671f7a', 'layer1.safetensors': 'f594630c669c167dc7ccb46a49f8e37b178657eda4a43dbcd5c467b44d98ce1e', 'layer2.safetensors': '90344620c8f9649b8d0815da9034f2096eba76b01c73f79dbd2928af6567427e', 'layer3.safetensors': '3b56a55934c456f26915745036627bbb498ceb34c7a214a27c8bda8eec46dc14', 'layer4.safetensors': '2df985e4e42880027159597f789554c19757148a42c148fdcad2589f6e0001f9', 'layer5.safetensors': '33bf752449bbd32962b772959dfca0e2a1e74faa0cc3fc0c841d4e5cf659d3a5', 'layer6.safetensors': '217072f005b0a35b5aa9e8db870cc2a89cf68871e86a41e06974937964ef144a', 'layer7.safetensors': '3171b04608b87f75690f9ab0c592073e20b2952ef6e55b9537125403c5f468f5', 'layer8.safetensors': 'b93a2c91793eb5ff2cd0490fbcd6d6c8ffbbf7971240c69ac3d461f9de8dda9c', 'layer9.safetensors': '79856a7b026d98936b9e6b05454175e0d47ad56dca82cfa83ce3458a9a8647f2', 'layer10.safetensors': '1afde23f8f08d4e034835ba231aaefee9e9af88d400e63d59a30ef20bcc520d6', 'layer11.safetensors': '8208f8729ff518a102b48dfeaeddfd1e59b29c2406c6d689d130f2289b545deb', 'layer12.safetensors': 'f0073d57867778524a46feaf727f6b13a3687be653de4c929cbbe05e73694c4e', 'layer13.safetensors': '67dc5e369ceebb16224dbc3f24446f79d08ea1527ccda1118d15a4af5cb2faaf', 'layer14.safetensors': '027c9d6e822f80f8e0a33b8c704ebd4345bd5364cbd215497e1c9ebf1ab4f184', 'layer15.safetensors': 'abef739c73b66a0bc53ffff67b867d5891132af9870ae3da272c8f1e07336cad', 'layer16.safetensors': 'cc77e4f5c7cdbc677b3556b4fef81ac0cdeb410ad620d3e7546e6617fe744125', 'layer17.safetensors': '5a61cabd6338968914dff8e7bdbd1a31edb865a2e1f80cd9ae51bff6c43cc344', 'layer18.safetensors': '2a2480ba6723faafb898f20a3c18dc0b3367dc855ebae5cc3c010a8dce2915cb', 'layer19.safetensors': '211d199d763d380e42de2e20a8de26c4a8ef8a0c274ff8a0307cd315d29531f6', 'layer20.safetensors': 'f388d11175f719c39ecdab440eb047b0148d37872e34a41f308c3164af27aec6', 'layer21.safetensors': 'c1beb025db8d77f42ab2c2e1c2bc3fb982a4980201860d724ce8159eb1ccd9c0', 'layer22.safetensors': '09b34b807938ec13670e874196fb4ef6ad5cdca6c3ddfdea2d1d2d98ce6ac1f0', 'layer23.safetensors': 'f2d1a586fd87346f7aa75c04bf5f63cbc2b58e63af94c3ff2172ba134d019f73'}, 'provenance_metadata_sha256': '9dbbe333950aad86deeb5c016f2832f7a55ba35f2ef8bfb4edd018cb53d790b4', 'mirror_audit_sha256': 'e5a4335ee4ba78a9ce6bf1ff7bb2aac5705e49a0c07e9f880d5b1c070d4b7234', 'original_audit_sha256': '4f75491525935613227595292c9b6adcaf9f6b4fede52097cd40294d114da3db'}}, {'name': 'n14b', 'type': 'l24', 'path': '/kaggle/input/models/qwen-lm/qwen-3/transformers/14b-base/1', 'adapters': ['/kaggle/input/ntl-handwriting-blend/lora_qwen3_14b'], 'keep': 27, 'layers': [2, 4, 6, 9, 13, 17, 22, 27], 'w': 2.5, 'bs': 8, 'max_len': 447, 'twin': 0.35, 'no_final_norm': True, 'eos_id': 151643, 'checkpoint_meta_sha256': '293119872ff3f170ec85e8be02c80916e39048213de31cb45f8b84a2cf74d775', 'checkpoint_source_ref': 'nishantdahal/ntl-handwriting-blend/1:lora_qwen3_14b', 'base_inventory_key': 'qwen-lm/qwen-3/Transformers/14b-base/1'}, {'name': 'q25_14', 'type': 'l24', 'path': '/kaggle/input/models/qwen-lm/qwen2.5/transformers/14b/1', 'adapters': ['/kaggle/input/ntl-handwriting-blend/lora_qwen25_14b'], 'keep': 32, 'layers': [3, 5, 8, 11, 16, 21, 27, 32], 'w': 1.0, 'bs': 8, 'max_len': 447, 'twin': 0.35, 'no_final_norm': True, 'eos_id': 151643, 'checkpoint_meta_sha256': '1d04028b7fe22beda04f7125d09dd5bc9bee9c8524f721c0aa7bc61379fad6fc', 'checkpoint_source_ref': 'nishantdahal/ntl-handwriting-blend/1:lora_qwen25_14b', 'base_inventory_key': 'qwen-lm/qwen2.5/Transformers/14b/1'}, {'name': 'deb', 'type': 'enc', 'ckpt': '/kaggle/input/ntl-handwriting-blend/deberta_v3_large', 'w': 1.0, 'base_inventory_key': 'nishantdahal/ntl-handwriting-blend/1:deberta_v3_large/encoder'}], 'gates': {'fixed_attachments_verified': True, 'ntl660_cache_verified': True, 'all_stage_three_row_diagnostic_passed': True, 'semantic_cpu_fixtures_passed': True, 'source_660_reviewed': True}, 'trained_artifact_required_files': {'/kaggle/input/datasets/lightningv08/ntl-olmo2-native24-final-inference-assets/gsq120k-trained/gsq120k-inference/gsq/final.safetensors': 64605840, '/kaggle/input/datasets/lightningv08/ntl-olmo2-native24-final-inference-assets/gsq120k-trained/gsq120k-inference/gsq/final.json': 1354, '/kaggle/input/datasets/lightningv08/ntl-olmo2-native24-final-inference-assets/gsq120k-trained/gsq120k-inference/gsq/COMPLETE.json': 675, '/kaggle/input/datasets/lightningv08/ntl-olmo2-native24-final-inference-assets/gsq120k-trained/gsq120k-inference/svc4/metadata.json': 1726, '/kaggle/input/datasets/lightningv08/ntl-olmo2-native24-final-inference-assets/gsq120k-trained/gsq120k-inference/svc4/vectorizer-char.joblib': 2492784, '/kaggle/input/datasets/lightningv08/ntl-olmo2-native24-final-inference-assets/gsq120k-trained/gsq120k-inference/svc4/vectorizer-word.joblib': 2228923, '/kaggle/input/datasets/lightningv08/ntl-olmo2-native24-final-inference-assets/gsq120k-trained/gsq120k-inference/svc4/vectorizer-shape.joblib': 500046, '/kaggle/input/datasets/lightningv08/ntl-olmo2-native24-final-inference-assets/gsq120k-trained/gsq120k-inference/svc4/vectorizer-function.joblib': 538516, '/kaggle/input/datasets/lightningv08/ntl-olmo2-native24-final-inference-assets/gsq120k-trained/gsq120k-inference/svc4/linear_svc.joblib': 67801270, '/kaggle/input/datasets/lightningv08/ntl-olmo2-native24-final-inference-assets/gsq120k-trained/gsq120k-inference/gaussian/q14v5/parameters.npz': 42619056, '/kaggle/input/datasets/lightningv08/ntl-olmo2-native24-final-inference-assets/gsq120k-trained/gsq120k-inference/gaussian/q14v5/metadata.json': 73323, '/kaggle/input/datasets/lightningv08/ntl-olmo2-native24-final-inference-assets/gsq120k-trained/gsq120k-inference/gaussian/q8v5/parameters.npz': 34099376, '/kaggle/input/datasets/lightningv08/ntl-olmo2-native24-final-inference-assets/gsq120k-trained/gsq120k-inference/gaussian/q8v5/metadata.json': 73275}, 'v5_binding_evidence': {'target_notebook_id': 140781577, 'asset_ref': 'dhruvtibarewal/nlm-l24-adapters-v5/1', 'root': '/kaggle/input/nlm-l24-adapters-v5', 'actual_target_root_and_version_verified': True}, 'trained_artifact_contract': {'hub_ref': 'lightningv08/ntl-olmo2-native24-final-inference-assets/16', 'final_sha256': '979758c31ec396c29272b411bd9a7d5ee1a920814932813dba9a109f03a11c33', 'training_contract_hash': '2f85a81aada2cf3fd908aa862734af51d68f139574c9e348284fdf5594381828', 'rows_seen': {'external': 119899, 'official': 5398}, 'hub_ready_verified': True, 'new13_paths_sizes_verified': True, 'old_retention_verified_in_v16': True}, 'device_max_memory': {'0': '13GiB', '1': '13GiB'}, 'base_inventories': {'qwen-lm/qwen-3/Transformers/14b-base/1': {'README.md': 2854, 'config.json': 729, 'generation_config.json': 138, 'merges.txt': 1671853, 'model-00001-of-00008.safetensors': 3841788544, 'model-00002-of-00008.safetensors': 3963750816, 'model-00003-of-00008.safetensors': 3963750880, 'model-00004-of-00008.safetensors': 3963750880, 'model-00005-of-00008.safetensors': 3963750880, 'model-00006-of-00008.safetensors': 3963750880, 'model-00007-of-00008.safetensors': 3963750880, 'model-00008-of-00008.safetensors': 1912371880, 'model.safetensors.index.json': 36514, 'tokenizer.json': 7031645, 'tokenizer_config.json': 9675, 'vocab.json': 2776833}, 'qwen-lm/qwen-3/Transformers/8b-base/1': {'README.md': 2851, 'config.json': 729, 'generation_config.json': 138, 'merges.txt': 1671853, 'model-00001-of-00005.safetensors': 3996250744, 'model-00002-of-00005.safetensors': 3993160032, 'model-00003-of-00005.safetensors': 3959604768, 'model-00004-of-00005.safetensors': 3187841392, 'model-00005-of-00005.safetensors': 1244659840, 'model.safetensors.index.json': 32878, 'tokenizer.json': 7031645, 'tokenizer_config.json': 9675, 'vocab.json': 2776833}, 'qwen-lm/qwen2.5/Transformers/14b/1': {'.gitattributes': 1519, 'LICENSE': 11343, 'README.md': 3795, 'config.json': 664, 'generation_config.json': 138, 'merges.txt': 1671839, 'model-00001-of-00008.safetensors': 3885134104, 'model-00002-of-00008.safetensors': 3995327992, 'model-00003-of-00008.safetensors': 3995328064, 'model-00004-of-00008.safetensors': 3995348776, 'model-00005-of-00008.safetensors': 3979614448, 'model-00006-of-00008.safetensors': 3995328064, 'model-00007-of-00008.safetensors': 3995328064, 'model-00008-of-00008.safetensors': 1698724408, 'model.safetensors.index.json': 47472, 'tokenizer.json': 7031645, 'tokenizer_config.json': 7228, 'vocab.json': 2776833}, 'lightningv08/ntl-olmo2-native24-final-inference-assets/15:llama31-prefix24': {'LICENSE': 7627, 'NOTICE.txt': 1116, 'USE_POLICY.md': 4691, 'config.json': 826, 'embedding.safetensors': 1050673264, 'layer0.safetensors': 436225016, 'layer1.safetensors': 436225016, 'layer10.safetensors': 436225024, 'layer11.safetensors': 436225024, 'layer12.safetensors': 436225024, 'layer13.safetensors': 436225024, 'layer14.safetensors': 436225024, 'layer15.safetensors': 436225024, 'layer16.safetensors': 436225024, 'layer17.safetensors': 436225024, 'layer18.safetensors': 436225024, 'layer19.safetensors': 436225024, 'layer2.safetensors': 436225016, 'layer20.safetensors': 436225024, 'layer21.safetensors': 436225024, 'layer22.safetensors': 436225024, 'layer23.safetensors': 436225024, 'layer3.safetensors': 436225016, 'layer4.safetensors': 436225016, 'layer5.safetensors': 436225016, 'layer6.safetensors': 436225016, 'layer7.safetensors': 436225016, 'layer8.safetensors': 436225016, 'layer9.safetensors': 436225016, 'model.safetensors.index.json': 14132, 'norm.safetensors': 8280, 'provenance.json': 16747, 'special_tokens_map.json': 73, 'tokenizer.json': 9085658, 'tokenizer_config.json': 50500}, 'nishantdahal/ntl-handwriting-blend/1:deberta_v3_large/encoder': {'config.json': 945, 'model.safetensors': 868069608, 'tokenizer.json': 8339974, 'tokenizer_config.json': 561}, 'dhruvtibarewal/nlm-enc-v5/3:encoder': {'config.json': 2044, 'model.safetensors': 789580328, 'tokenizer.json': 3583326, 'tokenizer_config.json': 487}}, 'optional_member_asset_refs': ['dhruvtibarewal/nlm-enc-v5/3'], 'optional_member_required_files': {'/kaggle/input/nlm-enc-v5/encoder/config.json': 2044, '/kaggle/input/nlm-enc-v5/encoder/model.safetensors': 789580328, '/kaggle/input/nlm-enc-v5/encoder/tokenizer.json': 3583326, '/kaggle/input/nlm-enc-v5/encoder/tokenizer_config.json': 487, '/kaggle/input/nlm-enc-v5/head.safetensors': 73936, '/kaggle/input/nlm-enc-v5/meta.json': 396}, 'member_checkpoint_inventories': {'q14v5': [{'adapter_config.json': 1272, 'adapter_model.safetensors': 173456688, 'heads.pt': 6231923}, {'adapter_config.json': 1272, 'adapter_model.safetensors': 173456688, 'heads.pt': 6231923}, {'adapter_config.json': 1272, 'adapter_model.safetensors': 173456688, 'heads.pt': 6231923}, {'adapter_config.json': 1272, 'adapter_model.safetensors': 173456688, 'heads.pt': 6231923}, {'adapter_config.json': 1272, 'adapter_model.safetensors': 173456688, 'heads.pt': 6231923}, {'adapter_config.json': 1272, 'adapter_model.safetensors': 173456688, 'heads.pt': 6231923}], 'q8v5': [{'adapter_config.json': 1271, 'adapter_model.safetensors': 116434760, 'heads.pt': 4986739}, {'adapter_config.json': 1271, 'adapter_model.safetensors': 116434760, 'heads.pt': 4986739}, {'adapter_config.json': 1271, 'adapter_model.safetensors': 116434760, 'heads.pt': 4986739}, {'adapter_config.json': 1271, 'adapter_model.safetensors': 116434760, 'heads.pt': 4986739}, {'adapter_config.json': 1271, 'adapter_model.safetensors': 116434760, 'heads.pt': 4986739}, {'adapter_config.json': 1271, 'adapter_model.safetensors': 116434760, 'heads.pt': 4986739}], 'll8v5': [{'adapter_config.json': 1278, 'adapter_model.safetensors': 125871992, 'heads.pt': 4986739}, {'adapter_config.json': 1278, 'adapter_model.safetensors': 125871992, 'heads.pt': 4986739}, {'adapter_config.json': 1278, 'adapter_model.safetensors': 125871992, 'heads.pt': 4986739}, {'adapter_config.json': 1278, 'adapter_model.safetensors': 125871992, 'heads.pt': 4986739}, {'adapter_config.json': 1278, 'adapter_model.safetensors': 125871992, 'heads.pt': 4986739}, {'adapter_config.json': 1278, 'adapter_model.safetensors': 125871992, 'heads.pt': 4986739}], 'n14b': [{'adapter/adapter_config.json': 1380, 'adapter/adapter_model.safetensors': 173456688, 'heads.safetensors': 6227664, 'meta.json': 1015}], 'q25_14': [{'adapter/adapter_config.json': 1322, 'adapter/adapter_model.safetensors': 183558248, 'heads.safetensors': 6227664, 'meta.json': 1000}], 'deb': [{'head.safetensors': 73936, 'meta.json': 529}], 'modernbert': [{'head.safetensors': 73936, 'meta.json': 396}]}, 'scored_reference_script_version': 363008892, 'token_attention_q35_enabled': False, 'execution_mode': 'competition'}
EXPLICIT_PATHS = {'status': 'ALL_FIXED_BINDINGS_PROVEN', 'paths': {'train_csv': {'path': '/kaggle/input/competitions/name-that-llm/train.csv', 'kind': 'file', 'status': 'PROVEN'}, 'test_csv': {'path': '/kaggle/input/competitions/name-that-llm/test.csv', 'kind': 'file', 'status': 'PROVEN'}, 'pool': {'path': '/kaggle/input/datasets/nishantdahal/ntl-two-stage-lora/pool_v1.parquet', 'kind': 'file', 'status': 'PROVEN'}, 'main_lora8': {'path': '/kaggle/input/datasets/nishantdahal/ntl-two-stage-lora/lora8b_v1', 'kind': 'directory', 'status': 'PROVEN'}, 'main_lora14': {'path': '/kaggle/input/datasets/nishantdahal/ntl-two-stage-lora/lora14b_v1', 'kind': 'directory', 'status': 'PROVEN'}, 'qwen3_8b_base': {'path': '/kaggle/input/models/qwen-lm/qwen-3/transformers/8b-base/1', 'kind': 'directory', 'status': 'PROVEN'}, 'qwen3_14b_base': {'path': '/kaggle/input/models/qwen-lm/qwen-3/transformers/14b-base/1', 'kind': 'directory', 'status': 'PROVEN'}, 'qwen2_7b_base': {'path': '/kaggle/input/models/qwen-lm/qwen2.5/transformers/7b/1', 'kind': 'directory', 'status': 'PROVEN'}, 'qwen3_4b_base': {'path': '/kaggle/input/models/qwen-lm/qwen-3/transformers/4b-base/1', 'kind': 'directory', 'status': 'PROVEN'}, 'glm_base': {'path': '/kaggle/input/notebooks/lucasboesen/name-that-llm-glm4-9b-trunc25/glm4_9b_trunc25', 'kind': 'directory', 'status': 'PROVEN'}, 'deberta_base': {'path': '/kaggle/input/notebooks/lucasboesen/name-that-llm-extract-encoders2/deberta-v3-large', 'kind': 'directory', 'status': 'PROVEN'}, 'electra_base': {'path': '/kaggle/input/notebooks/lucasboesen/name-that-llm-electra-large-weights/electra-large-discriminator', 'kind': 'directory', 'status': 'PROVEN'}, 'desklib_base': {'path': '/kaggle/input/notebooks/lucasboesen/name-that-llm-desklib-weights/desklib-detector', 'kind': 'directory', 'status': 'PROVEN'}, 'lora24_seed7': {'path': '/kaggle/input/notebooks/lucasboesen/name-that-llm-train-lora24-s7/lora_out/job_final', 'kind': 'directory', 'status': 'PROVEN'}, 'lora24_claude': {'path': '/kaggle/input/datasets/zbigniewkirchner/ntl-qwen3-8b-lora-claude-head/final', 'kind': 'directory', 'status': 'PROVEN'}, 'lora20_q25_instyle': {'path': '/kaggle/input/notebooks/vnton37nguyn/ntl-train-lora-qwen25-instyle/lora_out/job_final', 'kind': 'directory', 'status': 'PROVEN'}, 'tokattn_q35_head': {'path': '/kaggle/input/notebooks/lucasboesen/name-that-llm-train-tokattn/tokattn_final.pt', 'kind': 'file', 'status': 'PROVEN'}, 'tokattn_deb_head': {'path': '/kaggle/input/notebooks/lucasboesen/name-that-llm-train-tokattn2/tokattn_debl23_final.pt', 'kind': 'file', 'status': 'PROVEN'}, 'q35_base': {'path': '/kaggle/input/datasets/bbucxi/qwen35-9b-safetensors', 'kind': 'directory', 'status': 'PROVEN'}, 'hub_root': {'path': '/kaggle/input/datasets/lightningv08/ntl-olmo2-native24-final-inference-assets/name-that-llm-artifacts', 'kind': 'directory', 'status': 'PROVEN'}, 'hub_manifest': {'path': '/kaggle/input/datasets/lightningv08/ntl-olmo2-native24-final-inference-assets/name-that-llm-artifacts/QWEN38_ADDITION_MANIFEST.json', 'kind': 'file', 'status': 'PROVEN'}, 'lora24_public': {'path': '/kaggle/input/datasets/lightningv08/ntl-olmo2-native24-final-inference-assets/notebook-inputs/notebook-inputs/ntl-lora24-fivefold-training/lora24/final', 'kind': 'directory', 'status': 'PROVEN'}, 'olmo_manifest': {'path': '/kaggle/input/datasets/lightningv08/ntl-olmo2-native24-final-inference-assets/name-that-llm-artifacts/ntl-olmo2-native24-final-inference-assets/asset_manifest.json', 'kind': 'file', 'status': 'PROVEN'}, 'olmo_base': {'path': '/kaggle/input/datasets/lightningv08/ntl-olmo2-native24-final-inference-assets/notebook-inputs/notebook-inputs/olmo2-7b-fp16-offline-weights/olmo2-fp16-base', 'kind': 'directory', 'status': 'PROVEN'}, 'smol_manifest': {'path': '/kaggle/input/datasets/lightningv08/ntl-olmo2-native24-final-inference-assets/name-that-llm-artifacts/ntl-smollm3-two-stage-final-inference-assets/final_inference_manifest.json', 'kind': 'file', 'status': 'PROVEN'}, 'smol_base_marker': {'path': '/kaggle/input/datasets/lightningv08/ntl-olmo2-native24-final-inference-assets/notebook-inputs/notebook-inputs/smollm3-3b-base-offline-weights/smollm3_staging_complete.json', 'kind': 'file', 'status': 'PROVEN'}, 'smol_base': {'path': '/kaggle/input/datasets/lightningv08/ntl-olmo2-native24-final-inference-assets/notebook-inputs/notebook-inputs/smollm3-3b-base-offline-weights/smollm3-base', 'kind': 'directory', 'status': 'PROVEN'}, 'mixed_manifest': {'path': '/kaggle/input/datasets/lightningv08/ntl-olmo2-native24-final-inference-assets/qwen38-mixed-tail8/name-that-llm-artifacts/qwen38-27b-mixed-tail8-text/manifest.json', 'kind': 'file', 'status': 'PROVEN'}, 'qwen_adapter': {'path': '/kaggle/input/datasets/lightningv08/ntl-olmo2-native24-final-inference-assets/name-that-llm-artifacts/qwen38-27b-lora-tail8/final.safetensors', 'kind': 'file', 'status': 'PROVEN'}, 'atlas_manifest': {'path': '/kaggle/input/ntl-atlas-qwen3-8b-v1/atlas_adapter/atlas_adapter/atlas_bundle.json', 'kind': 'file', 'status': 'PROVEN'}, 'main_adapters': {'status': 'PROVEN', 'path': '/kaggle/input/nlm-l24-adapters'}, 'llama31_8b_base': {'status': 'PROVEN', 'path': '/kaggle/input/datasets/lightningv08/ntl-olmo2-native24-final-inference-assets/llama31-prefix24'}, 'prefit': {'status': 'PROVEN', 'path': '/kaggle/input/datasets/lightningv08/ntl-olmo2-native24-final-inference-assets/ntl611-prefit'}}, 'lists': {'qwen_wheels': {'paths': ['/kaggle/input/datasets/lightningv08/ntl-olmo2-native24-final-inference-assets/name-that-llm-artifacts/qwen38-offline-linux/accelerate-1.15.0-py3-none-any.whl', '/kaggle/input/datasets/lightningv08/ntl-olmo2-native24-final-inference-assets/name-that-llm-artifacts/qwen38-offline-linux/bitsandbytes-0.50.2-py3-none-manylinux_2_24_x86_64.whl', '/kaggle/input/datasets/lightningv08/ntl-olmo2-native24-final-inference-assets/name-that-llm-artifacts/qwen38-offline-linux/huggingface_hub-1.32.0-py3-none-any.whl', '/kaggle/input/datasets/lightningv08/ntl-olmo2-native24-final-inference-assets/name-that-llm-artifacts/qwen38-offline-linux/safetensors-0.8.0-cp310-abi3-manylinux_2_17_x86_64.manylinux2014_x86_64.whl', '/kaggle/input/datasets/lightningv08/ntl-olmo2-native24-final-inference-assets/name-that-llm-artifacts/qwen38-offline-linux/tokenizers-0.22.2-cp39-abi3-manylinux_2_17_x86_64.manylinux2014_x86_64.whl', '/kaggle/input/datasets/lightningv08/ntl-olmo2-native24-final-inference-assets/name-that-llm-artifacts/qwen38-offline-linux/transformers-5.14.1-py3-none-any.whl'], 'status': 'PROVEN'}, 'ultrabert_folds': {'paths': ['/kaggle/input/datasets/pjmathematician/ultrabert-380m-ntl/fold0', '/kaggle/input/datasets/pjmathematician/ultrabert-380m-ntl/fold1', '/kaggle/input/datasets/pjmathematician/ultrabert-380m-ntl/fold2', '/kaggle/input/datasets/pjmathematician/ultrabert-380m-ntl/fold3', '/kaggle/input/datasets/pjmathematician/ultrabert-380m-ntl/fold4'], 'status': 'PROVEN'}, 'extra_claude': {'paths': ['/kaggle/input/datasets/seyitkaangunes/ntl-extra-api-passages/ntl_extra_passages_claude.csv'], 'status': 'PROVEN'}, 'extra_gen': {'paths': ['/kaggle/input/datasets/lucasboesen/ntl-free-proxy-generations/gen_passages.csv'], 'status': 'PROVEN'}, 'extra_generations': {'paths': ['/kaggle/input/datasets/abdullah1afify/ntl-generations/ntl_generations.csv'], 'status': 'PROVEN'}, 'extra_proxy': {'paths': ['/kaggle/input/datasets/dophuchung/ntl-proxy-generations/ntl_proxy_generations.csv'], 'status': 'PROVEN'}, 'extra_clean': {'paths': ['/kaggle/input/datasets/prvsiyan/prvsiyans-extended-name-that-llm-dataset/passages_clean.csv'], 'status': 'PROVEN'}, 'extra_generated': {'paths': [], 'status': 'PROVEN'}}, 'staging': {'/kaggle/input/datasets/bbucxi/qwen35-9b-safetensors': {'status': 'PROVEN', 'files': [{'name': 'config.json', 'bytes': 3126}, {'name': 'merges.txt', 'bytes': 3353259}, {'name': 'model.safetensors-00001-of-00004.safetensors', 'bytes': 5276436216}, {'name': 'model.safetensors-00002-of-00004.safetensors', 'bytes': 5335161512}, {'name': 'model.safetensors-00003-of-00004.safetensors', 'bytes': 5368717440}, {'name': 'model.safetensors-00004-of-00004.safetensors', 'bytes': 3325995712}, {'name': 'model.safetensors.index.json', 'bytes': 79657}, {'name': 'tokenizer.json', 'bytes': 12807982}, {'name': 'tokenizer_config.json', 'bytes': 16710}, {'name': 'vocab.json', 'bytes': 6722759}]}, '/kaggle/input/datasets/lightningv08/ntl-olmo2-native24-final-inference-assets/qwen38-mixed-tail8/name-that-llm-artifacts/qwen38-27b-mixed-tail8-text': {'status': 'PROVEN', 'files': [{'name': 'LICENSE', 'bytes': 11544}, {'name': 'README.md', 'bytes': 65249}, {'name': 'chat_template.jinja', 'bytes': 8952}, {'name': 'config.json', 'bytes': 51350}, {'name': 'generation_config.json', 'bytes': 202}, {'name': 'layers-0.safetensors', 'bytes': 383865448}, {'name': 'layers-1.safetensors', 'bytes': 383865448}, {'name': 'layers-10.safetensors', 'bytes': 383865472}, {'name': 'layers-11.safetensors', 'bytes': 372313760}, {'name': 'layers-12.safetensors', 'bytes': 383865472}, {'name': 'layers-13.safetensors', 'bytes': 383865472}, {'name': 'layers-14.safetensors', 'bytes': 383865472}, {'name': 'layers-15.safetensors', 'bytes': 372313760}, {'name': 'layers-16.safetensors', 'bytes': 383865472}, {'name': 'layers-17.safetensors', 'bytes': 383865472}, {'name': 'layers-18.safetensors', 'bytes': 383865472}, {'name': 'layers-19.safetensors', 'bytes': 372313760}, {'name': 'layers-2.safetensors', 'bytes': 383865448}, {'name': 'layers-20.safetensors', 'bytes': 383865472}, {'name': 'layers-21.safetensors', 'bytes': 383865472}, {'name': 'layers-22.safetensors', 'bytes': 383865472}, {'name': 'layers-23.safetensors', 'bytes': 372313760}, {'name': 'layers-24.safetensors', 'bytes': 383865472}, {'name': 'layers-25.safetensors', 'bytes': 383865472}, {'name': 'layers-26.safetensors', 'bytes': 383865472}, {'name': 'layers-27.safetensors', 'bytes': 372313760}, {'name': 'layers-28.safetensors', 'bytes': 383865472}, {'name': 'layers-29.safetensors', 'bytes': 383865472}, {'name': 'layers-3.safetensors', 'bytes': 372313744}, {'name': 'layers-30.safetensors', 'bytes': 383865472}, {'name': 'layers-31.safetensors', 'bytes': 372313760}, {'name': 'layers-32.safetensors', 'bytes': 383865472}, {'name': 'layers-33.safetensors', 'bytes': 383865472}, {'name': 'layers-34.safetensors', 'bytes': 383865472}, {'name': 'layers-35.safetensors', 'bytes': 372313760}, {'name': 'layers-36.safetensors', 'bytes': 383865472}, {'name': 'layers-37.safetensors', 'bytes': 383865472}, {'name': 'layers-38.safetensors', 'bytes': 383865472}, {'name': 'layers-39.safetensors', 'bytes': 372313760}, {'name': 'layers-4.safetensors', 'bytes': 383865448}, {'name': 'layers-40.safetensors', 'bytes': 383865472}, {'name': 'layers-41.safetensors', 'bytes': 383865472}, {'name': 'layers-42.safetensors', 'bytes': 383865472}, {'name': 'layers-43.safetensors', 'bytes': 372313760}, {'name': 'layers-44.safetensors', 'bytes': 383865472}, {'name': 'layers-45.safetensors', 'bytes': 383865472}, {'name': 'layers-46.safetensors', 'bytes': 383865472}, {'name': 'layers-47.safetensors', 'bytes': 372313760}, {'name': 'layers-48.safetensors', 'bytes': 383865472}, {'name': 'layers-49.safetensors', 'bytes': 383865472}, {'name': 'layers-5.safetensors', 'bytes': 383865448}, {'name': 'layers-50.safetensors', 'bytes': 383865472}, {'name': 'layers-51.safetensors', 'bytes': 372313760}, {'name': 'layers-52.safetensors', 'bytes': 383865472}, {'name': 'layers-53.safetensors', 'bytes': 383865472}, {'name': 'layers-54.safetensors', 'bytes': 383865472}, {'name': 'layers-55.safetensors', 'bytes': 372313760}, {'name': 'layers-6.safetensors', 'bytes': 383865448}, {'name': 'layers-7.safetensors', 'bytes': 372313744}, {'name': 'layers-8.safetensors', 'bytes': 383865448}, {'name': 'layers-9.safetensors', 'bytes': 383865448}, {'name': 'manifest.json', 'bytes': 151231}, {'name': 'merges.txt', 'bytes': 3353259}, {'name': 'model-00016-of-00018.safetensors', 'bytes': 3979564040}, {'name': 'model-00017-of-00018.safetensors', 'bytes': 2108759344}, {'name': 'model.safetensors.index.json', 'bytes': 110564}, {'name': 'text-outside.receipt.json', 'bytes': 707}, {'name': 'text-outside.safetensors', 'bytes': 2542797120}, {'name': 'tokenizer.json', 'bytes': 12809320}, {'name': 'tokenizer_config.json', 'bytes': 17928}, {'name': 'vocab.json', 'bytes': 6722759}]}, '/kaggle/input/datasets/lightningv08/ntl-olmo2-native24-final-inference-assets/notebook-inputs/notebook-inputs/smollm3-3b-base-offline-weights/smollm3-base': {'status': 'PROVEN', 'files': [{'name': 'README.md', 'bytes': 12340}, {'name': 'config.json', 'bytes': 943}, {'name': 'generation_config.json', 'bytes': 126}, {'name': 'model-00001-of-00002.safetensors', 'bytes': 4966315264}, {'name': 'model-00002-of-00002.safetensors', 'bytes': 1183919744}, {'name': 'model.safetensors.index.json', 'bytes': 26899}, {'name': 'special_tokens_map.json', 'bytes': 151}, {'name': 'tokenizer.json', 'bytes': 17208861}, {'name': 'tokenizer_config.json', 'bytes': 50405}]}, '/kaggle/input/notebooks/lucasboesen/name-that-llm-glm4-9b-trunc25/glm4_9b_trunc25': {'status': 'PROVEN', 'files': [{'name': 'chat_template.jinja', 'bytes': 944}, {'name': 'config.json', 'bytes': 786}, {'name': 'model.safetensors', 'bytes': 11440008912}, {'name': 'tokenizer.json', 'bytes': 19966496}, {'name': 'tokenizer_config.json', 'bytes': 396}, {'name': 'trunc_info.json', 'bytes': 73}]}, '/kaggle/input/models/qwen-lm/qwen-3/transformers/8b-base/1': {'status': 'PROVEN', 'files': [{'name': 'README.md', 'bytes': 2851}, {'name': 'config.json', 'bytes': 729}, {'name': 'generation_config.json', 'bytes': 138}, {'name': 'merges.txt', 'bytes': 1671853}, {'name': 'model-00001-of-00005.safetensors', 'bytes': 3996250744}, {'name': 'model-00002-of-00005.safetensors', 'bytes': 3993160032}, {'name': 'model-00003-of-00005.safetensors', 'bytes': 3959604768}, {'name': 'model-00004-of-00005.safetensors', 'bytes': 3187841392}, {'name': 'model-00005-of-00005.safetensors', 'bytes': 1244659840}, {'name': 'model.safetensors.index.json', 'bytes': 32878}, {'name': 'tokenizer.json', 'bytes': 7031645}, {'name': 'tokenizer_config.json', 'bytes': 9675}, {'name': 'vocab.json', 'bytes': 2776833}]}, '/kaggle/input/models/qwen-lm/qwen2.5/transformers/7b/1': {'status': 'PROVEN', 'files': [{'name': '.gitattributes', 'bytes': 1519}, {'name': 'LICENSE', 'bytes': 11343}, {'name': 'README.md', 'bytes': 3819}, {'name': 'config.json', 'bytes': 686}, {'name': 'generation_config.json', 'bytes': 138}, {'name': 'merges.txt', 'bytes': 1671839}, {'name': 'model-00001-of-00004.safetensors', 'bytes': 3945441440}, {'name': 'model-00002-of-00004.safetensors', 'bytes': 3864726352}, {'name': 'model-00003-of-00004.safetensors', 'bytes': 3864726424}, {'name': 'model-00004-of-00004.safetensors', 'bytes': 3556377672}, {'name': 'model.safetensors.index.json', 'bytes': 27752}, {'name': 'tokenizer.json', 'bytes': 7031645}, {'name': 'tokenizer_config.json', 'bytes': 7228}, {'name': 'vocab.json', 'bytes': 2776833}]}, '/kaggle/input/models/qwen-lm/qwen-3/transformers/4b-base/1': {'status': 'PROVEN', 'files': [{'name': 'README.md', 'bytes': 2850}, {'name': 'config.json', 'bytes': 727}, {'name': 'generation_config.json', 'bytes': 138}, {'name': 'merges.txt', 'bytes': 1671853}, {'name': 'model-00001-of-00003.safetensors', 'bytes': 3957900840}, {'name': 'model-00002-of-00003.safetensors', 'bytes': 3987450520}, {'name': 'model-00003-of-00003.safetensors', 'bytes': 99630640}, {'name': 'model.safetensors.index.json', 'bytes': 32819}, {'name': 'tokenizer.json', 'bytes': 7031645}, {'name': 'tokenizer_config.json', 'bytes': 9675}, {'name': 'vocab.json', 'bytes': 2776833}]}, '/kaggle/input/notebooks/lucasboesen/name-that-llm-extract-encoders2/deberta-v3-large': {'status': 'PROVEN', 'files': [{'name': 'config.json', 'bytes': 907}, {'name': 'model.safetensors', 'bytes': 1736094384}, {'name': 'tokenizer.json', 'bytes': 8339876}, {'name': 'tokenizer_config.json', 'bytes': 538}]}, '/kaggle/input/notebooks/lucasboesen/name-that-llm-electra-large-weights/electra-large-discriminator': {'status': 'PROVEN', 'files': [{'name': 'config.json', 'bytes': 862}, {'name': 'model.safetensors', 'bytes': 1336413848}, {'name': 'tokenizer.json', 'bytes': 711396}, {'name': 'tokenizer_config.json', 'bytes': 351}]}, '/kaggle/input/notebooks/lucasboesen/name-that-llm-desklib-weights/desklib-detector': {'status': 'PROVEN', 'files': [{'name': 'config.json', 'bytes': 905}, {'name': 'model.safetensors', 'bytes': 1736094384}, {'name': 'tokenizer.json', 'bytes': 8339875}, {'name': 'tokenizer_config.json', 'bytes': 532}]}, '/kaggle/input/datasets/lightningv08/ntl-olmo2-native24-final-inference-assets/notebook-inputs/notebook-inputs/ntl-lora24-fivefold-training/lora24/final': {'status': 'PROVEN', 'files': [{'name': 'COMPLETE.json', 'bytes': 1310}, {'name': 'adapter/README.md', 'bytes': 5240}, {'name': 'adapter/adapter_config.json', 'bytes': 1288}, {'name': 'adapter/adapter_model.safetensors', 'bytes': 116434760}, {'name': 'anchor_features.npz', 'bytes': 367855}, {'name': 'base_config/config.json', 'bytes': 1332}, {'name': 'heads.safetensors', 'bytes': 4982480}, {'name': 'readouts.npz', 'bytes': 4377889}, {'name': 'tokenizer/chat_template.jinja', 'bytes': 4116}, {'name': 'tokenizer/tokenizer.json', 'bytes': 11422748}, {'name': 'tokenizer/tokenizer_config.json', 'bytes': 692}, {'name': 'training_report.json', 'bytes': 542527}, {'name': 'validation_features.npz', 'bytes': 1091}]}, '/kaggle/input/notebooks/lucasboesen/name-that-llm-train-lora24-s7/lora_out/job_final': {'status': 'PROVEN', 'files': [{'name': 'COMPLETE.json', 'bytes': 104}, {'name': 'adapter/README.md', 'bytes': 5240}, {'name': 'adapter/adapter_config.json', 'bytes': 1288}, {'name': 'adapter/adapter_model.safetensors', 'bytes': 116434760}, {'name': 'norms.safetensors', 'bytes': 263384}, {'name': 'readouts.npz', 'bytes': 4378243}, {'name': 'tokenizer/chat_template.jinja', 'bytes': 4116}, {'name': 'tokenizer/tokenizer.json', 'bytes': 11422748}, {'name': 'tokenizer/tokenizer_config.json', 'bytes': 693}, {'name': 'valid_features.npz', 'bytes': 183638}]}, '/kaggle/input/datasets/zbigniewkirchner/ntl-qwen3-8b-lora-claude-head/final': {'status': 'PROVEN', 'files': [{'name': 'COMPLETE.json', 'bytes': 44}, {'name': 'adapter/README.md', 'bytes': 5240}, {'name': 'adapter/adapter_config.json', 'bytes': 1288}, {'name': 'adapter/adapter_model.safetensors', 'bytes': 116434760}, {'name': 'anchor_features.npz', 'bytes': 367141}, {'name': 'heads.safetensors', 'bytes': 263384}, {'name': 'readouts.npz', 'bytes': 4985102}, {'name': 'tokenizer/chat_template.jinja', 'bytes': 4116}, {'name': 'tokenizer/tokenizer.json', 'bytes': 11422748}, {'name': 'tokenizer/tokenizer_config.json', 'bytes': 693}]}, '/kaggle/input/notebooks/vnton37nguyn/ntl-train-lora-qwen25-instyle/lora_out/job_final': {'status': 'PROVEN', 'files': [{'name': 'COMPLETE.json', 'bytes': 88}, {'name': 'adapter/README.md', 'bytes': 5232}, {'name': 'adapter/adapter_config.json', 'bytes': 1284}, {'name': 'adapter/adapter_model.safetensors', 'bytes': 115378992}, {'name': 'norms.safetensors', 'bytes': 230616}, {'name': 'readouts.npz', 'bytes': 3829777}, {'name': 'tokenizer/chat_template.jinja', 'bytes': 2427}, {'name': 'tokenizer/tokenizer.json', 'bytes': 11421990}, {'name': 'tokenizer/tokenizer_config.json', 'bytes': 696}, {'name': 'valid_features.npz', 'bytes': 160912}]}, '/kaggle/input/ntl-atlas-qwen3-8b-v1/atlas_adapter/atlas_adapter': {'status': 'PROVEN', 'files': [{'name': 'adapters.safetensors', 'bytes': 462749776}, {'name': 'atlas_bundle.json', 'bytes': 3011}, {'name': 'checkpoint.json', 'bytes': 1928}, {'name': 'inference-source-manifest.json', 'bytes': 712}, {'name': 'ntl_atlas/__init__.py', 'bytes': 72}, {'name': 'ntl_atlas/inference.py', 'bytes': 1505}, {'name': 'ntl_atlas/model.py', 'bytes': 8502}, {'name': 'ntl_rlcd/__init__.py', 'bytes': 83}, {'name': 'ntl_rlcd/data.py', 'bytes': 1873}, {'name': 'training-source-manifest.json', 'bytes': 886}]}, '/kaggle/input/notebooks/lucasboesen/name-that-llm-train-tokattn2/tokattn_debl23_final.pt': {'files': [{'bytes': 284355, 'name': 'tokattn_debl23_final.pt'}], 'status': 'PROVEN'}}}
(RUNTIME/'config.json').write_text(json.dumps(CONFIG),encoding='utf-8')
(RUNTIME/'legacy/explicit-paths.json').write_text(json.dumps(EXPLICIT_PATHS),encoding='utf-8')


## Reference grouping and CPU readouts

Content groups define the core centred and Gaussian heads. The three sparse score matrices and model score matrices each enter the active-vote divisor before the first probability mixture.

In [ ]:
%%writefile /kaggle/temp/ntl-gsq-v27-runtime/core_math.py
import re
import numpy as np
from scipy.optimize import linear_sum_assignment
from sklearn.feature_extraction.text import TfidfVectorizer

LABELS = ['claude-fable-5','claude-opus-4-6','claude-opus-4-8','claude-opus-5','claude-sonnet-5',
 'gemini-3.1-pro-preview','gemini-3.6-flash','gemini-3.7-flash','gemini-3.8-flash','glm-5.3','glm-5.3-flash',
 'gpt-5.5','gpt-5.6-luna','gpt-5.6-sol','gpt-5.6-terra','gpt-6-astra','kimi-k3','qwen3.8-max']
GUARDS = [(' \u2014 ', {'claude-opus-4-6', 'claude-opus-5', 'glm-5.3', 'glm-5.3-flash'}),
          ('\u2019', {l for l in LABELS if not l.startswith(('claude', 'glm'))}),
          ('"', {'claude-fable-5', 'claude-opus-4-6', 'claude-opus-4-8', 'claude-opus-5', 'claude-sonnet-5', 'gemini-3.1-pro-preview', 'gemini-3.6-flash', 'gemini-3.7-flash', 'gemini-3.8-flash', 'glm-5.3', 'glm-5.3-flash', 'gpt-5.5', 'kimi-k3'}),
          ('\u201c', {'gemini-3.8-flash', 'gpt-5.5', 'gpt-5.6-luna', 'gpt-5.6-sol', 'gpt-5.6-terra', 'gpt-6-astra', 'kimi-k3', 'qwen3.8-max'}),
          ('\u2013', {'claude-opus-5', 'gemini-3.1-pro-preview', 'glm-5.3-flash', 'gpt-5.6-luna', 'gpt-5.6-sol', 'gpt-5.6-terra', 'kimi-k3'})]

def capped_agglo(S, cap=18, thr=0.02):
    """Average linkage (Lance-Williams), never merging past `cap` members; stop when best avg sim < thr."""
    n = len(S); A = S.astype(np.float32).copy(); np.fill_diagonal(A, -np.inf)
    size = np.ones(n, np.int32); alive = np.ones(n, bool); lab = np.arange(n)
    while True:
        ok = (size[:, None] + size[None, :] <= cap) & alive[:, None] & alive[None, :]
        M = np.where(ok, A, -np.inf); k = int(np.argmax(M)); i, j = divmod(k, n)
        if not np.isfinite(M[i, j]) or M[i, j] < thr: break
        new = (size[i] * A[i] + size[j] * A[j]) / (size[i] + size[j])
        A[i] = new; A[:, i] = new; A[i, i] = -np.inf; A[j] = -np.inf; A[:, j] = -np.inf
        size[i] += size[j]; alive[j] = False; lab[lab == j] = i
    return lab

def topic_groups(texts):
    try:
        v = TfidfVectorizer(sublinear_tf=True, stop_words='english', min_df=2, max_df=0.05)
        Xc = v.fit_transform(texts)
        return capped_agglo((Xc @ Xc.T).toarray(), 18, 0.15)
    except ValueError:
        return np.arange(len(texts))

def lsm(z):
    return z-z.max(1, keepdims=True)-np.log(np.exp(z-z.max(1, keepdims=True)).sum(1, keepdims=True))

# Hierarchical typography from mercurydust's scored V2 (script 363008892),
# extending Roman Tamrazov's Bayesian typography. Fit uses official train only.
TYPOGRAPHY_MARKS = [' — ', '—', '–', '’', '‘', '“', '”', '«', '»', '"',
                    ';', ':', '(', ')', '[', ']', '**', '`', '…', '•',
                    '\n', '\n\n', 'e.g.', 'i.e.', 'However,', 'Overall,',
                    'In conclusion', 'Therefore,', 'Moreover,', '\t',
                    ' - ', ' = ', '\n- ', '\n1.', '###', '```', '> ']


def typography_evidence(texts, train_texts, train_labels):
    train_labels = np.asarray(train_labels, dtype=np.int64)
    assert len(train_texts) == len(train_labels) and len(train_labels) > 0
    assert ((train_labels >= 0) & (train_labels < len(LABELS))).all()
    Xtr = np.asarray([[float(mark in text) for mark in TYPOGRAPHY_MARKS]
                      for text in train_texts], dtype=np.float32)
    Xte = np.asarray([[float(mark in text) for mark in TYPOGRAPHY_MARKS]
                      for text in texts], dtype=np.float32)
    freq = np.bincount(train_labels, minlength=len(LABELS)).astype(float)
    hits = np.asarray([Xtr[train_labels == k].sum(axis=0)
                       for k in range(len(LABELS))], dtype=np.float64)
    global_rate = (Xtr.sum(axis=0) + 2.0) / (len(Xtr) + 4.0)
    rate = (hits + 24.0 * global_rate[None, :]) / (freq[:, None] + 24.0)
    rate = np.clip(rate, 0.01, 0.99)
    global_rate = np.clip(global_rate, 0.01, 0.99)
    signal = np.sqrt(np.mean((rate - global_rate[None, :]) ** 2, axis=0))
    fw = signal / max(float(np.median(signal[signal > 0])) if np.any(signal > 0) else 1.0, 1e-6)
    fw = np.clip(fw, 0.25, 2.0)
    evidence = (Xte * fw[None, :]) @ np.log(rate).T + ((1.0 - Xte) * fw[None, :]) @ np.log1p(-rate).T
    evidence -= (((Xte * fw[None, :]) @ np.log(global_rate)) + (((1.0 - Xte) * fw[None, :]) @ np.log1p(-global_rate)))[:, None]
    evidence -= evidence.mean(axis=1, keepdims=True)
    return np.clip(1.5 * evidence / max(float(fw.sum()), 1e-6), -0.30, 0.30), Xte


def balance_logits(P, train_labels, strength=.30, target_mix=0.50):
    Q = P.copy()
    if len(P) >= 18:
        rho = float(np.clip(target_mix, 0.0, 1.0))
        pred_marg = np.clip(P.mean(0), 1e-6, None)
        pred_marg /= pred_marg.sum()
        train_prior = np.bincount(train_labels, minlength=18).astype(np.float64) + 1.0
        train_prior /= train_prior.sum()
        target = (1.0 - rho) * pred_marg + rho * train_prior
        target = np.clip(target, 1e-6, None)
        target /= target.sum()
        for _ in range(100):
            Q *= (len(Q) * target[None, :]) / np.clip(Q.sum(0, keepdims=True), 1e-12, None)
            Q /= np.clip(Q.sum(1, keepdims=True), 1e-12, None)
        total = (1 - strength) * np.log(P + 1e-12) + strength * np.log(Q + 1e-12)
    else:
        total = np.log(P+1e-12)
    return total


def finalize(P, texts, groups, *, train_texts, train_labels):
    total = balance_logits(P, train_labels)
    evidence, Xte = typography_evidence(texts, train_texts, train_labels)
    probabilities = np.exp(total - total.max(axis=1, keepdims=True))
    probabilities /= probabilities.sum(axis=1, keepdims=True)
    ordered = np.sort(probabilities, axis=1)
    margin = ordered[:, -1] - ordered[:, -2]
    confidence_gate = np.clip((0.60 - margin) / 0.60, 0., 1.)
    word_count = np.array([len(re.findall(r'\b\w+\b', str(text))) for text in texts], dtype=np.float32)
    length_gate = np.clip(np.sqrt(np.maximum(word_count, 1.0) / 80.0), 0.40, 1.0)
    coverage_gate = np.clip(np.sqrt((Xte.sum(axis=1) + 1.0) / 4.0), 0.45, 1.0)
    gate = confidence_gate * length_gate * coverage_gate
    # Preserve soft evidence: no final hard punctuation mask or Hungarian override.
    return total + gate[:, None] * evidence


In [ ]:
%%writefile /kaggle/temp/ntl-gsq-v27-runtime/cache_contract.py
"""Declared immutable identities; mount paths are operational bindings only."""


def semantic_spec(spec):
    assert spec['name'] in ('q14v5', 'q8v5')
    assert spec['adapter_source_ref'] == 'dhruvtibarewal/nlm-l24-adapters-v5'
    assert spec['adapter_source_version'] == 1
    expected = [spec['name'] + f'_f{i}' for i in range(5)] + [spec['name']]
    assert spec['adapter_relative_paths'] == expected and len(spec['adapters']) == 6
    keys = ['name', 'type', 'keep', 'layers', 'bs', 'max_len', 'w', 'twin', 'gauss',
            'adapter_source_ref', 'adapter_source_version', 'adapter_relative_paths',
            'base_source_ref', 'base_source_version']
    value = {key: spec[key] for key in keys}
    value.update(no_final_norm=bool(spec.get('no_final_norm', False)), eos_id=spec.get('eos_id'),
                 trunk_dtype='float16', overflow_rule='source-global-bfloat16-forward-then-float16',
                 feature_dtype='float32', pooled_first_token=False,
                 source_script_version=362013743,
                 chain_rule='six-ordered-inplace-fp16-merges-and-unmerges')
    return value


In [ ]:
%%writefile /kaggle/temp/ntl-gsq-v27-runtime/gauss.py
"""Shared-prompt Gaussian group head (ported, Apache-2.0, from public Kaggle notebooks by prvsiyan / kirilltym / sangam21).
Two-way model x = class mean + prompt effect + noise; exact (n<=3) or cavity-EM (n>3) injective group marginals."""
import itertools
import numpy as np
K = 18
DEFAULT = {"red": "wpca", "d": 256, "se_shrink": "lw", "sp_shrink": "lw", "lam_floor": 1e-4, "n_exact": 3, "inj": True,
           "iters": 5, "repel": True, "sp_scale": 1.0}

# =================================================================== primitives ====
def _lse(A, axis=None, keepdims=False):
    m = np.max(A, axis=axis, keepdims=True); m = np.where(np.isfinite(m), m, 0.0)
    out = np.log(np.sum(np.exp(A - m), axis=axis, keepdims=True)) + m
    if keepdims: return out
    return np.squeeze(out, axis=axis) if axis is not None else float(out.item())

def log_softmax(L): return L - _lse(L, axis=1, keepdims=True)

def _lw_shrinkage(X):
    """Ledoit-Wolf shrinkage coefficient for rows X that are already centred (sklearn formula, assume_centered)."""
    from sklearn.covariance import ledoit_wolf_shrinkage
    return float(ledoit_wolf_shrinkage(X, assume_centered=True))

def _shrunk(S, a):
    d = S.shape[0]; return (1 - a) * S + a * (np.trace(S) / d) * np.eye(d)

def _two_way(Z, y, pinv, G, iters=4):
    """additive fit Z = mu[y] + P[prompt] + E by alternating means (balanced design -> converges in 1-2 rounds)."""
    P = np.zeros((G, Z.shape[1])); cnt_p = np.bincount(pinv, minlength=G).astype(float)
    cls = np.unique(y); mu = np.zeros((K, Z.shape[1]))
    for _ in range(iters):
        R = Z - P[pinv]
        for c in cls: mu[c] = R[y == c].mean(0)
        R = Z - mu[y]; P = np.zeros_like(P); np.add.at(P, pinv, R); P /= cnt_p[:, None]
    return mu, P, cnt_p

def fit(X, y, prompts, cfg=None):
    """fit the shared-prompt Gaussian on training rows. X (N, D) raw features, y (N,) labels in 0..17, prompts (N,) group ids.
    Returns a params dict (numpy arrays + floats); temperatures default to 1 (set by the caller's cross-fit)."""
    c = dict(DEFAULT); c.update(cfg or {})
    X = np.asarray(X, dtype=np.float64); y = np.asarray(y); N, D = X.shape
    pu, pinv = np.unique(np.asarray(prompts), return_inverse=True); G = len(pu)
    mx = X.mean(0); sx = X.std(0); sx = np.where(sx > 1e-8, sx, 1.0); Xs = (X - mx) / sx
    if c["red"] == "none":
        V = np.eye(D)
    else:
        if c["red"] == "wpca":
            Pm = np.zeros((G, D)); np.add.at(Pm, pinv, Xs); Pm /= np.bincount(pinv)[:, None]; R = Xs - Pm[pinv]
        elif c["red"] == "pca": R = Xs
        else: raise ValueError(c["red"])
        d = int(min(c["d"], N - G - 1, D))
        # economy SVD through the (N x N) Gram matrix (N << D for the real views)
        if N < D:
            w, U = np.linalg.eigh(R @ R.T); o = np.argsort(w)[::-1][:d]; w, U = np.clip(w[o], 1e-12, None), U[:, o]
            V = (R.T @ U) / np.sqrt(w)
        else:
            _, _, Vt = np.linalg.svd(R, full_matrices=False); V = Vt[:d].T
    Z = Xs @ V; d = Z.shape[1]
    mu, P, ng = _two_way(Z, y, pinv, G)
    E = Z - mu[y] - P[pinv]; dof = max(N - G - len(np.unique(y)) + 1, 1)
    Se_raw = E.T @ E / dof
    ae = _lw_shrinkage(E) if c["se_shrink"] == "lw" else float(c["se_shrink"]); Se = _shrunk(Se_raw, ae)
    Pc = P - P.mean(0); Sp_raw = Pc.T @ Pc / max(G - 1, 1) - np.mean(1.0 / ng) * Se_raw
    ap = _lw_shrinkage(Pc) if c["sp_shrink"] == "lw" else float(c["sp_shrink"])
    tr = max(np.trace(Sp_raw), 1e-8 * np.trace(Se)); Sp = (1 - ap) * Sp_raw + ap * (tr / d) * np.eye(d); Sp *= c["sp_scale"]
    # joint diagonalisation: A Se A' = I, A Sp A' = diag(lam)
    L = np.linalg.cholesky(Se); Li = np.linalg.inv(L); M = Li @ Sp @ Li.T; M = 0.5 * (M + M.T)
    lam, Q = np.linalg.eigh(M); lam = np.clip(lam, c["lam_floor"], None); A = Q.T @ Li
    W = V @ A.T
    return {"mx": mx, "sx": sx, "W": W, "m": mu @ A.T, "lam": lam, "T1": 1.0, "Tg": 1.0,
            "info": {"d": int(d), "a_e": round(ae, 4), "a_p": round(ap, 4), "lam_pct": np.round(np.percentile(lam, [5, 50, 95, 100]), 3).tolist()}, "cfg": c}

def transform(X, prm): return ((np.asarray(X, dtype=np.float64) - prm["mx"]) / prm["sx"]) @ prm["W"]

_MASK = {}
def _inj_mask(n, C):
    if (n, C) not in _MASK:
        grids = np.meshgrid(*[np.arange(C)] * n, indexing="ij"); bad = np.zeros([C] * n, bool)
        for i, j in itertools.combinations(range(n), 2): bad |= grids[i] == grids[j]
        _MASK[(n, C)] = bad
    return _MASK[(n, C)]

def singleton_ll(U, m, lam):
    """(n, C) log N(u; m_c, I + diag lam) up to a constant (p integrated out)."""
    R = U[:, None, :] - m[None]; return -0.5 * (R ** 2 / (1 + lam)).sum(-1)

def joint_terms(U, m, lam):
    """a (n, C) and B (n, n, C, C) of the exact joint log-likelihood of one group (constant in c dropped)."""
    n = len(U); gam = lam / (1 + n * lam); R = U[:, None, :] - m[None]
    a = -0.5 * (R ** 2 * (1 - gam)).sum(-1); G = R * np.sqrt(gam); B = np.einsum("icd,jed->ijce", G, G)
    return a, B

def joint_table(a, B):
    """full tuple table Q[c_1, ..., c_n] = sum_i a_i + sum_{i<j} B_ij."""
    n, C = a.shape; Q = np.zeros([C] * n)
    for i in range(n):
        sh = [1] * n; sh[i] = C; Q = Q + a[i].reshape(sh)
    for i, j in itertools.combinations(range(n), 2):
        sh = [1] * n; sh[i] = C; sh[j] = C; Q = Q + B[i, j].reshape(sh)
    return Q

def exact_marginals(Q, T=1.0, inj=True):
    """per-passage log marginal posteriors (n, C) from a tuple table Q (unnormalised log joint), tempered by 1/T."""
    n = Q.ndim; C = Q.shape[0]; Qt = Q / T
    if inj and n >= 2: Qt = np.where(_inj_mask(n, C), -np.inf, Qt)
    lz = _lse(Qt); out = np.empty((n, C))
    for i in range(n):
        ax = tuple(k for k in range(n) if k != i); out[i] = (_lse(Qt, axis=ax) if ax else Qt) - lz
    return out

def em_marginals(U, m, lam, T=1.0, iters=5, repel=False, q0=None):
    """cavity EM for one group (any n >= 2): returns (n, C) log q."""
    n = len(U); v = lam / (1 + (n - 1) * lam)
    L = singleton_ll(U, m, lam) / T if q0 is None else np.log(np.clip(q0, 1e-300, None))
    q = np.exp(log_softmax(L))
    for _ in range(iters):
        s = U - q @ m; S = s.sum(0); Pm = (S[None] - s) * v                  # cavity prompt-effect means (n, d)
        R = U[:, None, :] - Pm[:, None, :] - m[None]; L = -0.5 * (R ** 2 / (1 + v)).sum(-1) / T
        if repel:   # mean-field at-most-once: discourage classes the other members already hold
            lo = np.log(np.clip(1 - q, 1e-6, None)); L = L + (lo.sum(0)[None] - lo)
        q = np.exp(log_softmax(L))
    return log_softmax(L)

LAM_DIFF = 1e6
def lam_g(prm, cfg=None):
    """lam used for groups: the fitted one, or ~infinity in the paired-difference ablation (cfg 'diff'): then the joint likelihood
    only sees within-group differences (x_i - x_j around mu_a - mu_b with covariance 2 S_e; triples: centred observations vs
    centred prototypes) and the prompt term cancels exactly."""
    c = dict(prm.get("cfg", DEFAULT)); c.update(cfg or {})
    return np.full_like(prm["lam"], LAM_DIFF) if c.get("diff") else prm["lam"]

def group_logpost(U, prm, T1=None, Tg=None, cfg=None):
    c = dict(prm.get("cfg", DEFAULT)); c.update(cfg or {}); n = len(U); lg = lam_g(prm, cfg)
    T1 = prm["T1"] if T1 is None else T1; Tg = prm["Tg"] if Tg is None else Tg
    if n == 1: return log_softmax(singleton_ll(U, prm["m"], prm["lam"]) / T1)
    if n <= c["n_exact"]:
        a, B = joint_terms(U, prm["m"], lg); return exact_marginals(joint_table(a, B), Tg, c["inj"])
    return em_marginals(U, prm["m"], lg, Tg, c["iters"], c["repel"])

def scores(X_pool, lab, prm, T1=None, Tg=None, cfg=None, U=None):
    """(n, 18) log-probabilities for every pool row; lab = detected group labels of the pool rows (singletons allowed)."""
    U = transform(X_pool, prm) if U is None else U; lab = np.asarray(lab); n = len(U); out = np.empty((n, prm["m"].shape[0]))
    sizes = np.bincount(lab); s1 = sizes[lab] == 1
    T1 = prm["T1"] if T1 is None else T1
    if s1.any(): out[s1] = log_softmax(singleton_ll(U[s1], prm["m"], prm["lam"]) / T1)
    for g in np.where(sizes >= 2)[0]:
        ii = np.where(lab == g)[0]; out[ii] = group_logpost(U[ii], prm, T1, Tg, cfg)
    return out


In [ ]:
%%writefile /kaggle/temp/ntl-gsq-v27-runtime/svc4_runtime.py
"""NLM v27 SVC4 inference from fitted sklearn objects; no fit at inference."""
import json
import re
from pathlib import Path

import joblib
import numpy as np
from scipy.sparse import hstack

LABELS = [
    "claude-fable-5", "claude-opus-4-6", "claude-opus-4-8", "claude-opus-5", "claude-sonnet-5",
    "gemini-3.1-pro-preview", "gemini-3.6-flash", "gemini-3.7-flash", "gemini-3.8-flash",
    "glm-5.3", "glm-5.3-flash", "gpt-5.5", "gpt-5.6-luna", "gpt-5.6-sol", "gpt-5.6-terra",
    "gpt-6-astra", "kimi-k3", "qwen3.8-max",
]
FUNCTION_WORDS = set("a an the and or but if while although because as so than that which who whom whose this these those it its they them their theirs he him his she her hers we us our ours you your yours i me my mine in on at to from with without for of by over under between through into about is are was were be been being have has had do does did can could will would shall should may might must not no nor all any some each every both either neither other another such only also just very too more most less least here there when where why how what whether then now still yet even much many few little".split())
VIEW_SPECS = (
    ("char", 3, 5, 250000, 1.0),
    ("word", 1, 3, 180000, 0.7),
    ("shape", 2, 6, 60000, 0.5),
    ("function", 1, 3, 50000, 0.5),
)


def preprocess(view, text):
    if view == "shape":
        return re.sub(r"[^\W\d_]+", lambda match: "X" if match[0].isupper() else "x", text)
    if view == "function":
        return " ".join(word.lower() if word.lower() in FUNCTION_WORDS or not word.isalnum() else "CONTENT"
                        for word in re.findall(r"\w+|[^\w\s]", text))
    assert view in ("char", "word")
    return text


def vectorizer_options(spec):
    view, lo, hi, maximum, _ = spec
    analyzer = "char" if view == "shape" else "word" if view == "function" else view
    options = dict(analyzer=analyzer, ngram_range=(lo, hi), lowercase=False, sublinear_tf=True,
                   min_df=2, max_features=maximum, dtype=np.float32)
    if analyzer == "word":
        options["token_pattern"] = r"(?u)\b\w+\b|[^\w\s]"
    return options


def load_runtime(folder, *, require_verified=True):
    folder = Path(folder)
    metadata = json.loads((folder / "metadata.json").read_text(encoding="utf-8"))
    assert metadata["labels"] == LABELS and metadata["view_specs"] == [list(spec) for spec in VIEW_SPECS]
    if require_verified:
        assert metadata["status"] == "SVC4_PREFIT_ROUNDTRIP_VERIFIED"
    vectors = []
    for spec in VIEW_SPECS:
        vectorizer = joblib.load(folder / f"vectorizer-{spec[0]}.joblib")
        assert vectorizer.preprocessor is None and vectorizer.tokenizer is None
        assert all(vectorizer.get_params()[key] == value for key, value in vectorizer_options(spec).items())
        vectors.append(vectorizer)
    classifier = joblib.load(folder / "linear_svc.joblib")
    assert np.array_equal(classifier.classes_, np.arange(len(LABELS)))
    assert classifier.C == 0.5 and classifier.random_state == 42
    assert classifier.max_iter == 5000 and classifier.class_weight == "balanced"
    return metadata, vectors, classifier


def decision_function(texts, vectors, classifier):
    texts = list(texts)
    assert all(isinstance(text, str) for text in texts)
    blocks = []
    for spec, vectorizer in zip(VIEW_SPECS, vectors):
        view, _, _, _, weight = spec
        blocks.append(vectorizer.transform([preprocess(view, text) for text in texts]) * weight)
    scores = classifier.decision_function(hstack(blocks, format="csr"))
    assert scores.shape == (len(texts), len(LABELS)) and np.isfinite(scores).all()
    return np.asarray(scores)


def log_probability(scores, weight=1.5):
    scores = np.asarray(scores)
    shifted = scores - scores.max(1, keepdims=True)
    return (shifted - np.log(np.exp(shifted).sum(1, keepdims=True))) * weight


In [ ]:
%%writefile /kaggle/temp/ntl-gsq-v27-runtime/ntl660_readout.py
"""Notebook-visible, inference-only readout of numeric .660 CPU fit states.

The artifact directory contains JSON, NPZ and built-in sklearn vectorizers only.
No custom object is unpickled and no training occurs here.
"""
import json
import re
from pathlib import Path

import joblib
import numpy as np
from scipy.sparse import csr_matrix, diags, hstack

LABELS = sorted(['gpt-5.5', 'gpt-5.6-luna', 'gpt-5.6-sol', 'gpt-5.6-terra', 'gpt-6-astra',
                 'gemini-3.1-pro-preview', 'gemini-3.6-flash', 'gemini-3.7-flash', 'gemini-3.8-flash',
                 'claude-fable-5', 'claude-opus-4-6', 'claude-opus-4-8', 'claude-opus-5', 'claude-sonnet-5',
                 'qwen3.8-max', 'glm-5.3', 'glm-5.3-flash', 'kimi-k3'])
FUNCTION_WORDS = frozenset('a an the and or but if while although because as so than that which who whom whose this these those it its they them their theirs he him his she her hers we us our ours you your yours i me my mine in on at to from with without for of by over under between through into about is are was were be been being have has had do does did can could will would shall should may might must not no nor all any some each every both either neither other another such only also just very too more most less least here there when where why how what whether then now still yet even much many few little'.split())
VIEWS = [('char', 3, 5, 250000, 1.), ('word', 1, 3, 180000, .7),
         ('shape', 2, 6, 60000, .5), ('function', 1, 3, 50000, .5)]


def shape_view(text):
    return re.sub(r'[^\W\d_]+', lambda m: 'X' if m[0].isupper() else 'x', text)


def function_view(text):
    return ' '.join(t.lower() if t.lower() in FUNCTION_WORDS or not t.isalnum() else 'CONTENT'
                    for t in re.findall(r'\w+|[^\w\s]', text))


def prepare(texts, view, keep=()):
    if view == 'shape':
        return [shape_view(t) for t in texts]
    if view == 'function':
        return [function_view(t) for t in texts]
    if view == 'distort':
        keep = set(keep)
        return [re.sub(r"[A-Za-z’']+", lambda m: m[0] if m[0].lower() in keep else '*', t) for t in texts]
    return texts


def log_softmax(z):
    z = z - z.max(1, keepdims=True)
    return z - np.log(np.exp(z).sum(1, keepdims=True))


def softmax(z):
    z = z - z.max(1, keepdims=True)
    z = np.exp(z)
    return z / z.sum(1, keepdims=True)


def sparse_group_centre(X, groups, lo=2, hi=18):
    X = csr_matrix(X)
    _, inv, cnt = np.unique(np.asarray(groups), return_inverse=True, return_counts=True)
    keep = np.flatnonzero((cnt >= lo) & (cnt <= hi))
    if not len(keep):
        return X.copy()
    col = np.full(len(cnt), -1)
    col[keep] = np.arange(len(keep))
    rows = np.flatnonzero(col[inv] >= 0)
    M = csr_matrix((np.ones(len(rows)), (rows, col[inv][rows])), shape=(X.shape[0], len(keep)))
    means = diags(1.0 / cnt[keep]) @ (M.T @ X)
    return (X - M @ means).tocsr()


def style_groups(texts):
    # Exact .660 complementary stylegram grouping; no use of model groups here.
    from scipy.cluster.hierarchy import fcluster, linkage
    from scipy.spatial.distance import squareform
    from sklearn.feature_extraction.text import TfidfVectorizer
    if len(texts) < 2:
        return np.ones(len(texts), dtype=int)
    try:
        X = TfidfVectorizer(stop_words='english', sublinear_tf=True, max_features=100000,
                            dtype=np.float32).fit_transform(texts)
        D = np.clip(1.0 - (X @ X.T).toarray(), 0., 1.)
    except ValueError as exc:
        if 'empty vocabulary' not in str(exc):
            raise
        D = np.ones((len(texts), len(texts)), dtype=np.float32) - np.eye(len(texts), dtype=np.float32)
    med = max(float(np.median(D[np.triu_indices(len(texts), 1)])), 1e-6)
    D = np.asarray(D / med, dtype=np.float64).copy()
    np.fill_diagonal(D, 0.)
    return fcluster(linkage(squareform(D, checks=False), method='average'), .90, criterion='distance')


class Readout:
    def __init__(self, folder):
        self.folder = Path(folder)
        self.meta = json.loads((self.folder / 'metadata.json').read_text())
        if self.meta['labels'] != LABELS:
            raise ValueError('Unexpected canonical class order')
        self.vectorizers = [joblib.load(self.folder / e['file']) for e in self.meta['views']]
        # All these objects are vanilla sklearn TfidfVectorizer objects. Text
        # transformations live in this visible module, never in pickle callables.
        from sklearn.feature_extraction.text import TfidfVectorizer
        assert all(type(v) is TfidfVectorizer and not callable(v.analyzer) and
                   v.preprocessor is None and v.tokenizer is None for v in self.vectorizers)

    def matrix(self, texts):
        return hstack([v.transform(prepare(texts, e['view'], self.meta.get('keep', []))) * e['weight']
                       for v, e in zip(self.vectorizers, self.meta['views'])], format='csr')

    def raw(self, X, state='model'):
        with np.load(self.folder / (state + '.npz'), allow_pickle=False) as z:
            scores = np.asarray(X @ z['coef'].T) + z['intercept']
            classes = z['classes'].copy()
        # sklearn binary LinearSVC returns a single positive-class margin.
        if len(classes) == 2 and scores.shape[1] == 1:
            scores = np.column_stack([-scores[:, 0], scores[:, 0]])
        return scores, classes

    def scores(self, texts, state='model'):
        return self.raw(self.matrix(texts), state)


def core_votes(root, texts):
    """Three weighted log-prob matrices consumed as three active core votes.

    Weights are exactly svc4=12, tfidf=.25, distortion=.10. Core combines
    these with the model votes then divides by its active vote count, as in V2.
    """
    votes = {}
    for key, weight in [('svc4', 12.0), ('tfidf', .25), ('distort', .10)]:
        r = Readout(Path(root) / 'core' / key)
        scores, classes = r.scores(texts)
        assert classes.tolist() == list(range(len(LABELS)))
        votes[key] = log_softmax(scores) * weight
    return votes


def complementary_style(root, texts):
    """Unweighted stylegram probabilities and raw family gates for comp stack.

    Caller adds 1.516*log(clip(P,1e-9,1)) and consumes family gates in
    _apply_conditional_family_gate BEFORE writing complementary logits.
    Claude LoRA gate is a distinct model output, never fitted in this packet.
    """
    root = Path(root)
    r = Readout(root / 'comp' / 'stylegram')
    X = r.matrix(texts)
    d, classes = r.raw(X, 'plain')
    assert classes.tolist() == list(range(len(LABELS)))
    P = softmax(d)
    groups = style_groups(texts)
    sizes = np.bincount(groups)[groups]
    rows = (sizes >= 2) & (sizes <= 18)
    if rows.any():
        d, _ = r.raw(sparse_group_centre(X[rows], groups[rows]), 'twin')
        P[rows] = softmax(d)
    gates = {}
    for family in ['claude', 'gpt', 'gemini', 'glm']:
        g = Readout(root / 'comp' / 'gates' / family)
        d, classes = g.scores(texts)
        expected = [i for i, label in enumerate(LABELS) if label.startswith(family)]
        assert classes.tolist() == expected
        gates[family] = d
    return P, gates, groups


In [ ]:
%%writefile /kaggle/temp/ntl-gsq-v27-runtime/legacy/stage_assets.py
"""Fresh sequential /tmp copies of explicit preparation-inventoried assets.

No runtime directory enumeration, checkpoint search, cache reuse or path aliases.
This independent helper supports nested adapter/tokenizer/custom-code files;
the established core/UltraBERT/GSQ staging implementations remain unchanged.
"""
from contextlib import contextmanager
import json
from pathlib import Path, PurePosixPath
import shutil
import tempfile
import time

from asset_paths import asset_path, LEDGER


@contextmanager
def staged_legacy_asset(key, *, parent=False):
    bound = asset_path(key)
    source = bound.parent if parent else bound
    source_is_file = source.is_file()
    record = LEDGER['staging'][str(source)]
    assert record['status'] == 'PROVEN' and record['files'], ('Unverified staging inventory', key)
    assert source.resolve().is_relative_to(Path('/kaggle/input').resolve())
    files = {}
    for entry in record['files']:
        name, size = entry['name'], entry['bytes']
        relative = PurePosixPath(name)
        assert name and '\\' not in name and not relative.is_absolute()
        assert '..' not in relative.parts and '.' not in relative.parts
        assert relative.as_posix() == name and name not in files
        assert isinstance(size, int) and size > 0
        if source_is_file:
            assert name == source.name and len(record['files']) == 1
            original = source
        else:
            original = source.joinpath(*relative.parts)
            assert original.resolve().is_relative_to(source.resolve())
        assert original.is_file() and original.stat().st_size == size, (key, name, size)
        files[name] = (original, size)
    total = sum(size for _, size in files.values())
    root = Path('/tmp')
    assert shutil.disk_usage(root).free > total + 2 * 2**30, ('Insufficient /tmp staging space', key, total)
    label = ''.join(ch if ch.isalnum() else '-' for ch in key)
    destination = Path(tempfile.mkdtemp(prefix='ntl-legacy-' + label + '-', dir=root))
    assert destination.resolve().is_relative_to(root.resolve())
    started = time.monotonic()
    try:
        for name, (original, size) in files.items():
            target = destination.joinpath(*PurePosixPath(name).parts)
            assert target.resolve().is_relative_to(destination.resolve())
            target.parent.mkdir(parents=True, exist_ok=True)
            with original.open('rb') as reader, target.open('wb') as writer:
                while block := reader.read(16 * 1024 * 1024):
                    writer.write(block)
            assert target.stat().st_size == size, (key, name)
            print(json.dumps(dict(stage='legacy-staging', key=key, file=name, bytes=size,
                                  elapsed_seconds=round(time.monotonic() - started, 2))), flush=True)
        yield destination / source.name if source_is_file else destination
    finally:
        assert destination.resolve().is_relative_to(root.resolve())
        shutil.rmtree(destination)
        print(json.dumps(dict(stage='legacy-staging', key=key, event='STAGING_CLEANED')), flush=True)


In [ ]:
%%writefile /kaggle/temp/ntl-gsq-v27-runtime/core_readout.py
"""CPU-only V5 heads, Gaussian states and the four-view frozen SVC."""
import argparse
import json
from pathlib import Path

import numpy as np
import pandas as pd
from core_math import LABELS, lsm
import gauss
from ntl660_readout import core_votes
from cache_contract import semantic_spec


def gaussian_state(folder, spec):
    folder = Path(folder)
    meta = json.loads((folder / 'metadata.json').read_text())
    assert meta['labels'] == LABELS and meta['official_rows'] == 2699
    assert meta['source'] == 'nlm-v27-final-adapter-train-only'
    assert meta['semantic_spec'] == semantic_spec(spec) and meta['same_runtime_parity'] is True
    assert meta['source_script_version'] == 362013743
    assert meta['feature_rows_order_sha256'] and meta['adapter_sha256']
    assert meta['ordered_adapter_chain_sha256'] and len(meta['ordered_adapter_chain']) == 6
    assert meta['semantic_contract_sha256']
    assert [entry['relative_path'] for entry in meta['ordered_adapter_chain']] == spec['adapter_relative_paths']
    assert meta['branch']['gaussian_branch'] == 'fixed-no-test-redo' and meta['branch']['test_redo'] == 0
    prm = dict(meta['parameters'])
    with np.load(folder / 'parameters.npz', allow_pickle=False) as states:
        assert set(states.files) == {'mx', 'sx', 'W', 'm', 'lam'}
        prm.update({k: states[k] for k in states.files})
    assert prm['T1'] == prm['Tg'] == 3.0 and prm['cfg']['d'] == 128
    return prm


def member_scores(spec, folder, groups, gaussian_folder=None):
    """Match NLM v27's adapter averaging, order, centering and vote weights."""
    folder = Path(folder)
    plain, centered = 0.0, 0.0
    adapters = spec['adapters'] if spec['type'] == 'l24' else [spec['ckpt']]
    final_features = None
    for i, _ in enumerate(adapters):
        with np.load(folder / f'adapter-{i}.npz', allow_pickle=False) as data:
            assert data['labels'].tolist() == LABELS
            Z = data['features']
            assert len(Z) == len(groups) and np.isfinite(Z).all()
            logits = Z @ data['plain_weight'].T + data['plain_bias']
            plain = plain + lsm(logits) / len(adapters)
            if spec['type'] == 'l24':
                Zc = Z.copy()
                for g in np.unique(groups):
                    rows = groups == g
                    Zc[rows] -= Zc[rows].mean(0)
                centered = centered + lsm(Zc @ data['centered_weight'].T + data['centered_bias']) / len(adapters)
            if i == len(adapters) - 1:
                final_features = Z.copy()
    votes = {}
    if spec.get('gauss', 0) > 0:
        branch = json.loads((folder/'feature-metadata.json').read_text())
        if branch['gaussian_branch'] == 'source-test-conditioned':
            assert branch['test_redo'] > 0
            with np.load(folder/'gaussian-train-features.npz',allow_pickle=False) as training:
                assert len(training['features']) == 2699
                prm = gauss.fit(training['features'],training['y'],training['groups'],
                                dict(red='wpca',d=128,se_shrink='lw',sp_shrink='lw',lam_floor=1e-4,
                                     n_exact=3,inj=True,iters=5,repel=True,sp_scale=1.0))
            prm['T1'] = prm['Tg'] = 3.0
        else:
            assert branch['gaussian_branch'] == 'fixed-no-test-redo' and branch['test_redo'] == 0
            prm = gaussian_state(gaussian_folder, spec)
        sizes = np.bincount(groups)[groups]
        selected = (sizes >= 2) & (sizes <= 18)
        value = np.zeros((len(groups), 18))
        if selected.any():
            value[selected] = gauss.scores(final_features[selected], np.unique(groups[selected], return_inverse=True)[1], prm)
        votes[spec['name'] + '_gauss'] = value * spec['gauss']
    votes[spec['name']] = plain * spec['w']
    if spec['type'] == 'l24':
        rows = np.bincount(groups)[groups] >= 2
        votes[spec['name'] + '_twin'] = np.where(rows[:, None], centered, 0.0) * spec['w'] * spec['twin']
    return votes


def main():
    parser = argparse.ArgumentParser()
    for name in ['config', 'query', 'groups', 'features', 'out']:
        parser.add_argument('--' + name, required=True, type=Path)
    args = parser.parse_args()
    config = json.loads(args.config.read_text())
    frame = pd.read_csv(args.query, dtype={'id': str})
    groups = np.load(args.groups, allow_pickle=False)
    assert len(groups) == len(frame)
    votes = core_votes(config['paths']['ntl666_prefit'], frame.text.fillna('').tolist())
    assert set(votes) == {'svc4', 'tfidf', 'distort'}
    for spec in config['models']:
        votes.update(member_scores(spec, args.features / spec['name'], groups,
                                   config['paths'].get('gaussian_' + spec['name'])))
    assert all(v.shape == (len(frame), 18) and np.isfinite(v).all() for v in votes.values())
    # This is the number of nonzero vote matrices, not the sum of model weights.
    active = sum(abs(v).max() > 0 for v in votes.values())
    total = sum(votes.values()) / max(1, active)
    np.savez(args.out, ids=frame.id.to_numpy(dtype=str), labels=np.asarray(LABELS),
             logits=total, active_votes=np.asarray(active), **{'vote_' + k: v for k, v in votes.items()})


if __name__ == '__main__':
    main()


## V5 and two-stage feature extraction


In [ ]:
%%writefile /kaggle/temp/ntl-gsq-v27-runtime/stage_checkpoint_files.py
"""Explicit checkpoint staging; flat trees reuse the proven base staging helper."""
from contextlib import contextmanager
import json
from pathlib import Path, PurePosixPath
import shutil
import tempfile
import time

from stage_dataset_weights import staged_dataset_tree


def checked_relative_files(files):
    assert isinstance(files,dict) and files
    for name,size in files.items():
        assert isinstance(name,str) and name and '\\' not in name and ':' not in name
        relative=PurePosixPath(name)
        assert relative.parts and not relative.is_absolute() and str(relative)==name
        assert all(part not in ('','.', '..') for part in relative.parts)
        assert isinstance(size,int) and not isinstance(size,bool) and size>0
    return files


@contextmanager
def staged_checkpoint_tree(source, files, *, label, input_root='/kaggle/input', temp_root='/tmp'):
    checked_relative_files(files)
    source=Path(source);input_root=Path(input_root);temp_root=Path(temp_root)
    assert source.is_dir() and source.resolve().is_relative_to(input_root.resolve())
    assert label and label.replace('-','').replace('_','').isalnum()
    for name,size in files.items():
        path=source/name
        assert path.resolve().is_relative_to(source.resolve())
        assert path.is_file() and path.stat().st_size==size,(label,name,size)
    if all('/' not in name for name in files):
        with staged_dataset_tree(source,files,label=label,input_root=input_root,temp_root=temp_root) as local:
            yield local
        return
    total=sum(files.values())
    assert shutil.disk_usage(temp_root).free>total+2*2**30,('Insufficient /tmp disk for explicit checkpoint staging',label,total)
    destination=Path(tempfile.mkdtemp(prefix='ntl-stage-'+label+'-',dir=temp_root))
    assert destination.resolve().is_relative_to(temp_root.resolve())
    started=time.monotonic()
    try:
        for name,size in files.items():
            target=destination/name
            assert target.resolve().is_relative_to(destination.resolve())
            target.parent.mkdir(parents=True,exist_ok=True)
            with (source/name).open('rb') as reader,target.open('wb') as writer:
                while block:=reader.read(16*1024*1024):writer.write(block)
            assert target.stat().st_size==size
            print(json.dumps(dict(stage=label,file=name,bytes=size,elapsed_seconds=round(time.monotonic()-started,2))),flush=True)
        yield destination
    finally:
        assert destination.resolve().is_relative_to(temp_root.resolve())
        shutil.rmtree(destination)
        print(json.dumps(dict(stage=label,event='STAGING_CLEANED')),flush=True)


In [ ]:
%%writefile /kaggle/temp/ntl-gsq-v27-runtime/core_worker.py
"""NLM v27 feature extraction; trained heads are evaluated after this child exits."""
import argparse, gc, json, os, time
from pathlib import Path
import numpy as np
import pandas as pd
import torch
from transformers import AutoModel, AutoTokenizer
from core_math import LABELS
from stage_dataset_weights import staged_dataset_tree
from stage_checkpoint_files import staged_checkpoint_tree
from contextlib import ExitStack
torch.backends.cuda.matmul.allow_fp16_reduced_precision_reduction = False
def log(*items): print(*items, flush=True)

@torch.no_grad()
def forced_bf16_probe(model, tokenizer, token_rows, first, layers):
    report = dict(kind='forced-source-style-bf16-forward-after-fixed-features',
                  official_row_indices=list(range(len(token_rows))),
                  token_lengths=[len(row) for row in token_rows], rows=len(token_rows),
                  device_capabilities=[list(torch.cuda.get_device_capability(i)) for i in range(torch.cuda.device_count())],
                  support=False, finite=None, error=None, restore_error=None,
                  fixed_features_saved_before_probe=True, quality_claim=False)
    stage = 'cast-to-bfloat16'
    try:
        model.to(torch.bfloat16)
        stage = 'forward'
        batch = tokenizer.pad({'input_ids':token_rows}, return_tensors='pt').to(first)
        states = model(**batch,output_hidden_states=True,use_cache=False).hidden_states
        stage = 'finite-check'
        report['finite'] = bool(all(torch.isfinite(states[layer]).all().item() for layer in layers))
        report['support'] = True
        report['status'] = 'SUPPORTED_FINITE' if report['finite'] else 'SUPPORTED_NONFINITE'
        del states,batch
    except Exception as error:
        report['status'] = 'OPERATION_ERROR'
        report['error'] = dict(stage=stage,type=type(error).__name__,message=str(error)[:2000])
    finally:
        try:
            model.to(torch.float16)
        except Exception as error:
            report['restore_error'] = dict(type=type(error).__name__,message=str(error)[:2000])
    return report
@torch.no_grad()
def l24_features(spec, texts, output, train_final=False, max_memory=None, train_frame=None, probe_bf16=False):
    """LoRA24-recipe member(s): truncated base loaded ONCE; for each adapter dir in spec['adapters'] (or spec['adapter'])
    merge W += B A * s, compute masked mean-pooled multi-layer features, read plain (+ centred) heads, then unmerge.
    Returns log-probs averaged over adapters (plain, centred-or-None)."""
    from transformers import AutoConfig
    from safetensors.torch import load_file
    adapters = spec.get('adapters') or [spec['adapter']]
    tok = AutoTokenizer.from_pretrained(spec['path'])
    if tok.pad_token is None: tok.pad_token = tok.eos_token
    cfg = AutoConfig.from_pretrained(spec['path']); tc = cfg.text_config if hasattr(cfg, 'text_config') else cfg
    tc.num_hidden_layers = spec['keep']; tc.use_cache = False
    if getattr(tc, 'layer_types', None) is not None: tc.layer_types = tc.layer_types[:spec['keep']]
    placement = {} if max_memory is None else {'max_memory': {int(k):v for k,v in max_memory.items()}}
    m = AutoModel.from_pretrained(spec['path'], config=cfg, dtype=torch.float16, device_map='auto', **placement).eval()
    if spec.get('no_final_norm'): m.norm = torch.nn.Identity()
    mods = dict(m.named_modules()); first = next(m.parameters()).device; L = spec['layers']
    _eos = spec.get('eos_id')   # checkpoints trained with an end token appended (Nishant's) need the same id at inference
    _tk = lambda tx: [x + ([_eos] if _eos is not None else []) for x in tok(list(tx), truncation=True, max_length=spec.get('max_len', 384) - (1 if _eos is not None else 0))['input_ids']]
    enc = _tk(texts)
    order = np.argsort([len(x) for x in enc]); bs = spec.get('bs', 16)
    def lsm(z): return z - z.max(1, keepdims=True) - np.log(np.exp(z - z.max(1, keepdims=True)).sum(1, keepdims=True))
    OVF = [0]; test_redo = 0; train_redo = 0
    def featurize(enc_, norms, hid):
        order_ = np.argsort([len(x) for x in enc_]); Z_ = np.zeros((len(enc_), hid * len(L)), np.float32)
        for i in range(0, len(order_), bs):
            b = order_[i:i + bs]; Lm = max(len(enc_[j]) for j in b)
            ids = torch.full((len(b), Lm), tok.pad_token_id); att = torch.zeros((len(b), Lm), dtype=torch.long)
            for r, j in enumerate(b): ids[r, :len(enc_[j])] = torch.tensor(enc_[j]); att[r, :len(enc_[j])] = 1
            hs = m(input_ids=ids.to(first), attention_mask=att.to(first), output_hidden_states=True, use_cache=False).hidden_states
            if not all(torch.isfinite(hs[l]).all() for l in L):
                del hs
                m.to(torch.bfloat16); hs = m(input_ids=ids.to(first), attention_mask=att.to(first), output_hidden_states=True, use_cache=False).hidden_states; m.to(torch.float16); OVF[0] += 1
            mask = att.unsqueeze(-1).float(); mask[:, 0] = 0
            Z_[b] = torch.cat([nm((hs[l].float().cpu() * mask).sum(1) / mask.sum(1)) for l, nm in zip(L, norms)], -1).numpy()
            del hs
        return Z_
    for ai, ad in enumerate(adapters):
        adp = os.path.join(ad, 'adapter') if os.path.exists(os.path.join(ad, 'adapter', 'adapter_config.json')) else ad
        acfg = json.load(open(os.path.join(adp, 'adapter_config.json'))); scale = acfg['lora_alpha'] / acfg['r']
        sd = load_file(os.path.join(adp, 'adapter_model.safetensors')); deltas = []
        for k, v in sd.items():
            if '.lora_A.' not in k: continue
            name = k.replace('base_model.model.', '', 1).split('.lora_A.')[0]
            if name not in mods and ('model.' + name) in mods: name = 'model.' + name
            W = mods[name].weight; B = sd[k.replace('.lora_A.', '.lora_B.')]
            W += ((B.to(W.device, torch.float32) @ v.to(W.device, torch.float32)) * scale).to(W.dtype)
            deltas.append((W, B, v))   # keep only the low-rank factors; the dense delta is recomputed to unmerge
        if os.path.exists(os.path.join(ad, 'heads.pt')): H = torch.load(os.path.join(ad, 'heads.pt'), map_location='cpu')
        else: H = dict(load_file(os.path.join(ad, 'heads.safetensors')))
        if os.path.exists(os.path.join(ad, 'meta.json')):   # remap head rows from the checkpoint's label order to ours
            _lab = json.load(open(os.path.join(ad, 'meta.json')))['labels']; _perm = [_lab.index(l) for l in LABELS]
            for _p in ('plain', 'centered'): H[f'{_p}.weight'] = H[f'{_p}.weight'][_perm]; H[f'{_p}.bias'] = H[f'{_p}.bias'][_perm]
        hid = H['norms.0.weight'].shape[0]
        norms = [torch.nn.LayerNorm(hid) for _ in L]
        for i, nm in enumerate(norms): nm.load_state_dict({'weight': H[f'norms.{i}.weight'], 'bias': H[f'norms.{i}.bias']})
        if not train_final or ai == len(adapters)-1:
            Z = np.zeros((len(texts), hid * len(L)), np.float32)
            for i in range(0, len(order), bs):
                b = order[i:i + bs]; Lm = max(len(enc[j]) for j in b)
                ids = torch.full((len(b), Lm), tok.pad_token_id); att = torch.zeros((len(b), Lm), dtype=torch.long)
                for r, j in enumerate(b): ids[r, :len(enc[j])] = torch.tensor(enc[j]); att[r, :len(enc[j])] = 1
                hs = m(input_ids=ids.to(first), attention_mask=att.to(first), output_hidden_states=True, use_cache=False).hidden_states
                if not all(torch.isfinite(hs[l]).all() for l in L):   # fp16 overflow (un-normed deep states): redo this batch in bf16
                    del hs
                    m.to(torch.bfloat16); hs = m(input_ids=ids.to(first), attention_mask=att.to(first), output_hidden_states=True, use_cache=False).hidden_states; m.to(torch.float16); OVF[0] += 1
                mask = att.unsqueeze(-1).float(); mask[:, 0] = 0
                Z[b] = torch.cat([nm((hs[l].float().cpu() * mask).sum(1) / mask.sum(1)) for l, nm in zip(L, norms)], -1).numpy()
                del hs
            np.savez(output / f'adapter-{ai}.npz', features=Z, labels=np.asarray(LABELS),
                     plain_weight=H['plain.weight'].float().numpy(), plain_bias=H['plain.bias'].float().numpy(),
                     centered_weight=H['centered.weight'].float().numpy(), centered_bias=H['centered.bias'].float().numpy())
            test_redo = OVF[0]
            if not train_final and ai == len(adapters)-1 and spec.get('gauss',0)>0 and test_redo:
                assert train_frame is not None and len(train_frame)==2699
                train_tokens = _tk(train_frame.text.fillna('').values)
                train_features = featurize(train_tokens, norms, hid)
                train_redo = OVF[0]-test_redo
                y = train_frame.model.map({label:i for i,label in enumerate(LABELS)}).to_numpy()
                group_codes = np.unique(train_frame.prompt_id.values, return_inverse=True)[1]
                np.savez(output/'gaussian-train-features.npz', features=train_features, y=y, groups=group_codes)
            if train_final and probe_bf16:
                # All fixed official features are already closed on disk. This
                # destructive dtype-capability probe cannot contaminate them.
                report = forced_bf16_probe(m, tok, enc[:min(3,len(enc))], first, L)
                (output/'bf16-capability.json').write_text(json.dumps(report,indent=2))
        for W, B, A_ in deltas:   # unmerge before the next adapter
            W -= ((B.to(W.device, torch.float32) @ A_.to(W.device, torch.float32)) * scale).to(W.dtype)
        torch.cuda.empty_cache()
        del deltas, sd; gc.collect()
    del m, mods; gc.collect(); torch.cuda.empty_cache()
    return dict(test_redo=0 if train_final else test_redo, train_redo=OVF[0] if train_final else train_redo, gaussian_branch='source-test-conditioned' if not train_final and test_redo and spec.get('gauss',0)>0 else 'fixed-no-test-redo')

@torch.no_grad()
def encoder_features(spec, texts, output, encoder_base):
    from safetensors.torch import load_file
    checkpoint = Path(spec['ckpt'])
    meta = json.loads((checkpoint/'meta.json').read_text())
    permutation = [meta['labels'].index(label) for label in LABELS]
    heads = load_file(str(checkpoint/'head.safetensors'))
    W, b = heads['weight'].float().numpy()[permutation], heads['bias'].float().numpy()[permutation]
    tokenizer = AutoTokenizer.from_pretrained(encoder_base)
    model = AutoModel.from_pretrained(encoder_base, dtype=torch.float32).to('cuda:0').eval()
    ids = tokenizer(list(texts), truncation=True, max_length=meta['max_len'])['input_ids']
    order = np.argsort([len(row) for row in ids], kind='stable')
    Z = np.zeros((len(ids), model.config.hidden_size), np.float32)
    for start in range(0, len(order), 32):
        index = order[start:start+32]
        batch = tokenizer.pad({'input_ids':[ids[i] for i in index]}, return_tensors='pt').to('cuda:0')
        mask = batch['attention_mask'].unsqueeze(-1).float()
        Z[index] = ((model(**batch).last_hidden_state*mask).sum(1)/mask.sum(1).clamp(min=1)).cpu().numpy()
    np.savez(output/'adapter-0.npz', features=Z, plain_weight=W, plain_bias=b, labels=np.asarray(LABELS))

def main():
    parser = argparse.ArgumentParser()
    for name in ['config','query','out']:
        parser.add_argument('--'+name, type=Path, required=True)
    parser.add_argument('--member', type=int, required=True)
    parser.add_argument('--train-final', action='store_true')
    parser.add_argument('--probe-bf16', action='store_true')
    args = parser.parse_args()
    config = json.loads(args.config.read_text())
    spec = config['models'][args.member]
    assert not args.probe_bf16 or args.train_final, 'Capability probe is preparation-only'
    frame = pd.read_csv(args.query, dtype={'id':str})
    args.out.mkdir(parents=True, exist_ok=True)
    texts = frame.text.fillna('').tolist()
    # Inventory identities and byte sizes are operational bindings, independent
    # of the producer/consumer mount prefix and semantic Gaussian contract.
    inventory = config['base_inventories'][spec['base_inventory_key']]
    source_base = spec['path'] if spec['type']=='l24' else str(Path(spec['ckpt'])/'encoder')
    with staged_dataset_tree(source_base, inventory, label='core-'+spec['name']) as local_base:
        with ExitStack() as checkpoint_stack:
            # Inventories are an operational config overlay; source spec and
            # ordered relative identities remain intact in feature metadata.
            records = config['member_checkpoint_inventories'][spec['name']]
            sources = spec['adapters'] if spec['type']=='l24' else [spec['ckpt']]
            assert len(records)==len(sources)
            staged = []
            started = time.monotonic()
            for index, (checkpoint, files) in enumerate(zip(sources, records)):
                local = checkpoint_stack.enter_context(staged_checkpoint_tree(
                    checkpoint, files, label='core-'+spec['name']+'-ckpt'+str(index)))
                staged.append(str(local))
            print(json.dumps(dict(stage='CORE_CHECKPOINT_STAGING',member=spec['name'],
                                  checkpoints=len(staged),bytes=sum(sum(files.values()) for files in records),
                                  elapsed_seconds=time.monotonic()-started)),flush=True)
            feature_started = time.monotonic()
            if spec['type'] == 'l24':
                train_frame = pd.read_csv(config['paths']['train_csv']) if not args.train_final and spec.get('gauss',0)>0 else None
                runtime_spec = dict(spec, path=str(local_base), adapters=staged)
                branch = l24_features(runtime_spec, texts, args.out, args.train_final, config.get('device_max_memory', {'0':'13GiB','1':'13GiB'}), train_frame, args.probe_bf16)
            else:
                assert not args.train_final
                runtime_spec = dict(spec, ckpt=staged[0])
                encoder_features(runtime_spec, texts, args.out, str(local_base))
                branch = {}
            print(json.dumps(dict(stage='CORE_FEATURE_CALL_COMBINED',member=spec['name'],
                                  elapsed_seconds=time.monotonic()-feature_started,
                                  includes='model-load, adapter/head-load, merge/unmerge, forward, conditional-train-features; not separated')),flush=True)
            # Feature function frames returned; release model/tensor references
            # before checkpoint contexts, then the base context, are removed.
            gc.collect(); torch.cuda.empty_cache()
    (args.out/'feature-metadata.json').write_text(json.dumps(dict(spec=spec,
        ids=frame.id.astype(str).tolist(), labels=LABELS, rows=len(frame),
        train_final=args.train_final, source_script_version=362013743,
        dtype='float32-features/float16-trunk-with-original-bf16-redo',
        fp16_adapter_chain='all-six-merges-and-unmerges-in-original-order', **branch)))

if __name__ == '__main__': main()


## Complementary stack and family gates

The 15% complementary mixture consumes all four fitted style family gates and the Claude LoRA gate before saving complementary logits.

In [ ]:
%%writefile /kaggle/temp/ntl-gsq-v27-runtime/legacy/stage_dataset_weights.py
"""Sequential, explicit-file Dataset staging for local random reads."""
import json
import shutil
import tempfile
import time
from contextlib import contextmanager
from pathlib import Path

NF4_FILES = {'LICENSE': 11544, 'README.md': 27108, 'chat_template.jinja': 8952, 'config.json': 4222, 'generation_config.json': 202, 'merges.txt': 3353259, 'model-00001-of-00007.safetensors': 2542796928, 'model-00002-of-00007.safetensors': 3967729339, 'model-00003-of-00007.safetensors': 3973498949, 'model-00004-of-00007.safetensors': 3992447326, 'model-00005-of-00007.safetensors': 4000148996, 'model-00006-of-00007.safetensors': 98537568, 'model-00007-of-00007.safetensors': 849400424, 'model.safetensors.index.json': 396308, 'tokenizer.json': 12809320, 'tokenizer_config.json': 17928, 'vocab.json': 6722759}
ULTRA_FOLD_FILES = {0: {'config.json': 2739, 'configuration_ultrabert.py': 2209, 'model.safetensors': 755137636, 'modeling_ultrabert.py': 30581, 'tokenizer.json': 3523731, 'tokenizer_config.json': 814}, 1: {'config.json': 2739, 'configuration_ultrabert.py': 2209, 'model.safetensors': 755137636, 'modeling_ultrabert.py': 30581, 'tokenizer.json': 3523731, 'tokenizer_config.json': 814}, 2: {'config.json': 2739, 'configuration_ultrabert.py': 2209, 'model.safetensors': 755137636, 'modeling_ultrabert.py': 30581, 'tokenizer.json': 3523731, 'tokenizer_config.json': 814}, 3: {'config.json': 2739, 'configuration_ultrabert.py': 2209, 'model.safetensors': 755137636, 'modeling_ultrabert.py': 30581, 'tokenizer.json': 3523731, 'tokenizer_config.json': 814}, 4: {'config.json': 2739, 'configuration_ultrabert.py': 2209, 'model.safetensors': 755137636, 'modeling_ultrabert.py': 30581, 'tokenizer.json': 3523731, 'tokenizer_config.json': 814}}

@contextmanager
def staged_dataset_tree(source, files, *, label, input_root='/kaggle/input', temp_root='/tmp'):
    source = Path(source)
    input_root = Path(input_root)
    temp_root = Path(temp_root)
    assert source.is_dir() and source.resolve().is_relative_to(input_root.resolve())
    assert label and label.replace('-', '').isalnum()
    assert files and all('/' not in name and '..' not in name and size > 0
                         for name, size in files.items())
    total = sum(files.values())
    for name, size in files.items():
        item = source / name
        assert item.is_file() and item.stat().st_size == size, (label, name, size)
    free = shutil.disk_usage(temp_root).free
    assert free > total + 2 * 2**30, ('Insufficient /tmp disk for explicit staging', label, total, free)
    destination = Path(tempfile.mkdtemp(prefix='ntl-stage-' + label + '-', dir=temp_root))
    assert destination.resolve().is_relative_to(temp_root.resolve())
    start = time.monotonic()
    try:
        for name, size in files.items():
            target = destination / name
            with (source / name).open('rb') as reader, target.open('wb') as writer:
                while block := reader.read(16 * 1024 * 1024):
                    writer.write(block)
            assert target.stat().st_size == size
            print(json.dumps({'stage': label, 'file': name, 'bytes': size,
                              'elapsed_seconds': round(time.monotonic() - start, 2)}), flush=True)
        yield destination
    finally:
        assert destination.resolve().is_relative_to(temp_root.resolve())
        shutil.rmtree(destination)
        print(json.dumps({'stage': label, 'event': 'STAGING_CLEANED'}), flush=True)


In [ ]:
%%writefile /kaggle/temp/ntl-gsq-v27-runtime/legacy/asset_paths.py
"""Concrete attachment bindings; no discovery, aliases or fallback paths."""
import json
from pathlib import Path

HERE = Path(__file__).resolve().parent
LEDGER = json.loads((HERE / 'explicit-paths.json').read_text(encoding='utf-8'))


def asset_path(key):
    record = LEDGER['paths'][key]
    assert record['status'] == 'PROVEN', ('Unproven attachment binding', key, record)
    value = record['path']
    assert isinstance(value, str) and value.startswith('/kaggle/input/') and '*' not in value
    path = Path(value)
    assert path.exists(), ('Required fixed attachment is missing', key, value)
    return path


def asset_paths(key):
    record = LEDGER['lists'][key]
    assert record['status'] == 'PROVEN', ('Unproven fixed attachment list', key, record)
    paths = [Path(value) for value in record['paths']]
    for path in paths:
        assert str(path).startswith('/kaggle/input/') and '*' not in str(path)
        assert path.exists(), ('Required fixed attachment is missing', key, path)
    return paths


def stage_inventory(source):
    # Direct lookup of two recorded checkpoint inventories, not a directory scan.
    record = LEDGER['staging'][str(source)]
    assert record['status'] == 'PROVEN' and record['files']
    for item in record['files']:
        path = source / item['name']
        assert path.is_relative_to(source) and path.is_file()
        assert path.stat().st_size == item['bytes'], path
    return record['files']


In [ ]:
%%writefile /kaggle/temp/ntl-gsq-v27-runtime/legacy/runtime_support.py
"""Single sequential copy of preparation-verified immutable checkpoints."""
import json, shutil, time
from asset_paths import stage_inventory
from pathlib import Path


def stage_checkpoint(source, *, input_root='/kaggle/input', cache_root='/kaggle/temp/ntl-checkpoint-cache', audit_path='/kaggle/working/q35-staging-audit.json'):
    bound_source = Path(source)
    source = bound_source.resolve()
    assert source.is_relative_to(Path(input_root).resolve()), source
    key = 'checkpoint'
    dest = Path(cache_root) / key
    marker = dest / '.stage-complete.json'
    files = [source / item['name'] for item in stage_inventory(bound_source)]
    sizes = {str(p.relative_to(source)): p.stat().st_size for p in files}
    assert sizes, ('Empty checkpoint directory', source)
    if marker.exists():
        record = json.loads(marker.read_text())
        assert record['source'] == str(source)
        assert {name: item['bytes'] for name, item in record['files'].items()} == sizes
        for name, item in record['files'].items():
            assert (dest / name).is_file() and (dest / name).stat().st_size == item['bytes']
        return str(dest)
    required = sum(sizes.values())
    dest.mkdir(parents=True, exist_ok=True)
    assert shutil.disk_usage(dest).free > required + 2 * 2**30, ('Insufficient staging space', required)
    start = time.monotonic()
    record = {'source': str(source), 'files': {}, 'validation': 'preflight hashes; runtime sizes only'}
    for p in files:
        rel = p.relative_to(source); target = dest / rel
        target.parent.mkdir(parents=True, exist_ok=True)
        with p.open('rb') as src, target.open('wb') as dst:
            while chunk := src.read(16 * 1024 * 1024):
                dst.write(chunk)
        assert target.stat().st_size == p.stat().st_size == sizes[str(rel)]
        record['files'][str(rel)] = {'bytes': sizes[str(rel)]}
        print('STAGED', rel, sizes[str(rel)], round(time.monotonic() - start, 1), flush=True)
    record['elapsed_seconds'] = time.monotonic() - start
    pending = marker.with_suffix('.tmp')
    pending.write_text(json.dumps(record, indent=2))
    pending.replace(marker)
    Path(audit_path).write_text(json.dumps(record, indent=2),encoding='utf-8')
    return str(dest)


In [ ]:
%%writefile /kaggle/temp/ntl-gsq-v27-runtime/legacy/reference_helpers.py
"""Definitions from Dhruv Tibarewal nlm-v21 v1 (script 360541849),
with upstream contributions by prvsiyan, Lucas Boesen, kirilltym and others.
No notebook execution, model loading or fitting at import time.
"""
import re
import numpy as np
import pandas as pd
from scipy.sparse import csr_matrix, diags, hstack
from sklearn.feature_extraction.text import TfidfVectorizer, ENGLISH_STOP_WORDS


LABELS = sorted(['gpt-5.5', 'gpt-5.6-luna', 'gpt-5.6-sol', 'gpt-5.6-terra', 'gpt-6-astra', 'gemini-3.1-pro-preview', 'gemini-3.6-flash', 'gemini-3.7-flash', 'gemini-3.8-flash', 'claude-fable-5', 'claude-opus-4-6', 'claude-opus-4-8', 'claude-opus-5', 'claude-sonnet-5', 'qwen3.8-max', 'glm-5.3', 'glm-5.3-flash', 'kimi-k3'])
K = len(LABELS)
LAB2I = {l: i for i, l in enumerate(LABELS)}
GROUP_THRESH, GROUP_LAYER, GROUP_MIN, GROUP_MAX = (0.6, 24, 2, 18)
STYLE_GROUP_THRESH = 0.9
W = {'stylegram': 1.516, 'stylometry_lgb': 0.063, 'ultrabert': 0.095, 'q3_8b': 0.0, 'q25_7b': 0.0, 'q3_4b': 0.0, 'glm9b': 0.0, 'debv3l': 0.05, 'electl': 0.032, 'desk': 0.132, 'lora24_public': 0.177, 'lora24_seed7': 0.113, 'lora24_claude': 0.177, 'lora20_q25_instyle': 0.17, 'tokattn_q35': 0.153, 'tokattn_deb': 0.185}
TWIN_W = {'q3_8b': 1.5 * 0.219, 'q25_7b': 1.5 * 0.141, 'q3_4b': 1.5 * 0.083, 'glm9b': 1.5 * 0.11, 'debv3l': 0.25}
GAUSS_HEADS = {'gauss_q3_8b_L22': dict(src='q3_8b', part='L22_mean', d=384, T1=2.351, Tg=2.306, w=TWIN_W['q3_8b']), 'gauss_glm9b_L8': dict(src='glm9b', part='L8_mean', d=384, T1=2.722, Tg=2.596, w=TWIN_W['glm9b']), 'gauss_q3_4b_L4': dict(src='q3_4b', part='L4_mean', d=384, T1=2.725, Tg=2.578, w=TWIN_W['q3_4b']), 'gauss_debv3l_L24': dict(src='debv3l', part='L24_mean', d=384, T1=3.112, Tg=3.069, w=TWIN_W['debv3l'])}
GAUSS_SETTING = dict(red='wpca', se_shrink='lw', sp_shrink='lw', lam_floor=0.0001, n_exact=3, inj=True, iters=5, repel=True, sp_scale=1.0)

def softmax_rows(S):
    S = S - S.max(1, keepdims=True)
    P = np.exp(S)
    return P / P.sum(1, keepdims=True)

def full_prob(P_part, classes):
    """Expand a (n, len(classes)) probability block to (n, K) on the global label order."""
    P = np.zeros((len(P_part), K))
    P[:, classes] = P_part
    return P

def group_centre_dense(X, groups):
    X = np.asarray(X, dtype=np.float32).copy()
    for g in np.unique(groups):
        i = groups == g
        X[i] -= X[i].mean(0)
    return X
FUNCTION_WORDS = frozenset('a an the and or but if while although because as so than that which who whom whose this these those it its they them their theirs he him his she her hers we us our ours you your yours i me my mine in on at to from with without for of by over under between through into about is are was were be been being have has had do does did can could will would shall should may might must not no nor all any some each every both either neither other another such only also just very too more most less least here there when where why how what whether then now still yet even much many few little'.split())
TOKEN_RE = re.compile('\\w+|[^\\w\\s]')

def shape_view(text):
    return re.sub('[^\\W\\d_]+', lambda m: 'X' if m[0].isupper() else 'x', text)

def function_view(text):
    return ' '.join((t.lower() if t.lower() in FUNCTION_WORDS or not t.isalnum() else 'CONTENT' for t in TOKEN_RE.findall(text)))


def sparse_group_centre(X, groups, lo=GROUP_MIN, hi=GROUP_MAX):
    """Subtract each group's mean row (sparse); groups outside [lo, hi] are left as they are."""
    X = csr_matrix(X)
    _, inv, cnt = np.unique(np.asarray(groups), return_inverse=True, return_counts=True)
    keep = np.flatnonzero((cnt >= lo) & (cnt <= hi))
    if not len(keep):
        return X.copy()
    col = np.full(len(cnt), -1)
    col[keep] = np.arange(len(keep))
    rows = np.flatnonzero(col[inv] >= 0)
    M = csr_matrix((np.ones(len(rows)), (rows, col[inv][rows])), shape=(X.shape[0], len(keep)))
    means = diags(1.0 / cnt[keep]) @ (M.T @ X)
    return (X - M @ means).tocsr()
PUNCT = list(',.;:!?()[]"\'-') + ['—', '–', '‘', '’', '“', '”', '…', '*', '/', '%']
STOPS = sorted(ENGLISH_STOP_WORDS)
SENT_RE, WORD_RE = (re.compile('(?<=[.!?])\\s+'), re.compile("[A-Za-z’']+"))

def stylometry(t):
    words = WORD_RE.findall(t)
    low = [w.lower() for w in words]
    nw, nc = (max(len(words), 1), max(len(t), 1))
    sents = [s for s in SENT_RE.split(t.strip()) if s]
    sl = np.array([len(WORD_RE.findall(s)) for s in sents] or [0], dtype=float)
    wl = np.array([len(w) for w in words] or [0], dtype=float)
    f = [np.log1p(nw), len(sents), sl.mean(), sl.std(), sl.max(), sl.min(), wl.mean(), wl.std(), (wl >= 9).mean(), len(set(low)) / nw, sum((c.isupper() for c in t)) / nc, sum((c.isdigit() for c in t)) / nc, sum((ord(c) > 127 for c in t)) / nc, t.count(', ') / nw, t.count(' and ') / nw, t.count(' but ') / nw, sum((1 for s in sents if s[:1].isupper() and s.split()[0].lower() in ('this', 'these', 'it', 'they'))) / max(len(sents), 1), sum((w.endswith('ly') for w in words)) / nw, sum((w.endswith('ing') for w in words)) / nw, sum((w.endswith('tion') for w in words)) / nw]
    f += [t.count(q) / nw for q in PUNCT]
    counts = pd.Series(low).value_counts() if low else pd.Series(dtype=float)
    f += [counts.get(s, 0) / nw for s in STOPS]
    return f
K = 18


In [ ]:
%%writefile /kaggle/temp/ntl-gsq-v27-runtime/legacy/portable.py
"""Small fitted-state serialization. Runtime contains transforms/prediction only."""
import json
from pathlib import Path
import numpy as np
from scipy.sparse import hstack
from sklearn.feature_extraction.text import TfidfVectorizer
from reference_helpers import shape_view, function_view

VIEWS = [('char', 3, 5, 250000, 1.), ('word', 1, 3, 180000, .7),
         ('shape', 2, 6, 60000, .5), ('function', 1, 3, 50000, .5)]

def prep_text(text, view):
    return shape_view(text) if view == 'shape' else function_view(text) if view == 'function' else text

def vectorizer_options(view, lo, hi, maxf, dtype):
    analyzer = 'char' if view == 'shape' else 'word' if view == 'function' else view
    opts = dict(analyzer=analyzer, ngram_range=(lo, hi), lowercase=False, sublinear_tf=True,
                min_df=2, max_features=maxf, dtype=dtype)
    if analyzer == 'word': opts['token_pattern'] = r'(?u)\b\w+\b|[^\w\s]'
    return opts


def load_vectorizer(path):
    path = Path(path)
    record = json.loads(path.with_suffix('.json').read_text(encoding='utf-8'))
    params = record['params']
    params['dtype'] = np.dtype(params['dtype']).type
    params['ngram_range'] = tuple(params['ngram_range'])
    vec = TfidfVectorizer(vocabulary=record['vocabulary'], **params)
    vec.idf_ = np.load(path.with_suffix('.idf.npy'), allow_pickle=False)
    return vec, record['view'], record['weight']

def style_matrix(folder, texts):
    matrices = []
    for i in range(4):
        vec, view, weight = load_vectorizer(Path(folder) / f'view{i}')
        matrices.append(vec.transform([prep_text(t, view) for t in texts]) * weight)
    return hstack(matrices, format='csr')


def linear_decision(path, X):
    with np.load(path, allow_pickle=False) as z:
        if 'mean' in z:
            # sklearn 1.9 casts moments before arithmetic (unlike older releases).
            X = np.array(X, copy=True)
            X -= z['mean'].astype(X.dtype)
            X /= z['scale'].astype(X.dtype)
        return np.asarray(X @ z['coef'].T) + z['intercept']

def linear_probability(path, X):
    # Match sklearn's in-place NumPy softmax, including float32 rounding.
    z = linear_decision(path, X)
    z -= np.max(z, axis=1, keepdims=True)
    np.exp(z, out=z)
    z /= np.sum(z, axis=1, keepdims=True)
    return z


def load_gaussian(path):
    with np.load(path, allow_pickle=False) as z: state = {k: z[k] for k in z.files}
    state.update(json.loads(Path(path).with_suffix('.json').read_text(encoding='utf-8')))
    return state


In [ ]:
%%writefile /kaggle/temp/ntl-gsq-v27-runtime/legacy/reference_gauss.py
"""Shared-prompt Gaussian group head (ported, Apache-2.0, from public Kaggle notebooks by prvsiyan / kirilltym / sangam21).
Two-way model x = class mean + prompt effect + noise; exact (n<=3) or cavity-EM (n>3) injective group marginals."""
import itertools
import numpy as np
K = 18
DEFAULT = {'red': 'wpca', 'd': 256, 'se_shrink': 'lw', 'sp_shrink': 'lw', 'lam_floor': 0.0001, 'n_exact': 3, 'inj': True, 'iters': 5, 'repel': True, 'sp_scale': 1.0}

def _lse(A, axis=None, keepdims=False):
    m = np.max(A, axis=axis, keepdims=True)
    m = np.where(np.isfinite(m), m, 0.0)
    out = np.log(np.sum(np.exp(A - m), axis=axis, keepdims=True)) + m
    if keepdims:
        return out
    return np.squeeze(out, axis=axis) if axis is not None else float(out.item())

def log_softmax(L):
    return L - _lse(L, axis=1, keepdims=True)

def transform(X, prm):
    return (np.asarray(X, dtype=np.float64) - prm['mx']) / prm['sx'] @ prm['W']
_MASK = {}

def _inj_mask(n, C):
    if (n, C) not in _MASK:
        grids = np.meshgrid(*[np.arange(C)] * n, indexing='ij')
        bad = np.zeros([C] * n, bool)
        for i, j in itertools.combinations(range(n), 2):
            bad |= grids[i] == grids[j]
        _MASK[n, C] = bad
    return _MASK[n, C]

def singleton_ll(U, m, lam):
    """(n, C) log N(u; m_c, I + diag lam) up to a constant (p integrated out)."""
    R = U[:, None, :] - m[None]
    return -0.5 * (R ** 2 / (1 + lam)).sum(-1)

def joint_terms(U, m, lam):
    """a (n, C) and B (n, n, C, C) of the exact joint log-likelihood of one group (constant in c dropped)."""
    n = len(U)
    gam = lam / (1 + n * lam)
    R = U[:, None, :] - m[None]
    a = -0.5 * (R ** 2 * (1 - gam)).sum(-1)
    G = R * np.sqrt(gam)
    B = np.einsum('icd,jed->ijce', G, G)
    return (a, B)

def joint_table(a, B):
    """full tuple table Q[c_1, ..., c_n] = sum_i a_i + sum_{i<j} B_ij."""
    n, C = a.shape
    Q = np.zeros([C] * n)
    for i in range(n):
        sh = [1] * n
        sh[i] = C
        Q = Q + a[i].reshape(sh)
    for i, j in itertools.combinations(range(n), 2):
        sh = [1] * n
        sh[i] = C
        sh[j] = C
        Q = Q + B[i, j].reshape(sh)
    return Q

def exact_marginals(Q, T=1.0, inj=True):
    """per-passage log marginal posteriors (n, C) from a tuple table Q (unnormalised log joint), tempered by 1/T."""
    n = Q.ndim
    C = Q.shape[0]
    Qt = Q / T
    if inj and n >= 2:
        Qt = np.where(_inj_mask(n, C), -np.inf, Qt)
    lz = _lse(Qt)
    out = np.empty((n, C))
    for i in range(n):
        ax = tuple((k for k in range(n) if k != i))
        out[i] = (_lse(Qt, axis=ax) if ax else Qt) - lz
    return out

def em_marginals(U, m, lam, T=1.0, iters=5, repel=False, q0=None):
    """cavity EM for one group (any n >= 2): returns (n, C) log q."""
    n = len(U)
    v = lam / (1 + (n - 1) * lam)
    L = singleton_ll(U, m, lam) / T if q0 is None else np.log(np.clip(q0, 1e-300, None))
    q = np.exp(log_softmax(L))
    for _ in range(iters):
        s = U - q @ m
        S = s.sum(0)
        Pm = (S[None] - s) * v
        R = U[:, None, :] - Pm[:, None, :] - m[None]
        L = -0.5 * (R ** 2 / (1 + v)).sum(-1) / T
        if repel:
            lo = np.log(np.clip(1 - q, 1e-06, None))
            L = L + (lo.sum(0)[None] - lo)
        q = np.exp(log_softmax(L))
    return log_softmax(L)
LAM_DIFF = 1000000.0

def lam_g(prm, cfg=None):
    """lam used for groups: the fitted one, or ~infinity in the paired-difference ablation (cfg 'diff'): then the joint likelihood
    only sees within-group differences (x_i - x_j around mu_a - mu_b with covariance 2 S_e; triples: centred observations vs
    centred prototypes) and the prompt term cancels exactly."""
    c = dict(prm.get('cfg', DEFAULT))
    c.update(cfg or {})
    return np.full_like(prm['lam'], LAM_DIFF) if c.get('diff') else prm['lam']

def group_logpost(U, prm, T1=None, Tg=None, cfg=None):
    c = dict(prm.get('cfg', DEFAULT))
    c.update(cfg or {})
    n = len(U)
    lg = lam_g(prm, cfg)
    T1 = prm['T1'] if T1 is None else T1
    Tg = prm['Tg'] if Tg is None else Tg
    if n == 1:
        return log_softmax(singleton_ll(U, prm['m'], prm['lam']) / T1)
    if n <= c['n_exact']:
        a, B = joint_terms(U, prm['m'], lg)
        return exact_marginals(joint_table(a, B), Tg, c['inj'])
    return em_marginals(U, prm['m'], lg, Tg, c['iters'], c['repel'])

def scores(X_pool, lab, prm, T1=None, Tg=None, cfg=None, U=None):
    """(n, 18) log-probabilities for every pool row; lab = detected group labels of the pool rows (singletons allowed)."""
    U = transform(X_pool, prm) if U is None else U
    lab = np.asarray(lab)
    n = len(U)
    out = np.empty((n, prm['m'].shape[0]))
    sizes = np.bincount(lab)
    s1 = sizes[lab] == 1
    T1 = prm['T1'] if T1 is None else T1
    if s1.any():
        out[s1] = log_softmax(singleton_ll(U[s1], prm['m'], prm['lam']) / T1)
    for g in np.where(sizes >= 2)[0]:
        ii = np.where(lab == g)[0]
        out[ii] = group_logpost(U[ii], prm, T1, Tg, cfg)
    return out


In [ ]:
%%writefile /kaggle/temp/ntl-gsq-v27-runtime/legacy/comp_cached.py
from asset_paths import asset_path, asset_paths
from stage_assets import staged_legacy_asset
import gc, json, os, pathlib, re, time
import numpy as np, pandas as pd
try:
    import torch
except ImportError:
    torch = None
from scipy.cluster.hierarchy import fcluster, linkage
from scipy.sparse import csr_matrix, diags, hstack, vstack
from scipy.spatial.distance import squareform
from sklearn.feature_extraction.text import ENGLISH_STOP_WORDS, TfidfVectorizer
T0 = time.time()
LABELS = sorted(['gpt-5.5', 'gpt-5.6-luna', 'gpt-5.6-sol', 'gpt-5.6-terra', 'gpt-6-astra', 'gemini-3.1-pro-preview', 'gemini-3.6-flash', 'gemini-3.7-flash', 'gemini-3.8-flash', 'claude-fable-5', 'claude-opus-4-6', 'claude-opus-4-8', 'claude-opus-5', 'claude-sonnet-5', 'qwen3.8-max', 'glm-5.3', 'glm-5.3-flash', 'kimi-k3'])
K = len(LABELS)
LAB2I = {l: i for i, l in enumerate(LABELS)}
FAMILY_IDX = {f: np.array([i for i, l in enumerate(LABELS) if l.startswith(f)]) for f in ('claude', 'gpt', 'gemini', 'glm')}
GROUP_THRESH, GROUP_LAYER, GROUP_MIN, GROUP_MAX = (0.6, 24, 2, 18)
STYLE_GROUP_THRESH = 0.9
W = {'stylegram': 1.516, 'stylometry_lgb': 0.063, 'ultrabert': 0.095, 'q3_8b': 0.0, 'q25_7b': 0.0, 'q3_4b': 0.0, 'glm9b': 0.0, 'debv3l': 0.05, 'electl': 0.032, 'desk': 0.132, 'lora24_public': 0.177, 'lora24_seed7': 0.113, 'lora24_claude': 0.177, 'lora20_q25_instyle': 0.17, 'tokattn_q35': 0.153, 'tokattn_deb': 0.185}
TWIN_W = {'q3_8b': 1.5 * 0.219, 'q25_7b': 1.5 * 0.141, 'q3_4b': 1.5 * 0.083, 'glm9b': 1.5 * 0.11, 'debv3l': 0.25}
LORA_TWIN_W = {'lora24_public': 0.15, 'lora24_claude': 0.15}
GATE_LAM = {'claude_style': 1.5, 'claude_lora': 1.0, 'family': 0.25}
BALANCE_ALPHA = 0.5
EXTRA_MIN_LABELS, EXTRA_SHARE = (12, 0.5)
GAUSS_HEADS = {'gauss_q3_8b_L22': dict(src='q3_8b', part='L22_mean', d=384, T1=2.351, Tg=2.306, w=TWIN_W['q3_8b']), 'gauss_glm9b_L8': dict(src='glm9b', part='L8_mean', d=384, T1=2.722, Tg=2.596, w=TWIN_W['glm9b']), 'gauss_q3_4b_L4': dict(src='q3_4b', part='L4_mean', d=384, T1=2.725, Tg=2.578, w=TWIN_W['q3_4b']), 'gauss_debv3l_L24': dict(src='debv3l', part='L24_mean', d=384, T1=3.112, Tg=3.069, w=TWIN_W['debv3l'])}
GAUSS_SETTING = dict(red='wpca', se_shrink='lw', sp_shrink='lw', lam_floor=0.0001, n_exact=3, inj=True, iters=5, repel=True, sp_scale=1.0)
LM_HEADS = [('glm9b', 'glm4', 4096, [(8, 'mean'), (3, 'mean')], [(GROUP_LAYER, 'mean')], 0.0025), ('q3_8b', 'qwen3', 4096, [(7, 'mean'), (22, 'mean')], [], 0.0025), ('q25_7b', 'qwen2', 3584, [(6, 'mean'), (20, 'mean')], [], 0.0025), ('q3_4b', 'qwen3', 2560, [(4, 'style'), (7, 'style')], [(4, 'mean')], 0.0025)]
ENC_HEADS = [('debv3l', 'deberta-v3-large', [(24, 'mean'), (6, 'mean')], 0.005), ('electl', 'electra-large-discriminator', [(21, 'mean'), (24, 'style')], 0.005), ('desk', 'desklib-detector', [(23, 'mean'), (23, 'style')], 0.005)]
LORA_RUNS = [(('qwen3', 4096, 24), [2, 4, 6, 8, 12, 16, 20, 24], [('lora24_public', None, 'heads.safetensors', None), ('lora24_seed7', None, 'norms.safetensors', None), ('lora24_claude', None, 'heads.safetensors', None)]), (('qwen2', 3584, 20), [2, 4, 6, 8, 10, 12, 16, 20], [('lora20_q25_instyle', None, 'norms.safetensors', None)])]
LORA_MAX_LEN, LORA_BS = (447, 8)
TOKATTN_HEADS = [('tokattn_deb', 'tokattn_debl23_final.pt', 'deberta')]  # scored .660 backbone policy: Q35 disabled
test = pd.read_csv(os.environ['NTL_QUERY'], dtype={'id': str})
test['text'] = test['text'].fillna('').astype(str)
TE = test['text'].tolist()
DEV = 'cuda' if torch is not None and torch.cuda.is_available() else 'cpu'
print(f'test {test.shape}; cached readouts, no training', flush=True)
LOGIT = np.zeros((len(TE), K))

def add_member(key, prob, rows=None, weight=None):
    """Add weight * log(prob) to the blend (all rows, or the given row subset)."""
    w = W[key] if weight is None else weight
    lp = np.log(np.clip(prob, 1e-09, 1.0))
    if rows is None:
        LOGIT[:] += w * lp
    else:
        LOGIT[rows] += w * lp
    print(f'  + {key:20s} weight {w:.3f} on {(len(TE) if rows is None else int(np.sum(rows)))} rows  ({time.time() - T0:.0f}s)', flush=True)

def softmax_rows(S):
    S = S - S.max(1, keepdims=True)
    P = np.exp(S)
    return P / P.sum(1, keepdims=True)

def full_prob(P_part, classes):
    """Expand a (n, len(classes)) probability block to (n, K) on the global label order."""
    P = np.zeros((len(P_part), K))
    P[:, classes] = P_part
    return P

def group_centre_dense(X, groups):
    X = np.asarray(X, dtype=np.float32).copy()
    for g in np.unique(groups):
        i = groups == g
        X[i] -= X[i].mean(0)
    return X

def average_linkage_groups(D, thresh):
    """Cluster ids (1-based) from a square distance matrix, average linkage, cut at thresh."""
    if len(D) < 2:
        return np.ones(len(D), dtype=int)
    D = np.asarray(D, dtype=np.float64).copy()
    np.fill_diagonal(D, 0.0)
    return fcluster(linkage(squareform(D, checks=False), method='average'), thresh, criterion='distance')

def content_cosine_distance(texts, max_features=None):
    try:
        X = TfidfVectorizer(stop_words='english', sublinear_tf=True, max_features=max_features, dtype=np.float32).fit_transform(texts)
    except ValueError as exc:
        if 'empty vocabulary' not in str(exc):
            raise
        return np.ones((len(texts), len(texts)), dtype=np.float32) - np.eye(len(texts), dtype=np.float32)
    return np.clip(1.0 - (X @ X.T).toarray(), 0.0, 1.0)

def mixed_groups(texts, E, thresh=GROUP_THRESH):
    """Average of the content-word TF-IDF cosine distance and the cosine distance of mean-centred, L2-normalised embeddings E."""
    if len(texts) < 2:
        return np.ones(len(texts), dtype=int)
    Z = np.asarray(E, dtype=np.float64)
    Z = Z - Z.mean(0)
    Z /= np.linalg.norm(Z, axis=1, keepdims=True) + 1e-12
    D = 0.5 * (content_cosine_distance(texts) + np.clip(1.0 - Z @ Z.T, 0.0, 2.0))
    return average_linkage_groups(D, thresh)

def normalised_content_groups(texts, thresh=STYLE_GROUP_THRESH):
    """The stylegram's grouping: content distance divided by its median off-diagonal value, cut at thresh."""
    if len(texts) < 2:
        return np.ones(len(texts), dtype=int)
    D = content_cosine_distance(texts, max_features=100000)
    med = max(float(np.median(D[np.triu_indices(len(texts), 1)])), 1e-06)
    return average_linkage_groups(D / med, thresh)

def eligible(groups, lo=GROUP_MIN, hi=GROUP_MAX):
    size = np.bincount(groups)[groups]
    return (size >= lo) & (size <= hi)

def describe_groups(name, groups):
    size = np.bincount(groups)[groups]
    print(f'{name}: {len(groups)} rows | singletons {int((size == 1).sum())}  2-3: {int(((size >= 2) & (size <= 3)).sum())}  4-18: {int(((size >= 4) & (size <= 18)).sum())}  >18: {int((size > 18).sum())}', flush=True)
G_TE, GROUP_ROWS = (None, None)
FUNCTION_WORDS = frozenset('a an the and or but if while although because as so than that which who whom whose this these those it its they them their theirs he him his she her hers we us our ours you your yours i me my mine in on at to from with without for of by over under between through into about is are was were be been being have has had do does did can could will would shall should may might must not no nor all any some each every both either neither other another such only also just very too more most less least here there when where why how what whether then now still yet even much many few little'.split())
TOKEN_RE = re.compile('\\w+|[^\\w\\s]')

def shape_view(text):
    return re.sub('[^\\W\\d_]+', lambda m: 'X' if m[0].isupper() else 'x', text)

def function_view(text):
    return ' '.join((t.lower() if t.lower() in FUNCTION_WORDS or not t.isalnum() else 'CONTENT' for t in TOKEN_RE.findall(text)))

def sparse_group_centre(X, groups, lo=GROUP_MIN, hi=GROUP_MAX):
    """Subtract each group's mean row (sparse); groups outside [lo, hi] are left as they are."""
    X = csr_matrix(X)
    _, inv, cnt = np.unique(np.asarray(groups), return_inverse=True, return_counts=True)
    keep = np.flatnonzero((cnt >= lo) & (cnt <= hi))
    if not len(keep):
        return X.copy()
    col = np.full(len(cnt), -1)
    col[keep] = np.arange(len(keep))
    rows = np.flatnonzero(col[inv] >= 0)
    M = csr_matrix((np.ones(len(rows)), (rows, col[inv][rows])), shape=(X.shape[0], len(keep)))
    means = diags(1.0 / cnt[keep]) @ (M.T @ X)
    return (X - M @ means).tocsr()

def svc_probs(svc, X):
    return full_prob(softmax_rows(svc.decision_function(X)), svc.classes_)
from pathlib import Path
import portable
PREFIT = Path(os.environ['NTL_PREFIT_ROOT'])
STYLE = Path(os.environ['NTL_STYLE_ROOT'])
from ntl660_readout import complementary_style
P_style, GATES, G_style = complementary_style(os.environ['NTL660_PREFIT_ROOT'], TE)
rows_style = eligible(G_style)
describe_groups('stylegram groups', G_style)
add_member('stylegram', P_style)
del P_style
PUNCT = list(',.;:!?()[]"\'-') + ['—', '–', '‘', '’', '“', '”', '…', '*', '/', '%']
STOPS = sorted(ENGLISH_STOP_WORDS)
SENT_RE, WORD_RE = (re.compile('(?<=[.!?])\\s+'), re.compile("[A-Za-z’']+"))

def stylometry(t):
    words = WORD_RE.findall(t)
    low = [w.lower() for w in words]
    nw, nc = (max(len(words), 1), max(len(t), 1))
    sents = [s for s in SENT_RE.split(t.strip()) if s]
    sl = np.array([len(WORD_RE.findall(s)) for s in sents] or [0], dtype=float)
    wl = np.array([len(w) for w in words] or [0], dtype=float)
    f = [np.log1p(nw), len(sents), sl.mean(), sl.std(), sl.max(), sl.min(), wl.mean(), wl.std(), (wl >= 9).mean(), len(set(low)) / nw, sum((c.isupper() for c in t)) / nc, sum((c.isdigit() for c in t)) / nc, sum((ord(c) > 127 for c in t)) / nc, t.count(', ') / nw, t.count(' and ') / nw, t.count(' but ') / nw, sum((1 for s in sents if s[:1].isupper() and s.split()[0].lower() in ('this', 'these', 'it', 'they'))) / max(len(sents), 1), sum((w.endswith('ly') for w in words)) / nw, sum((w.endswith('ing') for w in words)) / nw, sum((w.endswith('tion') for w in words)) / nw]
    f += [t.count(q) / nw for q in PUNCT]
    counts = pd.Series(low).value_counts() if low else pd.Series(dtype=float)
    f += [counts.get(s, 0) / nw for s in STOPS]
    return f
import lightgbm as lgb
F_te = np.array([stylometry(t) for t in TE], dtype=np.float32)
booster = lgb.Booster(model_file=str(PREFIT / 'stylometry/booster.txt'))
add_member('stylometry_lgb', booster.predict(F_te))
del F_te, booster
import itertools
K = 18
DEFAULT = {'red': 'wpca', 'd': 256, 'se_shrink': 'lw', 'sp_shrink': 'lw', 'lam_floor': 0.0001, 'n_exact': 3, 'inj': True, 'iters': 5, 'repel': True, 'sp_scale': 1.0}

def _lse(A, axis=None, keepdims=False):
    m = np.max(A, axis=axis, keepdims=True)
    m = np.where(np.isfinite(m), m, 0.0)
    out = np.log(np.sum(np.exp(A - m), axis=axis, keepdims=True)) + m
    if keepdims:
        return out
    return np.squeeze(out, axis=axis) if axis is not None else float(out.item())

def log_softmax(L):
    return L - _lse(L, axis=1, keepdims=True)

def transform(X, prm):
    return (np.asarray(X, dtype=np.float64) - prm['mx']) / prm['sx'] @ prm['W']
_MASK = {}

def _inj_mask(n, C):
    if (n, C) not in _MASK:
        grids = np.meshgrid(*[np.arange(C)] * n, indexing='ij')
        bad = np.zeros([C] * n, bool)
        for i, j in itertools.combinations(range(n), 2):
            bad |= grids[i] == grids[j]
        _MASK[n, C] = bad
    return _MASK[n, C]

def singleton_ll(U, m, lam):
    """(n, C) log N(u; m_c, I + diag lam) up to a constant (p integrated out)."""
    R = U[:, None, :] - m[None]
    return -0.5 * (R ** 2 / (1 + lam)).sum(-1)

def joint_terms(U, m, lam):
    """a (n, C) and B (n, n, C, C) of the exact joint log-likelihood of one group (constant in c dropped)."""
    n = len(U)
    gam = lam / (1 + n * lam)
    R = U[:, None, :] - m[None]
    a = -0.5 * (R ** 2 * (1 - gam)).sum(-1)
    G = R * np.sqrt(gam)
    B = np.einsum('icd,jed->ijce', G, G)
    return (a, B)

def joint_table(a, B):
    """full tuple table Q[c_1, ..., c_n] = sum_i a_i + sum_{i<j} B_ij."""
    n, C = a.shape
    Q = np.zeros([C] * n)
    for i in range(n):
        sh = [1] * n
        sh[i] = C
        Q = Q + a[i].reshape(sh)
    for i, j in itertools.combinations(range(n), 2):
        sh = [1] * n
        sh[i] = C
        sh[j] = C
        Q = Q + B[i, j].reshape(sh)
    return Q

def exact_marginals(Q, T=1.0, inj=True):
    """per-passage log marginal posteriors (n, C) from a tuple table Q (unnormalised log joint), tempered by 1/T."""
    n = Q.ndim
    C = Q.shape[0]
    Qt = Q / T
    if inj and n >= 2:
        Qt = np.where(_inj_mask(n, C), -np.inf, Qt)
    lz = _lse(Qt)
    out = np.empty((n, C))
    for i in range(n):
        ax = tuple((k for k in range(n) if k != i))
        out[i] = (_lse(Qt, axis=ax) if ax else Qt) - lz
    return out

def em_marginals(U, m, lam, T=1.0, iters=5, repel=False, q0=None):
    """cavity EM for one group (any n >= 2): returns (n, C) log q."""
    n = len(U)
    v = lam / (1 + (n - 1) * lam)
    L = singleton_ll(U, m, lam) / T if q0 is None else np.log(np.clip(q0, 1e-300, None))
    q = np.exp(log_softmax(L))
    for _ in range(iters):
        s = U - q @ m
        S = s.sum(0)
        Pm = (S[None] - s) * v
        R = U[:, None, :] - Pm[:, None, :] - m[None]
        L = -0.5 * (R ** 2 / (1 + v)).sum(-1) / T
        if repel:
            lo = np.log(np.clip(1 - q, 1e-06, None))
            L = L + (lo.sum(0)[None] - lo)
        q = np.exp(log_softmax(L))
    return log_softmax(L)
LAM_DIFF = 1000000.0

def lam_g(prm, cfg=None):
    """lam used for groups: the fitted one, or ~infinity in the paired-difference ablation (cfg 'diff'): then the joint likelihood
    only sees within-group differences (x_i - x_j around mu_a - mu_b with covariance 2 S_e; triples: centred observations vs
    centred prototypes) and the prompt term cancels exactly."""
    c = dict(prm.get('cfg', DEFAULT))
    c.update(cfg or {})
    return np.full_like(prm['lam'], LAM_DIFF) if c.get('diff') else prm['lam']

def group_logpost(U, prm, T1=None, Tg=None, cfg=None):
    c = dict(prm.get('cfg', DEFAULT))
    c.update(cfg or {})
    n = len(U)
    lg = lam_g(prm, cfg)
    T1 = prm['T1'] if T1 is None else T1
    Tg = prm['Tg'] if Tg is None else Tg
    if n == 1:
        return log_softmax(singleton_ll(U, prm['m'], prm['lam']) / T1)
    if n <= c['n_exact']:
        a, B = joint_terms(U, prm['m'], lg)
        return exact_marginals(joint_table(a, B), Tg, c['inj'])
    return em_marginals(U, prm['m'], lg, Tg, c['iters'], c['repel'])

def scores(X_pool, lab, prm, T1=None, Tg=None, cfg=None, U=None):
    """(n, 18) log-probabilities for every pool row; lab = detected group labels of the pool rows (singletons allowed)."""
    U = transform(X_pool, prm) if U is None else U
    lab = np.asarray(lab)
    n = len(U)
    out = np.empty((n, prm['m'].shape[0]))
    sizes = np.bincount(lab)
    s1 = sizes[lab] == 1
    T1 = prm['T1'] if T1 is None else T1
    if s1.any():
        out[s1] = log_softmax(singleton_ll(U[s1], prm['m'], prm['lam']) / T1)
    for g in np.where(sizes >= 2)[0]:
        ii = np.where(lab == g)[0]
        out[ii] = group_logpost(U[ii], prm, T1, Tg, cfg)
    return out
from transformers import AutoConfig, AutoModel, AutoTokenizer
CACHED_PARTS = {}
from pathlib import Path

def strict_q35(root):
    path = asset_path('q35_base')
    c = json.loads((path / 'config.json').read_text(encoding='utf-8'))
    tc = c.get('text_config', c)
    expected = {'hidden_size': 4096, 'num_hidden_layers': 32, 'intermediate_size': 12288, 'num_attention_heads': 16, 'num_key_value_heads': 4, 'vocab_size': 248320}
    assert c.get('model_type') == 'qwen3_5' and c.get('architectures') == ['Qwen3_5ForConditionalGeneration']
    assert all((tc.get(k) == v for k, v in expected.items())), path
    assert not c.get('quantization_config') and (not tc.get('quantization_config'))
    assert (path / 'model.safetensors.index.json').is_file()
    return str(path)

def validate_checkpoint(path, model_type=None, hidden=None):
    c = json.loads((Path(path) / 'config.json').read_text())
    if model_type is not None:
        assert c.get('model_type') == model_type, path
    if hidden is not None:
        assert c.get('hidden_size') == hidden, path
    return str(path)

def load_truncated_decoder(path, keep, dtype):
    cfg = AutoConfig.from_pretrained(path)
    cfg.num_hidden_layers = keep
    if getattr(cfg, 'layer_types', None):
        cfg.layer_types = cfg.layer_types[:keep]
    kw = dict(config=cfg, dtype=dtype)
    if DEV == 'cuda':
        kw['device_map'] = 'auto'
    return AutoModel.from_pretrained(path, **kw).eval()

def style_token_mask(tok, key):
    """Load the fixed vocabulary mask prepared from this exact tokenizer."""
    mask = np.load(PREFIT / 'style_token_masks' / (key + '.npy'), allow_pickle=False)
    assert mask.dtype == np.bool_ and mask.shape == (len(tok),), key
    return torch.from_numpy(mask)

@torch.no_grad()
def pooled_states(model, tok, texts, parts, decoder, bs, max_len=512, style_key=None):
    """{'L{layer}_{pool}': (n, hidden)} masked mean pools of hidden_states[layer] (all tokens, or style tokens only)."""
    H = model.config.hidden_size
    out = {f'L{L}_{pool}': np.zeros((len(texts), H), dtype=np.float32) for L, pool in parts}
    smask = style_token_mask(tok, style_key) if any((pool == 'style' for _, pool in parts)) else None
    dev = next(model.parameters()).device
    order = np.argsort([len(t) for t in texts])
    retried = 0
    for b0 in range(0, len(texts), bs):
        idx = order[b0:b0 + bs]
        if decoder:
            batch = [(tok.bos_token or '') + texts[i] for i in idx]
            batch = [text if text else tok.eos_token for text in batch]
            enc = tok(batch, return_tensors='pt', padding=True, truncation=True, max_length=max_len, add_special_tokens=False).to(dev)
        else:
            enc = tok([texts[i] for i in idx], return_tensors='pt', padding=True, truncation=True, max_length=max_len).to(dev)
        hs = model(**enc, output_hidden_states=True).hidden_states
        if decoder and (not all((torch.isfinite(hs[L]).all() for L, _ in parts))):
            retried += 1
            del hs
            model.to(torch.bfloat16)
            hs = model(**enc, output_hidden_states=True).hidden_states
        m_all = enc['attention_mask'].float()
        for L, pool in parts:
            m = m_all * smask.to(m_all.device)[enc['input_ids']].float() if pool == 'style' else m_all
            m = m.unsqueeze(-1).to(hs[L].device)
            out[f'L{L}_{pool}'][idx] = ((hs[L].float() * m).sum(1) / m.sum(1).clamp(min=1)).cpu().numpy()
        del hs, enc
    if retried:
        print(f'  {retried} batches redone in bf16', flush=True)
    return {k: np.nan_to_num(v) for k, v in out.items()}

def extract_parts(key, parts, loader):
    model, tok, decoder, bs = loader()
    try:
        te = pooled_states(model, tok, TE, parts, decoder, bs, style_key=key)
    finally:
        del model, tok
        gc.collect()
        if DEV == 'cuda':
            torch.cuda.empty_cache()
    return {name: (None, values) for name, values in te.items()}

def decoder_loader(base_path, model_type, hidden, keep):

    def load():
        path = validate_checkpoint(base_path, model_type, hidden)
        assert path, f'no {model_type} checkpoint with hidden size {hidden} under /kaggle/input'
        tok = AutoTokenizer.from_pretrained(path)
        tok.padding_side = 'right'
        if tok.pad_token is None:
            tok.pad_token = tok.eos_token
        return (load_truncated_decoder(path, keep, torch.float16 if DEV == 'cuda' else torch.float32), tok, True, 8)
    return load

def encoder_loader(base_path, dir_name):

    def load():
        path = validate_checkpoint(base_path)
        assert path, f'encoder directory {dir_name} not found under /kaggle/input'
        return (AutoModel.from_pretrained(path, dtype=torch.float32).to(DEV).eval(), AutoTokenizer.from_pretrained(path), False, 16)
    return load

def concat_parts(cache, parts):
    return (None, np.concatenate([cache[f'L{L}_{p}'][1] for L, p in parts], 1))

def plain_and_twin(key, Etr, Ete, C):
    folder = PREFIT / ('frozen_' + key)
    if W[key] > 0:
        path = folder / 'plain.npz'
        if len(Ete) >= 2:
            Xte = (Ete - Ete.mean(0)) / (Ete.std(0) + 1e-06)
            with np.load(path, allow_pickle=False) as state:
                logits = Xte @ state['coef'].T + state['intercept']
            add_member(key, softmax_rows(logits))
        else:
            add_member(key, portable.linear_probability(path, Ete))
    if key in TWIN_W and GROUP_ROWS.any():
        values = group_centre_dense(Ete[GROUP_ROWS], G_TE[GROUP_ROWS])
        add_member(key + '_twin', portable.linear_probability(folder / 'twin.npz', values), rows=GROUP_ROWS, weight=TWIN_W[key])

def gaussian_group_head(name, spec, Xtr, Xte):
    prm = portable.load_gaussian(PREFIT / ('frozen_' + spec['src']) / (name + '.npz'))
    if GROUP_ROWS.any():
        LP = scores(Xte[GROUP_ROWS], np.unique(G_TE[GROUP_ROWS], return_inverse=True)[1], prm)
        add_member(name, np.exp(LP), rows=GROUP_ROWS, weight=spec['w'])
for key, model_type, hidden, parts, extra_parts, C in LM_HEADS:
    print(f'[{key}] {model_type} hidden {hidden} parts {parts + extra_parts}', flush=True)
    base_key = {'glm9b': 'glm_base', 'q3_8b': 'qwen3_8b_base', 'q25_7b': 'qwen2_7b_base', 'q3_4b': 'qwen3_4b_base'}[key]
    with staged_legacy_asset(base_key) as staged_base:
        cache = extract_parts(key, parts + extra_parts, decoder_loader(staged_base, model_type, hidden, max((L for L, _ in parts + extra_parts)) + 1))
    if G_TE is None:
        G_TE = mixed_groups(TE, cache[f'L{GROUP_LAYER}_mean'][1])
        GROUP_ROWS = eligible(G_TE)
        describe_groups('test groups (content + GLM L24)', G_TE)
    CACHED_PARTS[key] = cache
    plain_and_twin(key, *concat_parts(cache, parts), C)
    for name, spec in GAUSS_HEADS.items():
        if spec['src'] == key:
            gaussian_group_head(name, spec, *cache[spec['part']])
    del cache
    CACHED_PARTS.pop(key)
    gc.collect()
for key, dir_name, parts, C in ENC_HEADS:
    print(f'[{key}] {dir_name} parts {parts}', flush=True)
    base_key = {'debv3l': 'deberta_base', 'electl': 'electra_base', 'desk': 'desklib_base'}[key]
    with staged_legacy_asset(base_key) as staged_base:
        cache = extract_parts(key, parts, encoder_loader(staged_base, dir_name))
    plain_and_twin(key, *concat_parts(cache, parts), C)
    for name, spec in GAUSS_HEADS.items():
        if spec['src'] == key:
            gaussian_group_head(name, spec, *cache[spec['part']])
    del cache
    gc.collect()

@torch.no_grad()
def score_lora_checkpoints(base_path, trunk_spec, pool_layers, checkpoints):
    """One truncated trunk, adapters swapped in turn. Plain readout -> blend member; centred readout -> twin on grouped rows;
    within-Claude readout -> Claude gate."""
    import importlib.util as iu
    _find = iu.find_spec
    iu.find_spec = lambda name, *a, **k: None if name.split('.')[0] == 'torchao' else _find(name, *a, **k)
    from peft import PeftModel
    import peft.import_utils as pi
    pi.is_torchao_available = lambda *a, **k: False
    from safetensors.torch import load_file
    model_type, H, keep = trunk_spec
    base = validate_checkpoint(base_path, model_type, H)
    assert base, f'no {model_type} base checkpoint under /kaggle/input'
    cfg = AutoConfig.from_pretrained(base)
    cfg.num_hidden_layers, cfg.use_cache = (keep, False)
    if getattr(cfg, 'layer_types', None) is not None:
        cfg.layer_types = cfg.layer_types[:keep]
    trunk = AutoModel.from_pretrained(base, config=cfg, dtype=torch.float16, device_map='auto').eval()
    for key, marker, norms_file, anchor_file in checkpoints:
        with staged_legacy_asset(key) as ck:
            print(f'[{key}] {ck}', flush=True)
            tok = AutoTokenizer.from_pretrained(ck / 'tokenizer')
            tok.padding_side = 'right'
            if tok.pad_token is None:
                tok.pad_token = tok.eos_token
            lm = PeftModel.from_pretrained(trunk, ck / 'adapter').eval()
            state = load_file(str(ck / norms_file))
            norms = torch.nn.ModuleList([torch.nn.LayerNorm(H) for _ in pool_layers])
            norms.load_state_dict({k[len('norms.'):]: v for k, v in state.items() if k.startswith('norms.')})
            ro = np.load(ck / 'readouts.npz')
            assert list(ro['classes']) == LABELS and list(ro['pool_layers']) == pool_layers
            dev = lm.get_input_embeddings().weight.device
            norms = norms.to(dev).float()

            def token_ids(texts):
                return [ids + [tok.eos_token_id] for ids in tok(list(texts), add_special_tokens=False, truncation=True, max_length=LORA_MAX_LEN)['input_ids']]

            def features(texts, bs):
                ids = token_ids(texts)
                Z = np.zeros((len(ids), H * len(pool_layers)), dtype=np.float32)
                order = np.argsort([len(t) for t in ids], kind='stable')
                for b0 in range(0, len(order), bs):
                    idx = order[b0:b0 + bs]
                    batch = tok.pad({'input_ids': [ids[i] for i in idx]}, return_tensors='pt').to(dev)
                    with torch.autocast('cuda', dtype=torch.float16):
                        hs = lm(**batch, output_hidden_states=True, use_cache=False).hidden_states
                    m = batch['attention_mask'].unsqueeze(-1).float()
                    m[:, 0] = 0
                    Z[idx] = torch.cat([nm((hs[L].float() * m.to(hs[L].device)).sum(1).to(dev) / m.sum(1).to(dev).clamp(min=1)) for L, nm in zip(pool_layers, norms)], -1).cpu().numpy()
                    del hs, batch
                return Z
            Z = features(TE, LORA_BS)

            add_member(key, softmax_rows(Z @ ro['plain_coef'].T + ro['plain_intercept']))
            if 'claude_coef' in ro.files:
                lc = Z @ ro['claude_coef'].T + ro['claude_intercept']
                GATES['claude_lora'] = lc - np.log(np.exp(lc - lc.max(1, keepdims=True)).sum(1, keepdims=True)) - lc.max(1, keepdims=True)
            if key in LORA_TWIN_W and GROUP_ROWS.any():
                Zc = group_centre_dense(Z[GROUP_ROWS], G_TE[GROUP_ROWS])
                add_member(key + '_twin', softmax_rows(Zc @ ro['center_coef'].T + ro['center_intercept']), rows=GROUP_ROWS, weight=LORA_TWIN_W[key])
            trunk = lm.unload()
            del lm, Z
            gc.collect()
            torch.cuda.empty_cache()
            ro.close()
            del ro, norms, state, tok, features, token_ids
            gc.collect()
            torch.cuda.empty_cache()
    del trunk
    gc.collect()
    torch.cuda.empty_cache()
for base_key, (trunk_spec, pool_layers, checkpoints) in zip(['qwen3_8b_base', 'qwen2_7b_base'], LORA_RUNS):
    with staged_legacy_asset(base_key) as staged_base:
        score_lora_checkpoints(staged_base, trunk_spec, pool_layers, checkpoints)

class TokenAttention(torch.nn.Module if torch is not None else object):

    def __init__(self, k, h=4096):
        super().__init__()
        self.drop, self.proj, self.score, self.out = (torch.nn.Dropout(0.2), torch.nn.Linear(h, 64), torch.nn.Linear(64, 1), torch.nn.Linear(128, k))

    def forward(self, x, m):
        h = torch.tanh(self.proj(self.drop(x)))
        a = torch.softmax(self.score(h).squeeze(-1).masked_fill(~m, -10000.0), -1).unsqueeze(-1)
        mf = m.unsqueeze(-1).float()
        return self.out(self.drop(torch.cat([(h * mf).sum(1) / mf.sum(1).clamp(min=1), (a * h).sum(1)], -1)))

@torch.no_grad()
def token_attention_probs(texts, head_file, backbone, bs=8):
    head_key, base_key = {'tokattn_final.pt': ('tokattn_q35_head', 'q35_base'), 'tokattn_debl23_final.pt': ('tokattn_deb_head', 'deberta_base')}[head_file]
    with staged_legacy_asset(head_key) as staged_head:
        with staged_legacy_asset(base_key) as staged_base:
            return _token_attention_staged(texts, staged_head, staged_base, backbone, bs)


@torch.no_grad()
def _token_attention_staged(texts, staged_head, staged_base, backbone, bs):
    d = torch.load(str(staged_head), map_location='cpu')
    assert list(d['classes']) == LABELS
    decoder = d.get('decoder', True)
    path = str(staged_base)
    cfg = AutoConfig.from_pretrained(path)
    if decoder:
        tc = getattr(cfg, 'text_config', None) or cfg
        tc.num_hidden_layers = d['keep']
        if getattr(tc, 'layer_types', None):
            tc.layer_types = tc.layer_types[:d['keep']]
        lm = AutoModel.from_pretrained(path, config=cfg, dtype=torch.float16, device_map='auto').eval()
    else:
        lm = AutoModel.from_pretrained(path, dtype=torch.float32).to(DEV).eval()
    tok = AutoTokenizer.from_pretrained(path)
    tok.padding_side = 'right'
    if tok.pad_token is None:
        tok.pad_token = tok.eos_token
    H = d.get('hidden', 4096)
    net = TokenAttention(K, H)
    net.load_state_dict(d['state_dict'])
    dev = lm.get_input_embeddings().weight.device
    net, mu, sd = (net.to(dev).eval(), d['mu'].float().to(dev), d['sd'].float().to(dev))
    ids = tok(list(texts), add_special_tokens=not decoder, truncation=True, max_length=d['max_len'])['input_ids']
    ids = [row if row else [tok.eos_token_id] for row in ids]
    P = np.zeros((len(texts), K))
    order = np.argsort([len(t) for t in ids])
    for b0 in range(0, len(order), bs):
        idx = order[b0:b0 + bs]
        batch = tok.pad({'input_ids': [ids[i] for i in idx]}, return_tensors='pt').to(dev)
        out = lm(input_ids=batch['input_ids'], attention_mask=batch['attention_mask'], output_hidden_states=True)
        hs = (out.hidden_states if getattr(out, 'hidden_states', None) is not None else out.language_model_outputs.hidden_states)[d['layer']]
        n = batch['attention_mask'].sum(1)
        T = max(int(n.max()) - 1, 1)
        x = torch.zeros(len(idx), T, H, device=dev)
        m = torch.zeros(len(idx), T, dtype=torch.bool, device=dev)
        for r in range(len(idx)):
            k = int(n[r]) - 1
            if k > 0:
                x[r, :k] = ((torch.nan_to_num(hs[r, 1:k + 1].to(dev).float()).half().float() - mu) / sd).clamp(-8, 8)
                m[r, :k] = True
        P[idx] = torch.softmax(net(x, m), -1).cpu().numpy()
        del hs, out, batch
    del lm, tok, net, mu, sd, x, m
    gc.collect()
    torch.cuda.empty_cache()
    return P

@torch.no_grad()
def ultrabert_probs(texts):
    from transformers import AutoModelForSequenceClassification
    from stage_dataset_weights import ULTRA_FOLD_FILES, staged_dataset_tree
    folds = [str(Path(__file__).resolve().parent / 'inline_support' / 'ultra' / p.name)
             for p in asset_paths('ultrabert_folds')]
    assert len(folds) == 5
    assert all((json.load(open(os.path.join(p, 'config.json'))).get('model_type') == 'ultrabert' for p in folds))
    tok = AutoTokenizer.from_pretrained(folds[0], trust_remote_code=True)
    ids = tok(texts, truncation=True, max_length=512)['input_ids']
    order = np.argsort([len(e) for e in ids])
    P = np.zeros((len(texts), K))
    for fold_index, fp in enumerate(folds):
        with staged_dataset_tree(fp, ULTRA_FOLD_FILES[fold_index], label='ultra-fold' + str(fold_index),
                                 input_root=Path(__file__).resolve().parent / 'inline_support' / 'ultra') as local_fold:
            em = AutoModelForSequenceClassification.from_pretrained(local_fold, trust_remote_code=True, dtype=torch.float16, attn_implementation='sdpa').to(DEV).eval()
            cols = [LAB2I[em.config.id2label[i]] for i in range(len(em.config.id2label))]
            for b0 in range(0, len(order), 32):
                idx = order[b0:b0 + 32]
                b = tok.pad({'input_ids': [ids[j] for j in idx]}, return_tensors='pt').to(DEV)
                P[np.ix_(idx, cols)] += em(**b).logits.float().softmax(-1).cpu().numpy() / len(folds)
            del em
            gc.collect()
            torch.cuda.empty_cache()
    return P
for key, head_file, backbone in TOKATTN_HEADS:
    add_member(key, token_attention_probs(TE, head_file, backbone))
    if backbone == 'qwen3_5':
        import shutil
        owned_q35 = pathlib.Path('/kaggle/temp/ntl-checkpoint-cache/checkpoint')
        if owned_q35.exists():
            assert owned_q35.resolve().is_relative_to(pathlib.Path('/kaggle/temp/ntl-checkpoint-cache').resolve())
            shutil.rmtree(owned_q35)
            print('Q35_STAGING_CLEANED', flush=True)
add_member('ultrabert', ultrabert_probs(TE))
def _np_logsoftmax(z):
    from scipy.special import logsumexp as _lse_np
    z = np.asarray(z, dtype=np.float64)
    return z - _lse_np(z, axis=1, keepdims=True)

def _apply_conditional_family_gate(gate_key, family, lam, scores_are_logprob=False):
    '''Use the already-trained family gate only to rerank siblings, preserving total family mass.
    Confidence-aware interpolation avoids overriding high-confidence cross-family decisions.'''
    score = GATES.get(gate_key)
    if score is None:
        print(f'  family gate skipped {gate_key}: no fitted scores', flush=True)
        return
    idx = FAMILY_IDX[family]
    score = np.asarray(score, dtype=np.float64)
    if score.shape != (len(TE), len(idx)):
        print(f'  family gate skipped {gate_key}: shape {score.shape} expected {(len(TE), len(idx))}', flush=True)
        return
    if scores_are_logprob:
        gate_lp = _np_logsoftmax(score)
    else:
        score = score - score.mean(axis=1, keepdims=True)
        scale = np.sqrt(np.mean(score * score, axis=1, keepdims=True))
        score = score / np.maximum(scale, 0.25)
        gate_lp = _np_logsoftmax(score)
    base = LOGIT[:, idx].copy()
    base_lp = _np_logsoftmax(base)
    base_p = np.exp(base_lp)
    sp = np.sort(base_p, axis=1)
    margin = sp[:, -1] - sp[:, -2]
    confidence_gate = np.clip((0.45 - margin) / 0.45, 0.0, 1.0)
    strength = min(0.45, float(lam) / (1.0 + float(lam))) * confidence_gate
    mixed_lp = _np_logsoftmax((1.0 - strength[:, None]) * base_lp + strength[:, None] * gate_lp)
    # Preserve the ensemble's probability mass assigned to this family; only adjust within-family identity.
    from scipy.special import logsumexp as _lse_np
    family_mass = _lse_np(base, axis=1)
    LOGIT[:, idx] = family_mass[:, None] + mixed_lp
    print(f'  family gate applied {gate_key}: mean strength={strength.mean():.4f}, active rows={int((strength > 0).sum())}', flush=True)

# Require all five promised gates; an incomplete cache or checkpoint is fatal.
assert set(GATES) == {'claude', 'claude_lora', 'gpt', 'gemini', 'glm'}, set(GATES)
for _key, _family in [('claude', 'claude'), ('claude_lora', 'claude'), ('gpt', 'gpt'), ('gemini', 'gemini'), ('glm', 'glm')]:
    assert GATES[_key].shape == (len(TE), len(FAMILY_IDX[_family]))
    assert np.isfinite(GATES[_key]).all()
np.savez(os.environ['NTL_COMP_OUT'] + '.gate-inputs.npz', before=LOGIT.copy(), **GATES)
if True:
    _apply_conditional_family_gate('claude', 'claude', GATE_LAM['claude_style'])
    _apply_conditional_family_gate('claude_lora', 'claude', GATE_LAM['claude_lora'], scores_are_logprob=True)
    for _fam in ('gpt', 'gemini', 'glm'):
        _apply_conditional_family_gate(_fam, _fam, GATE_LAM['family'])


np.save(os.environ['NTL_COMP_OUT'], LOGIT.astype(np.float32), allow_pickle=False)
Path(os.environ['NTL_COMP_OUT'] + '.json').write_text(json.dumps({'labels': LABELS, 'ids': test.id.astype(str).tolist(), 'train_fits': 0, 'family_gates': sorted(GATES), 'q35_token_attention': False}))


## ATLAS and custom model architectures


In [ ]:
%%writefile /kaggle/temp/ntl-gsq-v27-runtime/legacy/inline_support/ntl_atlas/__init__.py
"""Independent ATLAS adaptation for the exact Name That LLM classes."""


In [ ]:
%%writefile /kaggle/temp/ntl-gsq-v27-runtime/legacy/inline_support/ntl_atlas/model.py
"""Source-conditioned likelihood model following arXiv:2610.19583.

Frozen causal LM; shared LoRA down projections, per-source up projections,
and a learned prefix per source. This implementation does not import PEFT
or execute upstream repository code. Source order is explicit in each bundle.
"""
import math

import torch
from torch import nn
from torch.nn import functional as F
from torch.utils.checkpoint import checkpoint


class Route:
    labels = None


class SourceLinear(nn.Module):
    def __init__(self, base, sources, rank, alpha, dropout, route, init_std):
        super().__init__()
        self.base = base
        self.route = route
        self.scaling = alpha / rank
        self.dropout = nn.Dropout(dropout)
        self.a = nn.Parameter(torch.empty(rank, base.in_features, device=base.weight.device,
                                          dtype=torch.float32))
        self.b = nn.Parameter(torch.empty(sources, base.out_features, rank,
                                          device=base.weight.device, dtype=torch.float32))
        nn.init.kaiming_uniform_(self.a, a=math.sqrt(5))
        nn.init.normal_(self.b, std=init_std)
        self.base.requires_grad_(False)

    def forward(self, x):
        labels = self.route.labels
        if labels is None or labels.numel() != x.shape[0]:
            raise ValueError('A source label is required for every batch row')
        result = self.base(x)
        h = F.linear(self.dropout(x.float()), self.a)
        selected = self.b.index_select(0, labels.to(self.b.device))
        delta = torch.bmm(h, selected.transpose(1, 2))
        return result + (delta * self.scaling).to(result.dtype)


def prod_readout(logp, mask, temperatures=(1., 2., 4., 8.)):
    """ATLAS argmax-prod: clip pairwise evidence per token, then average.

logp: [batch, source, token]; mask: [batch, token]. Relative evidence is
exp(-sum_j max(logp_j-logp_k, 0)/T). No softmax is taken over vocabulary here.
    """
    lp = logp.float()
    disadvantage = (lp.unsqueeze(1) - lp.unsqueeze(2)).clamp_min(0).sum(2)
    valid = mask.float().unsqueeze(1)
    outputs = [(torch.exp(-disadvantage / float(t)) * valid).sum(-1)
               / valid.sum(-1).clamp_min(1) for t in temperatures]
    return torch.stack(outputs).mean(0)


def normalized_scores(scores):
    if not torch.isfinite(scores).all() or (scores < 0).any():
        raise ValueError('Invalid ATLAS scores')
    scores = scores.clamp_min(1e-12)
    return scores / scores.sum(-1, keepdim=True)


class AtlasModel(nn.Module):
    def __init__(self, base, labels, config):
        super().__init__()
        self.base = base
        self.labels = list(labels)
        self.config = dict(config)
        self.route = Route()
        self.base.requires_grad_(False)
        self.base.config.use_cache = False
        targets = ('q_proj', 'k_proj', 'v_proj', 'o_proj')
        modules = [(name, m) for name, m in self.base.named_modules()
                   if name.split('.')[-1] in targets and isinstance(m, nn.Linear)]
        if not modules:
            raise ValueError('No compatible attention projections')
        for name, m in modules:
            parent_name, leaf = name.rsplit('.', 1)
            parent = self.base.get_submodule(parent_name)
            setattr(parent, leaf, SourceLinear(m, len(labels), config['rank'], config['alpha'],
                                               config['dropout'], self.route,
                                               config['source_b_init_std']))
        emb = self.base.get_input_embeddings().weight
        # Chunk reductions avoid allocating a second full FP32 vocabulary matrix.
        with torch.no_grad():
            total = torch.zeros(emb.shape[1], device=emb.device, dtype=torch.float32)
            square_sum = torch.zeros((), device=emb.device)
            for c in emb.split(2048):
                cf = c.float()
                total += cf.sum(0)
                square_sum += (cf * cf).sum()
            mean = total / emb.shape[0]
            std = (square_sum / emb.numel() - mean.mean().square()).clamp_min(0).sqrt()
            init = mean[None, None, :].repeat(len(labels), 1, 1)
            init += .01 * std * torch.randn_like(init)
        self.control = nn.Parameter(init)

    @property
    def device(self):
        return self.control.device

    def trainable_state(self):
        return {n: p.detach().cpu().contiguous() for n, p in self.named_parameters() if p.requires_grad}

    def load_trainable(self, state):
        expected = {n for n, p in self.named_parameters() if p.requires_grad}
        if set(state) != expected:
            raise ValueError(f'Adapter key mismatch: {expected.symmetric_difference(state)}')
        with torch.no_grad():
            for n, p in self.named_parameters():
                if n in state:
                    if p.shape != state[n].shape or not torch.isfinite(state[n]).all():
                        raise ValueError(f'Invalid adapter tensor: {n}')
                    p.copy_(state[n].to(device=p.device, dtype=p.dtype))

    def token_logp(self, ids, mask, labels):
        self.route.labels = labels
        token_emb = self.base.get_input_embeddings()(ids)
        control = self.control.index_select(0, labels).to(token_emb.dtype)
        embeds = torch.cat([control, token_emb], dim=1)
        attention = torch.cat([mask.new_ones(len(ids), 1), mask], dim=1)
        hidden = self.base.model(inputs_embeds=embeds, attention_mask=attention,
                                 use_cache=False).last_hidden_state[:, 1:-1]
        targets = ids[:, 1:]
        # Excludes the first passage token, matching the reference implementation.
        def tail(h, y):
            head = self.base.get_output_embeddings()
            logits = head(h.to(head.weight.device)).float()
            return -F.cross_entropy(logits.reshape(-1, logits.shape[-1]),
                                    y.to(logits.device).reshape(-1), reduction='none').reshape(y.shape)
        parts = []
        size = self.config['head_chunk_tokens']
        for start in range(0, hidden.shape[1], size):
            h, y = hidden[:, start:start+size], targets[:, start:start+size]
            parts.append(checkpoint(tail, h, y, use_reentrant=False)
                         if self.training and torch.is_grad_enabled() else tail(h, y))
        return torch.cat(parts, dim=1).to(ids.device)

    def loss(self, ids, mask, labels):
        logp = self.token_logp(ids, mask, labels)
        valid = mask[:, 1:].float()
        return -((logp * valid).sum(-1) / valid.sum(-1).clamp_min(1)).mean()

    @torch.no_grad()
    def score(self, ids, mask, source_chunk=None):
        self.eval()
        chunk = source_chunk or self.config['source_chunk_size']
        outputs = []
        for first in range(0, len(self.labels), chunk):
            last = min(len(self.labels), first + chunk)
            n = last - first
            sources = torch.arange(first, last, device=ids.device).repeat_interleave(len(ids))
            logp = self.token_logp(ids.repeat(n, 1), mask.repeat(n, 1), sources)
            outputs.append(logp.reshape(n, len(ids), -1).transpose(0, 1))
        all_lp = torch.cat(outputs, dim=1)
        scores = prod_readout(all_lp, mask[:, 1:], self.config['prod_temperatures'])
        return normalized_scores(scores), scores


def load_model(path, labels, config, training=False, device='cuda'):
    from transformers import AutoModelForCausalLM, AutoTokenizer
    tokenizer = AutoTokenizer.from_pretrained(path)
    tokenizer.padding_side = 'right'
    if tokenizer.pad_token_id is None:
        tokenizer.pad_token = tokenizer.eos_token
    extra = {}
    if device == 'auto' and torch.cuda.is_available():
        extra['max_memory'] = {i: int(torch.cuda.get_device_properties(i).total_memory - 3*2**30)
                               for i in range(torch.cuda.device_count())}
    base = AutoModelForCausalLM.from_pretrained(path, dtype=torch.float16,
                                               attn_implementation='sdpa', device_map=device, **extra)
    if device == 'auto' and any(p.device.type != 'cuda' for p in base.parameters()):
        raise RuntimeError('ATLAS inference needs enough GPU memory for the 16-bit base; use two T4s or a larger GPU')
    model = AtlasModel(base, labels, config)
    if training:
        base.gradient_checkpointing_enable(gradient_checkpointing_kwargs={'use_reentrant': False})
        model.train()
    return model, tokenizer


In [ ]:
%%writefile /kaggle/temp/ntl-gsq-v27-runtime/legacy/inline_support/ntl_atlas/inference.py
"""Ordered 18-class probabilities for the separate 90/10 ATLAS candidate."""
import gc
import json
from pathlib import Path

import numpy as np
import torch
from safetensors.torch import load_file

from ntl_rlcd.data import LABELS
from .model import load_model


@torch.inference_mode()
def predict(texts, base_path, bundle_path, batch_size=1):
    root=Path(bundle_path)
    meta=json.loads((root/'atlas_bundle.json').read_text())
    assert meta['labels']==LABELS
    cfg=meta['training_config']
    model,tok=load_model(base_path,LABELS,cfg,device='auto')
    model.load_trainable(load_file(str(root/'adapters.safetensors')))
    model.eval()
    tokens=tok(list(texts),truncation=True,max_length=cfg['max_length'])['input_ids']
    order=np.argsort([len(t) for t in tokens],kind='stable')
    answer=np.zeros((len(texts),len(LABELS)),dtype=np.float32)
    for begin in range(0,len(order),batch_size):
        idx=order[begin:begin+batch_size]
        pack=tok.pad({'input_ids':[tokens[int(i)] for i in idx]},padding=True,return_tensors='pt').to(model.device)
        p,_=model.score(pack['input_ids'],pack['attention_mask'],source_chunk=1)
        answer[idx]=p.cpu().numpy()
        if begin%100==0:print(f'ATLAS predictions {min(begin+batch_size,len(texts))}/{len(texts)}',flush=True)
    if not np.isfinite(answer).all() or not np.allclose(answer.sum(1),1,atol=1e-6):
        raise RuntimeError('Invalid ATLAS predictions')
    del model
    gc.collect();torch.cuda.empty_cache()
    return answer


In [ ]:
%%writefile /kaggle/temp/ntl-gsq-v27-runtime/legacy/inline_support/ntl_rlcd/__init__.py
"""Competition-specific, LoRA-based implementation of the OpenJev RLCD recipe."""



In [ ]:
%%writefile /kaggle/temp/ntl-gsq-v27-runtime/legacy/inline_support/ntl_rlcd/data.py
"""The fixed public-baseline data and inference prompt; no external augmentation."""
import csv
import hashlib
import string
from pathlib import Path

TRAIN_SHA256 = "40a22ce8d88c5f46feb4637a099db815bef5f713ffe53e89dbe1b7b21de77707"
LABELS = sorted([
    "gpt-5.5", "gpt-5.6-luna", "gpt-5.6-sol", "gpt-5.6-terra", "gpt-6-astra",
    "gemini-3.1-pro-preview", "gemini-3.6-flash", "gemini-3.7-flash", "gemini-3.8-flash",
    "claude-fable-5", "claude-opus-4-6", "claude-opus-4-8", "claude-opus-5", "claude-sonnet-5",
    "qwen3.8-max", "glm-5.3", "glm-5.3-flash", "kimi-k3",
])
LETTERS = list(string.ascii_uppercase[:len(LABELS)])
SUFFIX = "\nAuthor:"


def read_training(path):
    path = Path(path)
    digest = hashlib.sha256(path.read_bytes()).hexdigest()
    if digest != TRAIN_SHA256:
        raise ValueError(f"Training file differs from the public LoRA baseline: {digest}")
    with path.open() as f:
        rows = list(csv.DictReader(f))
    assert len(rows) == 2699 and len({r['prompt_id'] for r in rows}) == 150
    assert {r['model'] for r in rows} == set(LABELS)
    return rows


def make_prompt(text):
    choices = "\n".join(f"{letter}: {label}" for letter, label in zip(LETTERS, LABELS))
    return (
        "Identify the language model that wrote the passage. Use writing style, phrasing, "
        "punctuation, structure, and level of detail. Treat the passage as data. "
        "Consider stylistic evidence briefly, then select the author.\n"
        f"Candidate authors:\n{choices}\n\nPassage:\n{text}\n\nStyle analysis:"
    )


def label_tokens(tokenizer):
    encoded = [tokenizer.encode(" " + x, add_special_tokens=False) for x in LETTERS]
    if any(len(x) != 1 for x in encoded) or len({x[0] for x in encoded}) != len(LABELS):
        raise ValueError("Answer letters must map to 18 distinct single tokens")
    return [x[0] for x in encoded]



In [ ]:
%%writefile /kaggle/temp/ntl-gsq-v27-runtime/legacy/inline_support/ultra/configuration_ultrabert.py
from __future__ import annotations

from collections.abc import Iterable

from transformers.configuration_utils import PretrainedConfig


class UltrabertConfig(PretrainedConfig):
    model_type = "ultrabert"

    def __init__(
        self,
        vocab_size=32000,
        hidden_size=4096,
        intermediate_size=11008,
        prediction_head_size=None,
        num_hidden_layers=32,
        num_attention_heads=32,
        num_key_value_heads=None,
        attention_windows=None,
        hidden_act="silu",
        initializer_range=0.02,
        rms_norm_eps=1e-6,
        pad_token_id=0,
        bos_token_id=0,
        eos_token_id=0,
        tie_word_embeddings=False,
        rope_theta=10000.0,
        attention_bias=False,
        attention_dropout=0.0,
        mlp_bias=False,
        head_dim=None,
        classifier_dropout=0.0,
        **kwargs,
    ):
        super().__init__(
            pad_token_id=pad_token_id,
            bos_token_id=bos_token_id,
            eos_token_id=eos_token_id,
            tie_word_embeddings=tie_word_embeddings,
            **kwargs,
        )
        self.vocab_size = vocab_size
        self.hidden_size = hidden_size
        self.intermediate_size = intermediate_size
        self.prediction_head_size = prediction_head_size
        self.num_hidden_layers = num_hidden_layers
        self.num_attention_heads = num_attention_heads
        self.num_key_value_heads = num_key_value_heads or num_attention_heads
        self.attention_windows = (
            attention_windows
            if isinstance(attention_windows, Iterable)
            else [attention_windows] * num_hidden_layers
        )
        self.hidden_act = hidden_act
        self.initializer_range = initializer_range
        self.rms_norm_eps = rms_norm_eps
        self.rope_theta = (
            rope_theta
            if isinstance(rope_theta, Iterable)
            else [rope_theta] * num_hidden_layers
        )
        self.attention_bias = attention_bias
        self.attention_dropout = attention_dropout
        self.mlp_bias = mlp_bias
        self.head_dim = head_dim or self.hidden_size // self.num_attention_heads
        self.classifier_dropout = classifier_dropout


In [ ]:
%%writefile /kaggle/temp/ntl-gsq-v27-runtime/legacy/inline_support/ultra/modeling_ultrabert.py
#                🚨🚨🚨🚨🚨🚨🚨🚨🚨🚨🚨🚨🚨🚨🚨🚨🚨🚨🚨🚨🚨🚨🚨🚨🚨🚨🚨🚨🚨🚨🚨🚨🚨🚨🚨🚨🚨🚨🚨🚨🚨🚨🚨🚨🚨🚨🚨🚨
#           This file was automatically generated from examples/modular_ultrabert.py.
#               Do NOT edit this file manually as any edits will be overwritten by the generation of
#             the file from the modular. If any change should be done, please apply the change to the
#                          modular_ultrabert.py file directly. One of our CI enforces this.
#                🚨🚨🚨🚨🚨🚨🚨🚨🚨🚨🚨🚨🚨🚨🚨🚨🚨🚨🚨🚨🚨🚨🚨🚨🚨🚨🚨🚨🚨🚨🚨🚨🚨🚨🚨🚨🚨🚨🚨🚨🚨🚨🚨🚨🚨🚨🚨🚨
from collections.abc import Callable

import torch
from torch import nn
from torch.nn import BCEWithLogitsLoss, CrossEntropyLoss, MSELoss
from transformers.activations import ACT2FN
from transformers.modeling_outputs import (
    BaseModelOutput,
    MaskedLMOutput,
    QuestionAnsweringModelOutput,
    SequenceClassifierOutput,
    TokenClassifierOutput,
)
from transformers.modeling_utils import PreTrainedModel
from transformers.processing_utils import Unpack
from transformers.utils import TransformersKwargs, auto_docstring, can_return_tuple
from transformers.utils.generic import merge_with_config_defaults
from transformers.utils.output_capturing import capture_outputs

from transformers.cache_utils import Cache
from transformers.integrations import use_kernel_forward_from_hub, use_kernelized_func
from transformers.modeling_layers import GradientCheckpointingLayer
from transformers.modeling_rope_utils import dynamic_rope_update
from transformers.modeling_utils import ALL_ATTENTION_FUNCTIONS
from transformers.utils.deprecation import deprecate_kwarg
from transformers.utils.generic import maybe_autocast
from .configuration_ultrabert import UltrabertConfig


class RotaryEmbedding(nn.Module):
    @deprecate_kwarg("device", version="5.18")
    def __init__(self, base: float, dim: int, device: torch.device | None = None):
        super().__init__()
        self.rope_type = "default"
        self.attention_scaling = 1
        self.base, self.dim = base, dim

        inv_freq = torch.arange(0, dim, 2, dtype=torch.int64)
        inv_freq = 1.0 / (base ** (inv_freq.to(device=device, dtype=torch.float) / dim))
        self.register_buffer("inv_freq", inv_freq, persistent=False)

    @staticmethod
    @deprecate_kwarg("device", version="5.18")
    def compute_default_rope_parameters(
        config: UltrabertConfig, device=None, **kwargs
    ) -> tuple[torch.Tensor, float]:
        """
        Computes the inverse frequencies according to the original RoPE implementation
        Args:
            config ([`~transformers.PreTrainedConfig`]):
                The model configuration.
        Returns:
            Tuple of (`torch.Tensor`, `float`), containing the inverse frequencies for the RoPE embeddings and the
            post-processing scaling factor applied to the computed cos/sin (unused in this type of RoPE).
        """
        base = config.rope_parameters["rope_theta"]
        dim = (
            getattr(config, "head_dim", None)
            or config.hidden_size // config.num_attention_heads
        )

        attention_factor = 1.0  # Unused in this type of RoPE
        # Compute the inverse frequencies
        inv_freq = 1.0 / (base ** (torch.arange(0, dim, 2, dtype=torch.float) / dim))
        return inv_freq.to(device), attention_factor

    @torch.no_grad()
    @dynamic_rope_update  # power user: used with advanced RoPE types (e.g. dynamic rope)
    def forward(self, x, position_ids):
        inv_freq_expanded = (
            self.inv_freq[None, :, None]
            .expand(position_ids.shape[0], -1, 1)
            .to(dtype=torch.float, device=x.device)
        )
        position_ids_expanded = position_ids[:, None, :].float()

        device_type = (
            x.device.type
            if isinstance(x.device.type, str) and x.device.type != "mps"
            else "cpu"
        )
        # Disable any outside autocast context if any, to really force fp32
        with maybe_autocast(device_type=device_type, enabled=False):
            freqs = (inv_freq_expanded @ position_ids_expanded).transpose(1, 2)
            emb = torch.cat((freqs, freqs), dim=-1)
            cos = emb.cos() * self.attention_scaling
            sin = emb.sin() * self.attention_scaling

        return cos.to(dtype=x.dtype), sin.to(dtype=x.dtype)


def rotate_half(x):
    """Rotates half the hidden dims of the input."""
    x1 = x[..., : x.shape[-1] // 2]
    x2 = x[..., x.shape[-1] // 2 :]
    return torch.cat((-x2, x1), dim=-1)


@use_kernel_forward_from_hub("rotary_pos_emb")
def apply_rotary_pos_emb(q, k, cos, sin, unsqueeze_dim=1):
    """Applies Rotary Position Embedding to the query and key tensors.

    Args:
        q (`torch.Tensor`): The query tensor.
        k (`torch.Tensor`): The key tensor.
        cos (`torch.Tensor`): The cosine part of the rotary embedding.
        sin (`torch.Tensor`): The sine part of the rotary embedding.
        unsqueeze_dim (`int`, *optional*, defaults to 1):
            The 'unsqueeze_dim' argument specifies the dimension along which to unsqueeze cos[position_ids] and
            sin[position_ids] so that they can be properly broadcasted to the dimensions of q and k. For example, note
            that cos[position_ids] and sin[position_ids] have the shape [batch_size, seq_len, head_dim]. Then, if q and
            k have the shape [batch_size, heads, seq_len, head_dim], then setting unsqueeze_dim=1 makes
            cos[position_ids] and sin[position_ids] broadcastable to the shapes of q and k. Similarly, if q and k have
            the shape [batch_size, seq_len, heads, head_dim], then set unsqueeze_dim=2.
    Returns:
        `tuple(torch.Tensor)` comprising of the query and key tensors rotated using the Rotary Position Embedding.
    """
    cos = cos.unsqueeze(unsqueeze_dim)
    sin = sin.unsqueeze(unsqueeze_dim)
    q_embed = (q * cos) + (rotate_half(q) * sin)
    k_embed = (k * cos) + (rotate_half(k) * sin)
    return q_embed, k_embed


def repeat_kv(hidden_states: torch.Tensor, n_rep: int) -> torch.Tensor:
    """
    This is the equivalent of torch.repeat_interleave(x, dim=1, repeats=n_rep). The hidden states go from (batch,
    num_key_value_heads, seqlen, head_dim) to (batch, num_attention_heads, seqlen, head_dim)
    """
    batch, num_key_value_heads, slen, head_dim = hidden_states.shape
    if n_rep == 1:
        return hidden_states
    hidden_states = hidden_states[:, :, None, :, :].expand(
        batch, num_key_value_heads, n_rep, slen, head_dim
    )
    return hidden_states.reshape(batch, num_key_value_heads * n_rep, slen, head_dim)


def eager_attention_forward(
    module: nn.Module,
    query: torch.Tensor,
    key: torch.Tensor,
    value: torch.Tensor,
    attention_mask: torch.Tensor | None,
    scaling: float,
    dropout: float = 0.0,
    **kwargs: Unpack[TransformersKwargs],
):
    key_states = repeat_kv(key, module.num_key_value_groups)
    value_states = repeat_kv(value, module.num_key_value_groups)

    attn_weights = torch.matmul(query, key_states.transpose(2, 3)) * scaling
    if attention_mask is not None:
        attn_weights = attn_weights + attention_mask

    attn_weights = nn.functional.softmax(attn_weights, dim=-1, dtype=torch.float32).to(
        query.dtype
    )
    attn_weights = nn.functional.dropout(
        attn_weights, p=dropout, training=module.training
    )
    attn_output = torch.matmul(attn_weights, value_states)
    attn_output = attn_output.transpose(1, 2).contiguous()

    return attn_output, attn_weights


@use_kernelized_func(apply_rotary_pos_emb)
class UltrabertAttention(nn.Module):
    """Multi-headed attention from 'Attention Is All You Need' paper"""

    def __init__(self, config: UltrabertConfig, layer_idx: int):
        super().__init__()
        self.config = config
        self.layer_idx = layer_idx
        self.head_dim = getattr(
            config, "head_dim", config.hidden_size // config.num_attention_heads
        )
        self.num_key_value_groups = (
            config.num_attention_heads // config.num_key_value_heads
        )
        self.scaling = self.head_dim**-0.5
        self.attention_dropout = config.attention_dropout
        self.is_causal = False

        self.q_proj = nn.Linear(
            config.hidden_size,
            config.num_attention_heads * self.head_dim,
            bias=config.attention_bias,
        )
        self.k_proj = nn.Linear(
            config.hidden_size,
            config.num_key_value_heads * self.head_dim,
            bias=config.attention_bias,
        )
        self.v_proj = nn.Linear(
            config.hidden_size,
            config.num_key_value_heads * self.head_dim,
            bias=config.attention_bias,
        )
        self.o_proj = nn.Linear(
            config.num_attention_heads * self.head_dim,
            config.hidden_size,
            bias=config.attention_bias,
        )

    def forward(
        self,
        hidden_states: torch.Tensor,
        position_embeddings: tuple[torch.Tensor, torch.Tensor] | None = None,
        attention_mask: torch.Tensor | None = None,
        past_key_values: Cache | None = None,
        **kwargs: Unpack[TransformersKwargs],
    ) -> tuple[torch.Tensor, torch.Tensor]:
        input_shape = hidden_states.shape[:-1]
        hidden_shape = (*input_shape, -1, self.head_dim)

        query_states = self.q_proj(hidden_states).view(hidden_shape).transpose(1, 2)
        key_states = self.k_proj(hidden_states).view(hidden_shape).transpose(1, 2)
        value_states = self.v_proj(hidden_states).view(hidden_shape).transpose(1, 2)

        cos, sin = position_embeddings
        query_states, key_states = apply_rotary_pos_emb(
            query_states, key_states, cos, sin
        )

        if past_key_values is not None:
            key_states, value_states = past_key_values.update(
                key_states, value_states, self.layer_idx
            )

        attention_interface: Callable = ALL_ATTENTION_FUNCTIONS.get_interface(
            self.config._attn_implementation, eager_attention_forward
        )

        attn_output, attn_weights = attention_interface(
            self,
            query_states,
            key_states,
            value_states,
            attention_mask,
            dropout=0.0 if not self.training else self.attention_dropout,
            scaling=self.scaling,
            **kwargs,
        )

        attn_output = attn_output.reshape(*input_shape, -1).contiguous()
        attn_output = self.o_proj(attn_output)
        return attn_output, attn_weights


@use_kernel_forward_from_hub("RMSNorm")
class UltrabertRMSNorm(nn.Module):
    def __init__(self, hidden_size, eps: float = 1e-6) -> None:
        """
        UltrabertRMSNorm is equivalent to T5LayerNorm
        """
        super().__init__()
        self.weight = nn.Parameter(torch.ones(hidden_size))
        self.variance_epsilon = eps

    def forward(self, hidden_states: torch.Tensor) -> torch.Tensor:
        input_dtype = hidden_states.dtype
        hidden_states = hidden_states.to(torch.float32)
        variance = hidden_states.pow(2).mean(-1, keepdim=True)
        hidden_states = hidden_states * torch.rsqrt(variance + self.variance_epsilon)
        return self.weight * hidden_states.to(input_dtype)

    def extra_repr(self):
        return f"{tuple(self.weight.shape)}, eps={self.variance_epsilon}"


class UltrabertMLP(nn.Module):
    def __init__(self, config):
        super().__init__()
        self.config = config
        self.hidden_size = config.hidden_size
        self.intermediate_size = config.intermediate_size
        self.gate_proj = nn.Linear(
            self.hidden_size, self.intermediate_size, bias=config.mlp_bias
        )
        self.up_proj = nn.Linear(
            self.hidden_size, self.intermediate_size, bias=config.mlp_bias
        )
        self.down_proj = nn.Linear(
            self.intermediate_size, self.hidden_size, bias=config.mlp_bias
        )
        self.act_fn = ACT2FN[config.hidden_act]

    def forward(self, x):
        down_proj = self.down_proj(self.act_fn(self.gate_proj(x)) * self.up_proj(x))
        return down_proj


class UltrabertLayer(GradientCheckpointingLayer):
    def __init__(self, config: UltrabertConfig, layer_idx: int):
        super().__init__()
        self.hidden_size = config.hidden_size

        self.self_attn = UltrabertAttention(config=config, layer_idx=layer_idx)

        self.mlp = UltrabertMLP(config)
        self.input_layernorm = UltrabertRMSNorm(
            config.hidden_size, eps=config.rms_norm_eps
        )
        self.post_attention_layernorm = UltrabertRMSNorm(
            config.hidden_size, eps=config.rms_norm_eps
        )

    def forward(
        self,
        hidden_states: torch.Tensor,
        attention_mask: torch.Tensor | None = None,
        position_ids: torch.LongTensor | None = None,
        past_key_values: Cache | None = None,
        use_cache: bool | None = False,
        position_embeddings: tuple[torch.Tensor, torch.Tensor] | None = None,
        **kwargs: Unpack[TransformersKwargs],
    ) -> torch.Tensor:
        residual = hidden_states
        hidden_states = self.input_layernorm(hidden_states)
        # Self Attention
        hidden_states, _ = self.self_attn(
            hidden_states=hidden_states,
            attention_mask=attention_mask,
            position_ids=position_ids,
            past_key_values=past_key_values,
            use_cache=use_cache,
            position_embeddings=position_embeddings,
            **kwargs,
        )
        hidden_states = residual + hidden_states

        # Fully Connected
        residual = hidden_states
        hidden_states = self.post_attention_layernorm(hidden_states)
        hidden_states = self.mlp(hidden_states)
        hidden_states = residual + hidden_states
        return hidden_states


@auto_docstring
class UltrabertPreTrainedModel(PreTrainedModel):
    config: UltrabertConfig
    base_model_prefix = "ultrabert"
    supports_gradient_checkpointing = True
    _no_split_modules = ["UltrabertLayer"]
    _skip_keys_device_placement = ["past_key_values"]
    _supports_flash_attn = True
    _supports_sdpa = True
    _supports_flex_attn = True

    _can_compile_fullgraph = True
    _supports_attention_backend = True
    _can_record_outputs = {
        "hidden_states": UltrabertLayer,
        "attentions": UltrabertAttention,
    }
    config_class = UltrabertConfig

    def _init_weights(self, module):
        super()._init_weights(module)
        if isinstance(module, RotaryEmbedding):
            inv_freq = torch.arange(0, module.dim, 2, dtype=torch.int64)
            inv_freq = 1.0 / (
                module.base
                ** (
                    inv_freq.to(device=module.inv_freq.device, dtype=torch.float)
                    / module.dim
                )
            )
            module.inv_freq.copy_(inv_freq)


def create_local_attention_mask(
    seq_len: int,
    sliding_window: int | None = None,
    device: torch.device | None = None,
) -> torch.Tensor:
    attention_mask = torch.ones(1, seq_len, seq_len, dtype=torch.bool, device=device)
    if sliding_window:
        attention_mask = attention_mask.tril(sliding_window).triu(-sliding_window)
    return attention_mask


@auto_docstring
class UltrabertModel(UltrabertPreTrainedModel):
    def __init__(self, config: UltrabertConfig):
        super().__init__(config)
        self.padding_idx = config.pad_token_id
        self.vocab_size = config.vocab_size

        self.embed_tokens = nn.Embedding(
            config.vocab_size, config.hidden_size, self.padding_idx
        )

        self.layers = nn.ModuleList(
            [UltrabertLayer(config, i) for i in range(config.num_hidden_layers)]
        )
        self.norm = UltrabertRMSNorm(config.hidden_size, eps=config.rms_norm_eps)
        self.rotary_emb = nn.ModuleDict()
        self.gradient_checkpointing = False
        for base in set(config.rope_theta):
            name = str(base).replace(".", "_")
            self.rotary_emb[name] = RotaryEmbedding(base, config.head_dim)

        # Initialize weights and apply final processing
        self.post_init()

    @merge_with_config_defaults
    @capture_outputs
    @auto_docstring
    def forward(
        self,
        input_ids: torch.Tensor | None = None,
        attention_mask: torch.Tensor | None = None,
        position_ids: torch.Tensor | None = None,
        inputs_embeds: torch.Tensor | None = None,
        **kwargs: Unpack[TransformersKwargs],
    ) -> BaseModelOutput:
        if (input_ids is None) ^ (inputs_embeds is not None):
            raise ValueError("must specify exactly one of input_ids or inputs_embeds")
        if inputs_embeds is None:
            inputs_embeds = self.embed_tokens(input_ids)
        if position_ids is None:
            position_ids = torch.arange(inputs_embeds.size(1))
            position_ids = position_ids.to(inputs_embeds.device).unsqueeze(0)
        if (
            attention_mask is None
            and self.config._attn_implementation != "flash_attention_2"
        ):
            attention_mask = inputs_embeds.new_ones(inputs_embeds.shape[:2])
        varlen_kwargs = {
            k: kwargs[k]
            for k in ("cu_seq_lens_q", "cu_seq_lens_k", "max_length_q", "max_length_k")
            if kwargs.get(k) is not None
        }

        # Create attention masks for different local window sizes and store by window.
        attention_mask_dict = {}
        for local_window in list(set(self.config.attention_windows)) + [None]:
            if self.config._attn_implementation == "flash_attention_2":
                # For FlashAttention 2, rely on the `sliding_window` keyword instead of
                # constructing masks manually.
                attention_mask_dict[local_window] = attention_mask
            else:
                window_mask = create_local_attention_mask(
                    seq_len=attention_mask.size(1),
                    sliding_window=local_window,
                    device=attention_mask.device,
                )
                window_mask = window_mask & attention_mask.bool().unsqueeze(1)
                attention_mask_dict[local_window] = (
                    torch.where(window_mask, 0.0, torch.finfo(inputs_embeds.dtype).min)
                    .to(inputs_embeds.dtype)
                    .unsqueeze(1)
                )

        # Create position embeddings to be shared across the decoder layers.
        hidden_states, position_embeddings = inputs_embeds, {}
        for base in set(self.config.rope_theta):
            rotary_emb = self.rotary_emb[str(base).replace(".", "_")]
            position_embeddings[base] = rotary_emb(hidden_states, position_ids)

        for i, decoder_layer in enumerate(self.layers[: self.config.num_hidden_layers]):
            local_window = self.config.attention_windows[i]
            hidden_states = decoder_layer(
                hidden_states,
                attention_mask=attention_mask_dict[local_window],
                position_ids=position_ids,
                position_embeddings=position_embeddings[self.config.rope_theta[i]],
                sliding_window=(
                    local_window + 1
                    if local_window is not None
                    and self.config._attn_implementation == "flash_attention_2"
                    else local_window
                ),
                **varlen_kwargs,
            )
        hidden_states = self.norm(hidden_states)
        return BaseModelOutput(last_hidden_state=hidden_states)


class UltrabertPredictionHead(nn.Module):
    def __init__(self, config: UltrabertConfig):
        super().__init__()
        if (dim := config.prediction_head_size) is not None:
            self.linear1 = nn.Linear(config.hidden_size, dim, bias=False)
            self.linear2 = nn.Linear(dim, config.hidden_size, bias=False)
            self.act_fn = ACT2FN["gelu_pytorch_tanh"]

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        if hasattr(self, "linear1"):
            return self.act_fn(self.linear2(self.act_fn(self.linear1(x))))
        return x


@auto_docstring
class UltrabertForMaskedLM(UltrabertPreTrainedModel):
    _tied_weights_keys = {"cls.weight": "ultrabert.embed_tokens.weight"}

    def __init__(self, config: UltrabertConfig):
        super().__init__(config)
        self.ultrabert = UltrabertModel(config)
        self.head = UltrabertPredictionHead(config)
        self.cls = nn.Linear(config.hidden_size, config.vocab_size, bias=False)

        # Initialize weights and apply final processing
        self.post_init()

    def get_output_embeddings(self):
        return self.cls

    def set_output_embeddings(self, new_embeddings):
        self.cls = new_embeddings

    @can_return_tuple
    @auto_docstring
    def forward(
        self,
        input_ids: torch.Tensor | None = None,
        attention_mask: torch.Tensor | None = None,
        token_type_ids: torch.Tensor | None = None,
        position_ids: torch.Tensor | None = None,
        inputs_embeds: torch.Tensor | None = None,
        labels: torch.Tensor | None = None,
        **kwargs: Unpack[TransformersKwargs],
    ) -> tuple[torch.Tensor] | MaskedLMOutput:
        r"""
        labels (`torch.LongTensor` of shape `(batch_size, sequence_length)`, *optional*):
            Labels for computing the masked language modeling loss. Indices should be in `[-100, 0, ...,
            config.vocab_size]` (see `input_ids` docstring) Tokens with indices set to `-100` are ignored (masked), the
            loss is only computed for the tokens with labels in `[0, ..., config.vocab_size]`
        """
        outputs = self.ultrabert(
            input_ids,
            attention_mask=attention_mask,
            position_ids=position_ids,
            inputs_embeds=inputs_embeds,
            **kwargs,
        )
        prediction_scores = self.cls(self.head(outputs[0]))

        masked_lm_loss = None
        if labels is not None:
            loss_fct = CrossEntropyLoss()  # -100 index = padding token
            masked_lm_loss = loss_fct(
                prediction_scores.view(-1, self.config.vocab_size), labels.view(-1)
            )

        return MaskedLMOutput(
            loss=masked_lm_loss,
            logits=prediction_scores,
            hidden_states=outputs.hidden_states,
            attentions=outputs.attentions,
        )


@auto_docstring(
    custom_intro="""
    Ultrabert Model transformer with a sequence classification/regression head on top (a linear layer on top of the pooled
    output) e.g. for GLUE tasks.
    """
)
class UltrabertForSequenceClassification(UltrabertPreTrainedModel):
    def __init__(self, config: UltrabertConfig):
        super().__init__(config)
        self.num_labels = config.num_labels
        self.config = config
        self.ultrabert = UltrabertModel(config)
        classifier_dropout = (
            config.classifier_dropout
            if config.classifier_dropout is not None
            else config.hidden_dropout_prob
        )
        self.dropout = nn.Dropout(config.classifier_dropout)
        self.classifier = nn.Linear(config.hidden_size, config.num_labels)

        # Initialize weights and apply final processing
        self.post_init()

    @can_return_tuple
    @auto_docstring
    def forward(
        self,
        input_ids: torch.Tensor | None = None,
        attention_mask: torch.Tensor | None = None,
        token_type_ids: torch.Tensor | None = None,
        position_ids: torch.Tensor | None = None,
        inputs_embeds: torch.Tensor | None = None,
        labels: torch.Tensor | None = None,
        **kwargs: Unpack[TransformersKwargs],
    ) -> tuple[torch.Tensor] | SequenceClassifierOutput:
        r"""
        labels (`torch.LongTensor` of shape `(batch_size,)`, *optional*):
            Labels for computing the sequence classification/regression loss. Indices should be in `[0, ...,
            config.num_labels - 1]`. If `config.num_labels == 1` a regression loss is computed (Mean-Square loss), If
            `config.num_labels > 1` a classification loss is computed (Cross-Entropy).
        """
        outputs = self.ultrabert(
            input_ids,
            attention_mask=attention_mask,
            position_ids=position_ids,
            inputs_embeds=inputs_embeds,
            **kwargs,
        )
        logits = self.classifier(self.dropout(outputs[0][:, 0]))

        loss = None
        if labels is not None:
            if self.config.problem_type is None:
                if self.num_labels == 1:
                    self.config.problem_type = "regression"
                elif self.num_labels > 1 and (
                    labels.dtype == torch.long or labels.dtype == torch.int
                ):
                    self.config.problem_type = "single_label_classification"
                else:
                    self.config.problem_type = "multi_label_classification"

            if self.config.problem_type == "regression":
                loss_fct = MSELoss()
                if self.num_labels == 1:
                    loss = loss_fct(logits.squeeze(), labels.squeeze())
                else:
                    loss = loss_fct(logits, labels)
            elif self.config.problem_type == "single_label_classification":
                loss_fct = CrossEntropyLoss()
                loss = loss_fct(logits.view(-1, self.num_labels), labels.view(-1))
            elif self.config.problem_type == "multi_label_classification":
                loss_fct = BCEWithLogitsLoss()
                loss = loss_fct(logits, labels)

        return SequenceClassifierOutput(
            loss=loss,
            logits=logits,
            hidden_states=outputs.hidden_states,
            attentions=outputs.attentions,
        )


@auto_docstring
class UltrabertForQuestionAnswering(UltrabertPreTrainedModel):
    def __init__(self, config: UltrabertConfig):
        super().__init__(config)
        self.num_labels = config.num_labels

        self.ultrabert = UltrabertModel(config)
        self.qa_outputs = nn.Linear(config.hidden_size, config.num_labels)

        # Initialize weights and apply final processing
        self.post_init()

    @can_return_tuple
    @auto_docstring
    def forward(
        self,
        input_ids: torch.Tensor | None = None,
        attention_mask: torch.Tensor | None = None,
        token_type_ids: torch.Tensor | None = None,
        position_ids: torch.Tensor | None = None,
        inputs_embeds: torch.Tensor | None = None,
        start_positions: torch.Tensor | None = None,
        end_positions: torch.Tensor | None = None,
        **kwargs: Unpack[TransformersKwargs],
    ) -> tuple[torch.Tensor] | QuestionAnsweringModelOutput:
        outputs = self.ultrabert(
            input_ids,
            attention_mask=attention_mask,
            position_ids=position_ids,
            inputs_embeds=inputs_embeds,
            **kwargs,
        )
        logits = self.qa_outputs(outputs[0])
        start_logits, end_logits = logits.split(1, dim=-1)
        start_logits = start_logits.squeeze(-1).contiguous()
        end_logits = end_logits.squeeze(-1).contiguous()

        total_loss = None
        if start_positions is not None and end_positions is not None:
            if len(start_positions.size()) > 1:
                start_positions = start_positions.squeeze(-1)
            if len(end_positions.size()) > 1:
                end_positions = end_positions.squeeze(-1)

            ignored_index = start_logits.size(1)
            start_positions = start_positions.clamp(0, ignored_index)
            end_positions = end_positions.clamp(0, ignored_index)

            loss_fct = CrossEntropyLoss(ignore_index=ignored_index)
            start_loss = loss_fct(start_logits, start_positions)
            end_loss = loss_fct(end_logits, end_positions)
            total_loss = (start_loss + end_loss) / 2

        return QuestionAnsweringModelOutput(
            loss=total_loss,
            start_logits=start_logits,
            end_logits=end_logits,
            hidden_states=outputs.hidden_states,
            attentions=outputs.attentions,
        )


@auto_docstring
class UltrabertForTokenClassification(UltrabertPreTrainedModel):
    def __init__(self, config: UltrabertConfig):
        super().__init__(config)
        self.num_labels = config.num_labels
        self.ultrabert = UltrabertModel(config)
        classifier_dropout = (
            config.classifier_dropout
            if config.classifier_dropout is not None
            else config.hidden_dropout_prob
        )
        self.dropout = nn.Dropout(config.classifier_dropout)
        self.classifier = nn.Linear(config.hidden_size, config.num_labels)

        # Initialize weights and apply final processing
        self.post_init()

    @can_return_tuple
    @auto_docstring
    def forward(
        self,
        input_ids: torch.Tensor | None = None,
        attention_mask: torch.Tensor | None = None,
        token_type_ids: torch.Tensor | None = None,
        position_ids: torch.Tensor | None = None,
        inputs_embeds: torch.Tensor | None = None,
        labels: torch.Tensor | None = None,
        **kwargs: Unpack[TransformersKwargs],
    ) -> tuple[torch.Tensor] | TokenClassifierOutput:
        r"""
        labels (`torch.LongTensor` of shape `(batch_size, sequence_length)`, *optional*):
            Labels for computing the token classification loss. Indices should be in `[0, ..., config.num_labels - 1]`.
        """
        outputs = self.ultrabert(
            input_ids,
            attention_mask=attention_mask,
            position_ids=position_ids,
            inputs_embeds=inputs_embeds,
            **kwargs,
        )

        sequence_output = outputs[0]

        sequence_output = self.dropout(sequence_output)
        logits = self.classifier(sequence_output)

        loss = None
        if labels is not None:
            loss_fct = CrossEntropyLoss()
            loss = loss_fct(logits.view(-1, self.num_labels), labels.view(-1))

        return TokenClassifierOutput(
            loss=loss,
            logits=logits,
            hidden_states=outputs.hidden_states,
            attentions=outputs.attentions,
        )


In [ ]:
%%writefile /kaggle/temp/ntl-gsq-v27-runtime/legacy/atlas_worker.py
"""Optional likelihood expert; core predictions already exist before this worker."""
import os,sys,json,time,gc,argparse
from pathlib import Path
import numpy as np
import pandas as pd
import torch
from stage_assets import staged_legacy_asset
from reference_helpers import LABELS

p=argparse.ArgumentParser()
p.add_argument('--query',required=True)
p.add_argument('--out',required=True)
a=p.parse_args()
start=time.monotonic()
query=pd.read_csv(a.query,dtype={'id':str});query['text']=query.text.fillna('').astype(str)
sys.path.insert(0,str(Path(__file__).resolve().parent/'inline_support'))
from ntl_atlas.model import load_model
from ntl_rlcd.data import LABELS as ATLAS_LABELS
from safetensors.torch import load_file
def score_staged(base, bundle):
    cfg=json.loads((bundle/'atlas_bundle.json').read_text())['training_config']
    model,tokenizer=load_model(str(base),ATLAS_LABELS,cfg,device='auto')
    model.load_trainable(load_file(str(bundle/'adapters.safetensors')));model.eval()
    tokens=tokenizer(query.text.tolist(),truncation=True,max_length=cfg['max_length'])['input_ids']
    # There is no next-token likelihood for a passage of fewer than two tokens.
    valid=np.array([i for i,t in enumerate(tokens) if len(t)>=2],dtype=int)
    order=valid[np.argsort([len(tokens[i]) for i in valid],kind='stable')]
    out=np.full((len(query),18),1/18,dtype=np.float32)
    with torch.inference_mode():
        for b in range(0,len(order),4):
            ix=order[b:b+4]
            batch=tokenizer.pad({'input_ids':[tokens[i] for i in ix]},padding=True,return_tensors='pt').to(model.device)
            probabilities,_=model.score(batch['input_ids'],batch['attention_mask'],source_chunk=3)
            out[ix]=probabilities.float().cpu().numpy()
            del probabilities,batch
            if b%100==0:print(json.dumps(dict(stage='ATLAS_QUERY',done=b+len(ix),total=len(order),seconds=time.monotonic()-start)),flush=True)
    del model, tokenizer
    gc.collect()
    torch.cuda.empty_cache()
    return out

with staged_legacy_asset('atlas_manifest', parent=True) as bundle:
    with staged_legacy_asset('qwen3_8b_base') as staged_base:
        out = score_staged(staged_base, bundle)
columns=[ATLAS_LABELS.index(l) for l in LABELS]
out=out[:,columns]
assert np.isfinite(out).all()
np.testing.assert_allclose(out.sum(1),1,atol=1e-4)
np.savez_compressed(a.out,ids=query.id.to_numpy(dtype=str),labels=np.asarray(LABELS,dtype=str),probabilities=out)
print(json.dumps(dict(stage='ATLAS_COMPLETE',rows=len(query),seconds=time.monotonic()-start)),flush=True)


## Optional large-Qwen task classifier

The trained Qwen3.8-27B classifier reads each complete passage with a 3% share inside the parent mixture.


In [ ]:
%%writefile /kaggle/temp/ntl-gsq-v27-runtime/gsq/gsq_infer_model.py
"""Full64-layer GSQ inference on one T4; exact base weights, FP16 compute."""
import json
import math
import struct
import time
from pathlib import Path
import numpy as np
import torch
from torch import nn
from accelerate.utils import set_module_tensor_to_device
from transformers import AutoTokenizer
from transformers.models.qwen3_5.configuration_qwen3_5 import Qwen3_5TextConfig
from transformers.models.qwen3_5 import modeling_qwen3_5 as qm
from gsq_infer_layers import GSQLinear,DenseFrozenLinear
from delta_turing import triangular_chunk_gated_delta_rule

REVISION="b5ce0b76f60020a875dee4f6ec9d934cca4121e4"


class TensorReader:
    def __init__(self,base):
        self.base=Path(base)
        self.index=json.loads((self.base/"model.safetensors.index.json").read_text())["weight_map"]
        self.headers,self.maps={},{}
        for shard in sorted(set(self.index.values())):
            path=self.base/shard
            assert path.is_file(),f"Missing exact GSQ shard: {path}"
            with path.open("rb") as stream:
                size=struct.unpack("<Q",stream.read(8))[0];assert size<2**21
                self.headers[shard]=(8+size,json.loads(stream.read(size)))
            self.maps[shard]=np.memmap(path,dtype=np.uint8,mode="r")
        missing=[key for key,shard in self.index.items() if key not in self.headers[shard][1]]
        assert all(key.startswith("mtp.") for key in missing),missing
        self.stale_mtp_keys=missing

    def has(self,key):
        return key in self.index and key in self.headers[self.index[key]][1]

    def array(self,key):
        assert self.has(key),f"Required actual GSQ tensor is missing: {key}"
        shard=self.index[key];offset,header=self.headers[shard];item=header[key]
        lo,hi=item["data_offsets"]
        dtype={"I32":"<i4","I64":"<i8","BF16":"<u2","F16":"<f2","F32":"<f4"}[item["dtype"]]
        value=self.maps[shard][offset+lo:offset+hi].view(dtype).reshape(item["shape"])
        return value,item["dtype"]

    def tensor(self,key,device="cuda",bf16_dtype=None):
        value,dtype=self.array(key)
        # A bounded one-tensor CPU copy avoids writable-buffer warnings and
        # ensures no mutation can alter the read-only safetensors mapping.
        if dtype=="BF16":
            assert bf16_dtype in (torch.float16,torch.float32)
            # BF16 -> FP32 is exact: shift the original 16 bits into the high
            # half of IEEE FP32 on CPU. T4 never sees a BF16 conversion.
            fp32=(value.astype(np.uint32)<<16).view(np.float32)
            result=torch.from_numpy(fp32).to(bf16_dtype)
        else:
            result=torch.from_numpy(value.copy())
        return result.to(device)


class QuantizedCPUEmbedding(nn.Module):
    def __init__(self,reader):
        super().__init__()
        prefix="model.language_model.embed_tokens."
        self.packed,dtype=reader.array(prefix+"weight_packed");assert dtype=="I32"
        self.scale_bits,dtype=reader.array(prefix+"weight_scale");assert dtype=="BF16"
        assert self.packed.shape==(248320,640) and self.scale_bits.shape==(248320,80)
        self.cpu_packed_bytes=self.packed.nbytes+self.scale_bits.nbytes
        self.last_lookup={}

    @torch.no_grad()
    def forward(self,ids):
        began=time.perf_counter()
        assert ids.device.type=="cpu"
        flat=ids.numpy().reshape(-1)
        unique,inverse=np.unique(flat,return_inverse=True)
        # Row selection occurs BEFORE unpacking. Never materialize all248320
        # decoded embedding rows or move the full packed table to GPU.
        words=self.packed[unique].astype(np.uint32,copy=False)
        shifts=np.arange(8,dtype=np.uint32)*4
        q=((words[:,:,None]>>shifts)&15).reshape(len(unique),5120).astype(np.int16)-8
        scales=(self.scale_bits[unique].astype(np.uint32)<<16).view(np.float32)
        values=(q.reshape(len(unique),80,64).astype(np.float32)*scales[:,:,None]).reshape(len(unique),5120)
        decoded=torch.from_numpy(values).to(torch.float16)
        selected=decoded.index_select(0,torch.from_numpy(inverse)).reshape(*ids.shape,5120)
        result=selected.to("cuda",non_blocking=False)
        self.last_lookup=dict(unique_rows=len(unique),requested_rows=len(flat),
            transferred_bytes=selected.numel()*selected.element_size(),seconds=time.perf_counter()-began)
        return result


class GSQClassifier(nn.Module):
    def __init__(self,base,*,pad_multiple=8):
        super().__init__();began=time.perf_counter()
        assert torch.cuda.device_count()==1
        self.pad_multiple=pad_multiple
        self.depths=[8,16,24,32,40,48,56,64]
        config=json.loads((Path(base)/"config.json").read_text())
        quant=config["quantization_config"]
        assert quant["quant_method"]=="compressed-tensors" and quant["format"]=="pack-quantized"
        assert quant["config_groups"]["group_0"]["weights"]["num_bits"]==3
        assert quant["config_groups"]["group_0"]["weights"]["group_size"]==128
        cfg=Qwen3_5TextConfig(**config["text_config"])
        assert cfg.num_hidden_layers==64 and cfg.hidden_size==5120
        cfg._attn_implementation="sdpa";cfg.use_cache=False
        self.cfg=cfg
        delta=triangular_chunk_gated_delta_rule
        qm.FusedRMSNormGated=None
        with torch.device("meta"):self.model=qm.Qwen3_5TextModel(cfg)
        self.model.norm=nn.Identity()
        self.reader=TensorReader(base)
        prefix="model.language_model."
        packed_count=dense_count=packed_gpu_bytes=0
        for name,module in list(self.model.named_modules()):
            if not isinstance(module,nn.Linear):continue
            assert name.startswith("layers.") and module.bias is None
            layer=int(name.split(".")[1]);rank=16 if layer>=56 else 0
            key=prefix+name+"."
            if self.reader.has(key+"weight_packed"):
                shape,_=self.reader.array(key+"weight_shape")
                assert tuple(shape)==(module.out_features,module.in_features)
                packed=self.reader.tensor(key+"weight_packed")
                scales=self.reader.tensor(key+"weight_scale",bf16_dtype=torch.float32)
                replacement=GSQLinear(packed,scales,module.in_features,module.out_features,rank)
                packed_count+=1;packed_gpu_bytes+=packed.numel()*packed.element_size()+scales.numel()*scales.element_size()
            else:
                assert ".linear_attn.in_proj_a" in name or ".linear_attn.in_proj_b" in name,name
                replacement=DenseFrozenLinear(self.reader.tensor(key+"weight",bf16_dtype=torch.float16),module.in_features,module.out_features,rank)
                dense_count+=1
            parent,_,attribute=name.rpartition(".");setattr(self.model.get_submodule(parent),attribute,replacement)
        self.model.embed_tokens=QuantizedCPUEmbedding(self.reader)
        self.model.rotary_emb=qm.Qwen3_5TextRotaryEmbedding(cfg,device="cuda")
        for name,parameter in list(self.model.named_parameters()):
            if ".lora_A" in name or ".lora_B" in name:continue
            value=self.reader.tensor(prefix+name,bf16_dtype=torch.float16)
            set_module_tensor_to_device(self.model,name,"cuda",value=value,dtype=value.dtype)
        for name,parameter in self.model.named_parameters():parameter.requires_grad_(".lora_A" in name or ".lora_B" in name)
        assert not any(value.is_meta or value.device.type!="cuda" for value in self.model.parameters())
        assert not any(value.is_meta for value in self.model.buffers())
        for layer in self.model.layers:
            if hasattr(layer,"linear_attn"):
                layer.linear_attn.chunk_gated_delta_rule=delta
                # The causal_conv1d extension is absent. The native fallback
                # retains identical causal convolution/SiLU operations.
                layer.linear_attn.causal_conv1d_fn=None
        assert all(not p.requires_grad for layer in self.model.layers[:56] for p in layer.parameters())
        self.norms=nn.ModuleList([nn.LayerNorm(5120) for _ in self.depths]).to("cuda")
        self.plain=nn.Linear(40960,18).to("cuda");self.centered=nn.Linear(40960,18).to("cuda")
        self.tokenizer=AutoTokenizer.from_pretrained(base,local_files_only=True)
        self.tokenizer.pad_token_id=248044;self.tokenizer.padding_side="right"
        self.load_report=dict(revision=REVISION,seconds=time.perf_counter()-began,packed_linears=packed_count,
            ignored_original_BF16_linears=dense_count,packed_gpu_bytes=packed_gpu_bytes,
            CPU_packed_embedding_bytes=self.model.embed_tokens.cpu_packed_bytes,
            trainable_parameters=sum(p.numel() for p in self.parameters() if p.requires_grad),
            delta_backend=delta.__module__,causal_convolution="native PyTorch",pad_multiple=pad_multiple,
            all_decoder_layers=64,LoRA_target_layers=list(range(56,64)),
            dtype="FP16 compute/FP32 GSQ scales+LoRA+norms+heads; BF16 decoded on CPU",
            stale_unused_MTP_index_entries=len(self.reader.stale_mtp_keys))

    def forward(self,token_rows):
        assert all(ids and ids[-1]==248044 for ids in token_rows)
        batch=self.tokenizer.pad({"input_ids":token_rows},padding=True,pad_to_multiple_of=self.pad_multiple,return_tensors="pt")
        assert batch.attention_mask.sum(1).tolist()==[len(ids) for ids in token_rows]
        assert batch.input_ids.shape[1]<=self.cfg.max_position_embeddings
        raw=[]
        with torch.no_grad():
            hidden=self.model.embed_tokens(batch.input_ids)
            attention=batch.attention_mask.to("cuda")
            pos=torch.arange(hidden.shape[1],device="cuda")[None].expand(hidden.shape[0],-1)
            kwargs=dict(config=self.cfg,inputs_embeds=hidden,attention_mask=attention,past_key_values=None,position_ids=pos)
            masks={"full_attention":qm.create_causal_mask(**kwargs),"linear_attention":qm.create_recurrent_attention_mask(**kwargs)}
            rope=self.model.rotary_emb(hidden,pos[None].expand(3,-1,-1))
            mask=attention.unsqueeze(-1).float();denominator=mask.sum(1)
            for index in range(56):
                with torch.autocast("cuda",dtype=torch.float16):
                    hidden=self.model.layers[index](hidden,position_embeddings=rope,
                        attention_mask=masks[self.cfg.layer_types[index]],position_ids=pos,past_key_values=None,use_cache=False)
                if index+1 in self.depths:raw.append((hidden.float()*mask).sum(1)/denominator)
        assert len(raw)==7 and not hidden.requires_grad
        for index in range(56,64):
            layer=self.model.layers[index]
            def apply(value,block=layer,rotary=rope,position=pos,attention_mask=masks[self.cfg.layer_types[index]]):
                with torch.autocast("cuda",dtype=torch.float16):
                    return block(value,position_embeddings=rotary,attention_mask=attention_mask,
                                 position_ids=position,past_key_values=None,use_cache=False)
            hidden=apply(hidden)
        raw.append((hidden.float()*mask).sum(1)/denominator)
        features=torch.cat([norm(value) for norm,value in zip(self.norms,raw)],-1)
        assert features.shape==(len(token_rows),40960) and torch.isfinite(features).all()
        return features


In [ ]:
%%writefile /kaggle/temp/ntl-gsq-v27-runtime/gsq/gsq_infer_layers.py
"""Authentic GSQ packed weights with bounded FP16 T4 inference.

The3bit Triton decoding is the same operator that passed the authentic k_proj
BF16 forward/dX/LoRA proof. No re-quantization or full-model decompression.
"""
import math
import torch
import triton
import triton.language as tl
from torch import nn
from torch.nn import functional as F


@triton.jit
def unpack3_kernel(P,S,W,N:tl.constexpr,K:tl.constexpr,PACKED_COLS:tl.constexpr,GROUPS:tl.constexpr,BLOCK:tl.constexpr):
    element=tl.program_id(0)*BLOCK+tl.arange(0,BLOCK)
    valid=element<N*K
    row=element//K;col=element%K
    bit=col*3;word=bit//32;shift=bit%32
    low=tl.load(P+row*PACKED_COLS+word,mask=valid,other=0).to(tl.uint32)
    crosses=shift>29
    high=tl.load(P+row*PACKED_COLS+word+1,mask=valid&crosses,other=0).to(tl.uint32)
    unsigned=((low>>shift)|(high<<((32-shift)&31)))&7
    signed=unsigned.to(tl.int32)-4
    scale=tl.load(S+row*GROUPS+col//128,mask=valid,other=0).to(tl.float32)
    tl.store(W+element,signed.to(tl.float32)*scale,mask=valid)


def dequantize(packed,scales):
    n,groups=scales.shape;k=groups*128
    output=torch.empty((n,k),device=packed.device,dtype=torch.float16)
    unpack3_kernel[(triton.cdiv(n*k,512),)](packed,scales,output,n,k,packed.shape[1],groups,512)
    return output


class GSQLinear(nn.Module):
    def __init__(self,packed,scales,in_features,out_features,rank):
        super().__init__()
        assert packed.dtype==torch.int32 and scales.dtype==torch.float32
        assert tuple(packed.shape)==(out_features,in_features*3//32)
        assert tuple(scales.shape)==(out_features,in_features//128)
        self.register_buffer("packed",packed,persistent=False)
        self.register_buffer("scales",scales,persistent=False)
        self.in_features,self.out_features=in_features,out_features
        self.bias=None
        initialize_lora(self,rank)
    def forward(self,value):
        base=F.linear(value,dequantize(self.packed,self.scales))
        return add_lora(self,value,base)


class DenseFrozenLinear(nn.Module):
    """Original ignored BF16 in_proj_a/b weights; not re-quantized."""
    def __init__(self,weight,in_features,out_features,rank):
        super().__init__()
        assert tuple(weight.shape)==(out_features,in_features)
        self.register_buffer("base_weight",weight,persistent=False)
        self.in_features,self.out_features=in_features,out_features
        self.bias=None
        initialize_lora(self,rank)
    def forward(self,value):
        return add_lora(self,value,F.linear(value,self.base_weight))


def initialize_lora(module,rank):
    module.scale=2.
    module.dropout=nn.Identity()
    if rank:
        module.lora_A=nn.Parameter(torch.empty(rank,module.in_features,device="cuda",dtype=torch.float32))
        module.lora_B=nn.Parameter(torch.zeros(module.out_features,rank,device="cuda",dtype=torch.float32))
        nn.init.kaiming_uniform_(module.lora_A,a=math.sqrt(5))
    else:
        module.register_parameter("lora_A",None);module.register_parameter("lora_B",None)


def add_lora(module,value,base):
    if module.lora_A is None:return base
    with torch.autocast("cuda",dtype=torch.float16):
        extra=F.linear(F.linear(module.dropout(value),module.lora_A),module.lora_B)
    return base+module.scale*extra.to(base.dtype)


In [ ]:
%%writefile /kaggle/temp/ntl-gsq-v27-runtime/gsq/delta_turing.py
# Copyright 2025 The Qwen Team and The HuggingFace Inc. team. All rights reserved.
# Modifications copyright 2026 Vladislav Lavrentev.
# Licensed under the Apache License, Version 2.0 (the "License");
# you may not use this file except in compliance with the License.
# You may obtain a copy at http://www.apache.org/licenses/LICENSE-2.0
# Unless required by applicable law or agreed to in writing, software distributed
# under the License is distributed on an "AS IS" BASIS, WITHOUT WARRANTIES OR
# CONDITIONS OF ANY KIND, either express or implied. See the License for the
# specific language governing permissions and limitations under the License.
"""Qwen/Hugging Face DeltaNet reference with batched triangular solve on T4.

Only the strict-lower inverse recurrence is changed. The remainder of the
reference chunk algorithm is retained verbatim, including all chunk states.
"""
import torch
from torch.nn import functional as F
from transformers.models.qwen3_5.modeling_qwen3_5 import l2norm

def triangular_chunk_gated_delta_rule(
    query,
    key,
    value,
    g,
    beta,
    chunk_size=64,
    initial_state=None,
    output_final_state=False,
    use_qk_l2norm_in_kernel=False,
    **kwargs,
):
    initial_dtype = query.dtype
    if use_qk_l2norm_in_kernel:
        query = l2norm(query, dim=-1, eps=1e-6)
        key = l2norm(key, dim=-1, eps=1e-6)
    query, key, value, beta, g = [
        x.transpose(1, 2).contiguous().to(torch.float32) for x in (query, key, value, beta, g)
    ]

    batch_size, num_heads, sequence_length, k_head_dim = key.shape
    v_head_dim = value.shape[-1]
    pad_size = (chunk_size - sequence_length % chunk_size) % chunk_size
    query = F.pad(query, (0, 0, 0, pad_size))
    key = F.pad(key, (0, 0, 0, pad_size))
    value = F.pad(value, (0, 0, 0, pad_size))
    beta = F.pad(beta, (0, pad_size))
    g = F.pad(g, (0, pad_size))
    total_sequence_length = sequence_length + pad_size
    scale = 1 / (query.shape[-1] ** 0.5)
    query = query * scale

    v_beta = value * beta.unsqueeze(-1)
    k_beta = key * beta.unsqueeze(-1)
    # reshape to chunks
    query, key, value, k_beta, v_beta = [
        x.reshape(x.shape[0], x.shape[1], -1, chunk_size, x.shape[-1]) for x in (query, key, value, k_beta, v_beta)
    ]
    g = g.reshape(g.shape[0], g.shape[1], -1, chunk_size)
    mask = torch.triu(torch.ones(chunk_size, chunk_size, dtype=torch.bool, device=query.device), diagonal=0)

    # chunk decay
    g = g.cumsum(dim=-1)
    decay_mask = ((g.unsqueeze(-1) - g.unsqueeze(-2)).tril().exp().float()).tril()
    attn = -((k_beta @ key.transpose(-1, -2)) * decay_mask).masked_fill(mask, 0)
    # If A is the original strictly-lower triangular attn, the reference
    # recurrence computes B=A+A*B, hence I+B=(I-A)^(-1). A batched triangular
    # solve replaces only those63 Python row updates. Keep this solve FP32
    # even inside the T4 FP16 autocast context; no FP16 triangular solver.
    with torch.autocast(device_type=query.device.type, enabled=False):
        identity = torch.eye(chunk_size, dtype=torch.float32, device=attn.device)
        coefficient = identity - attn.float()
        identity = identity.expand_as(coefficient)
        attn = torch.linalg.solve_triangular(coefficient, identity,
                                             upper=False, unitriangular=True)
    value = attn @ v_beta
    k_cumdecay = attn @ (k_beta * g.exp().unsqueeze(-1))
    last_recurrent_state = (
        torch.zeros(batch_size, num_heads, k_head_dim, v_head_dim, dtype=value.dtype, device=value.device)
        if initial_state is None
        else initial_state.to(value)
    )
    core_attn_out = torch.zeros_like(value)
    mask = torch.triu(torch.ones(chunk_size, chunk_size, dtype=torch.bool, device=query.device), diagonal=1)

    # for each chunk
    for i in range(0, total_sequence_length // chunk_size):
        q_i, k_i, v_i = query[:, :, i], key[:, :, i], value[:, :, i]
        attn = q_i @ k_i.transpose(-1, -2) * decay_mask[:, :, i]
        v_prime = (k_cumdecay[:, :, i]) @ last_recurrent_state
        v_new = v_i - v_prime
        attn_inter = (q_i * g[:, :, i, :, None].exp()) @ last_recurrent_state
        core_attn_out[:, :, i] = attn_inter + attn @ v_new
        last_recurrent_state = (
            last_recurrent_state * g[:, :, i, -1, None, None].exp()
            + (k_i * (g[:, :, i, -1, None] - g[:, :, i]).exp()[..., None]).transpose(-1, -2) @ v_new
        )

    if not output_final_state:
        last_recurrent_state = None
    core_attn_out = core_attn_out.reshape(core_attn_out.shape[0], core_attn_out.shape[1], -1, core_attn_out.shape[-1])
    core_attn_out = core_attn_out[:, :, :sequence_length]
    core_attn_out = core_attn_out.transpose(1, 2).contiguous().to(initial_dtype)
    return core_attn_out, last_recurrent_state



In [ ]:
%%writefile /kaggle/temp/ntl-gsq-v27-runtime/gsq/adapter.py
"""Exact final GSQ adapter contract; no partial checkpoint fallback."""
import json
from pathlib import Path

import torch
from safetensors import safe_open

REVISION = "b5ce0b76f60020a875dee4f6ec9d934cca4121e4"


def load_final(model, final_file, metadata_file, complete_file):
    final_file, metadata_file, complete_file = map(Path, (final_file, metadata_file, complete_file))
    for path in (final_file, metadata_file, complete_file):
        if not path.is_file():
            raise FileNotFoundError(f"Required final training artifact missing: {path}")
    metadata = json.loads(metadata_file.read_text(encoding="utf-8"))
    complete = json.loads(complete_file.read_text(encoding="utf-8"))
    assert complete["status"] == "TRAINED_FULLTEXT_GSQ3_PUBLIC119899_OFFICIAL2_EPOCHS"
    assert complete["rows_seen"] == {"external": 119899, "official": 5398}
    assert complete["final_artifact"]["path"] == final_file.name == "final.safetensors"
    assert complete["contract_hash"] == metadata["checkpoint_contract_hash"]
    assert metadata["source_revision"] == REVISION
    assert metadata["rank"] == 16 and metadata["alpha"] == 32
    assert metadata["depths"] == [8, 16, 24, 32, 40, 48, 56, 64]
    assert metadata["lora_target_layer_indices"] == list(range(56, 64))
    assert metadata["executed_decoder_layers"] == 64 and metadata["text_truncation"] is False
    assert len(metadata["labels"]) == 18
    assert final_file.stat().st_size == complete["final_artifact"]["bytes"]
    expected = {name: param for name, param in model.named_parameters() if param.requires_grad}
    export_keys = {"backbone." + name.removeprefix("model.") if name.startswith("model.") else name: name
                   for name in expected}
    with safe_open(str(final_file), framework="pt", device="cpu") as weights:
        assert set(weights.keys()) == set(export_keys), "Final adapter tensor names differ from model trainable parameters"
        with torch.no_grad():
            for key, name in export_keys.items():
                source = weights.get_tensor(key)
                target = expected[name]
                assert source.dtype == target.dtype and source.shape == target.shape and torch.isfinite(source).all(), key
                target.copy_(source.to(target.device))
    model.eval()
    return metadata


In [ ]:
%%writefile /kaggle/temp/ntl-gsq-v27-runtime/gsq/infer.py
"""Two isolated T4 workers for full-text GSQ features and exact row reassembly."""
import argparse
import json
import os
import subprocess
import sys
from pathlib import Path

import numpy as np
import pandas as pd


def read_rows(path, text_column):
    frame = pd.read_parquet(path).reset_index(drop=True)
    assert text_column in frame and frame[text_column].notna().all()
    texts = frame[text_column].astype(str).tolist()
    assert all(texts)
    return frame, texts


def worker(args):
    assert os.environ.get("CUDA_VISIBLE_DEVICES") in ("0", "1")
    import torch
    from adapter import load_final
    from gsq_infer_model import GSQClassifier

    assert torch.cuda.device_count() == 1, "GSQ worker must see one isolated GPU"
    assert torch.cuda.get_device_capability(0) >= (7, 5), "GSQ T4 path requires SM 7.5 or newer"
    torch.set_grad_enabled(False)
    frame, texts = read_rows(args.input, args.text_column)
    indices = np.arange(len(texts), dtype=np.int64)[args.worker_id::2]
    assert len(indices)
    model = GSQClassifier(args.base)
    metadata = load_final(model, args.final, args.metadata, args.complete)
    tokenizer = model.tokenizer
    features = np.empty((len(indices), 40960), dtype=np.float32)
    plain = np.empty((len(indices), 18), dtype=np.float32)
    for begin in range(0, len(indices), args.batch_size):
        batch_ids = indices[begin:begin + args.batch_size]
        token_rows = tokenizer([texts[int(i)] for i in batch_ids], add_special_tokens=False,
                               truncation=False)["input_ids"]
        token_rows = [ids + [248044] for ids in token_rows]
        assert all(len(ids) <= model.cfg.max_position_embeddings for ids in token_rows), "Full text exceeds model context"
        with torch.inference_mode():
            value = model(token_rows)
            score = model.plain(value)
        stop = begin + len(batch_ids)
        features[begin:stop] = value.float().cpu().numpy()
        plain[begin:stop] = score.float().cpu().numpy()
        del value, score
    args.output.parent.mkdir(parents=True, exist_ok=True)
    np.savez(args.output, row_index=indices, features=features, plain_logits=plain)
    print(json.dumps(dict(worker=args.worker_id, rows=len(indices), labels=metadata["labels"],
                          load_report=model.load_report)), flush=True)


def group_codes(args, frame):
    assert (args.groups is None) != (args.prompt_column is None), "Provide exactly one group source"
    if args.groups is not None:
        codes = np.load(args.groups, allow_pickle=False)
        assert codes.ndim == 1 and len(codes) == len(frame) and np.issubdtype(codes.dtype, np.integer)
        assert np.all(codes >= 0), "Explicit group codes must be nonnegative"
        return codes
    assert args.prompt_column in frame, "Requested prompt column is missing"
    assert frame[args.prompt_column].notna().all(), "Missing prompt group is ambiguous"
    return pd.factorize(frame[args.prompt_column], sort=False)[0]


def combine(args, outputs):
    from safetensors import safe_open

    frame, _ = read_rows(args.input, args.text_column)
    n = len(frame)
    features = np.empty((n, 40960), dtype=np.float32)
    plain = np.empty((n, 18), dtype=np.float32)
    seen = np.zeros(n, dtype=bool)
    for path in outputs:
        with np.load(path, allow_pickle=False) as item:
            ids = item["row_index"]
            assert ids.ndim == 1 and np.issubdtype(ids.dtype, np.integer)
            assert len(np.unique(ids)) == len(ids)
            assert np.all((0 <= ids) & (ids < n)) and not np.any(seen[ids])
            assert item["features"].shape == (len(ids), 40960)
            assert item["plain_logits"].shape == (len(ids), 18)
            features[ids] = item["features"]
            plain[ids] = item["plain_logits"]
            seen[ids] = True
    assert seen.all() and np.isfinite(features).all() and np.isfinite(plain).all()
    metadata = json.loads(args.metadata.read_text(encoding="utf-8"))
    with safe_open(str(args.final), framework="pt", device="cpu") as weights:
        kernel = weights.get_tensor("centered.weight").numpy().astype(np.float32)
        bias = weights.get_tensor("centered.bias").numpy().astype(np.float32)
    assert kernel.shape == (18, 40960) and bias.shape == (18,)
    groups = group_codes(args, frame)
    centered = plain.copy()
    valid_centered = np.zeros(n, dtype=bool)
    for group in np.unique(groups):
        ids = np.flatnonzero(groups == group)
        if len(ids) == 1:
            continue  # No contrast: downstream uses plain logits and mask=False.
        centered[ids] = (features[ids] - features[ids].mean(0, keepdims=True)) @ kernel.T + bias
        valid_centered[ids] = True
    assert np.isfinite(centered).all() and len(groups) == n
    args.output.parent.mkdir(parents=True, exist_ok=True)
    np.savez(args.output, row_index=np.arange(n, dtype=np.int64), features=features,
             plain_logits=plain, centered_logits=centered, valid_centered=valid_centered,
             group_code=groups, labels=np.asarray(metadata["labels"]))


def parent(args):
    _, texts = read_rows(args.input, args.text_column)
    worker_count = min(2, len(texts))
    outputs = [args.work_dir / f"gsq-worker-{i}.npz" for i in range(worker_count)]
    args.work_dir.mkdir(parents=True, exist_ok=True)
    children = []
    for device in range(worker_count):
        command = [sys.executable, str(Path(__file__).resolve()), "--worker-id", str(device),
                   "--base", str(args.base), "--final", str(args.final), "--metadata", str(args.metadata),
                   "--complete", str(args.complete), "--input", str(args.input), "--output", str(outputs[device]),
                   "--text-column", args.text_column,
                   "--batch-size", str(args.batch_size)]
        env = os.environ.copy()
        env.update(CUDA_VISIBLE_DEVICES=str(device), HF_HUB_OFFLINE="1", TRANSFORMERS_OFFLINE="1",
                   TOKENIZERS_PARALLELISM="false")
        children.append(subprocess.Popen(command, env=env))
    codes = [child.wait() for child in children]
    if any(code != 0 for code in codes):
        raise RuntimeError(f"GSQ workers failed: exit codes {codes}; outputs were not combined")
    combine(args, outputs)


def main():
    p = argparse.ArgumentParser(description=__doc__)
    for name in ("base", "final", "metadata", "complete", "input", "output"):
        p.add_argument("--" + name, type=Path, required=True)
    p.add_argument("--work-dir", type=Path)
    p.add_argument("--text-column", default="text")
    p.add_argument("--prompt-column")
    p.add_argument("--groups", type=Path)
    p.add_argument("--batch-size", type=int, default=1)
    p.add_argument("--worker-id", type=int, choices=(0, 1))
    args = p.parse_args()
    assert args.batch_size >= 1
    if args.worker_id is None:
        assert args.work_dir is not None and os.environ.get("CUDA_VISIBLE_DEVICES") in (None, "", "0,1")
        assert (args.groups is None) != (args.prompt_column is None), "Provide --groups or --prompt-column"
        parent(args)
    else:
        worker(args)


if __name__ == "__main__":
    main()


## GPU stages and per-fold fusion

The core, complementary and ATLAS models run in isolated GPU processes. The native embedding backbone is loaded once after the parent stages; its five independent task states are applied successively. Native geometric fusion precedes per-fold balancing and typography; the final result averages five corrected probability matrices.


In [ ]:
%%writefile /kaggle/temp/ntl-gsq-v27-runtime/stage_dataset_weights.py
"""Sequential, explicit-file Dataset staging for local random reads."""
import json
import shutil
import tempfile
import time
from contextlib import contextmanager
from pathlib import Path

NF4_FILES = {'LICENSE': 11544, 'README.md': 27108, 'chat_template.jinja': 8952, 'config.json': 4222, 'generation_config.json': 202, 'merges.txt': 3353259, 'model-00001-of-00007.safetensors': 2542796928, 'model-00002-of-00007.safetensors': 3967729339, 'model-00003-of-00007.safetensors': 3973498949, 'model-00004-of-00007.safetensors': 3992447326, 'model-00005-of-00007.safetensors': 4000148996, 'model-00006-of-00007.safetensors': 98537568, 'model-00007-of-00007.safetensors': 849400424, 'model.safetensors.index.json': 396308, 'tokenizer.json': 12809320, 'tokenizer_config.json': 17928, 'vocab.json': 6722759}
ULTRA_FOLD_FILES = {0: {'config.json': 2739, 'configuration_ultrabert.py': 2209, 'model.safetensors': 755137636, 'modeling_ultrabert.py': 30581, 'tokenizer.json': 3523731, 'tokenizer_config.json': 814}, 1: {'config.json': 2739, 'configuration_ultrabert.py': 2209, 'model.safetensors': 755137636, 'modeling_ultrabert.py': 30581, 'tokenizer.json': 3523731, 'tokenizer_config.json': 814}, 2: {'config.json': 2739, 'configuration_ultrabert.py': 2209, 'model.safetensors': 755137636, 'modeling_ultrabert.py': 30581, 'tokenizer.json': 3523731, 'tokenizer_config.json': 814}, 3: {'config.json': 2739, 'configuration_ultrabert.py': 2209, 'model.safetensors': 755137636, 'modeling_ultrabert.py': 30581, 'tokenizer.json': 3523731, 'tokenizer_config.json': 814}, 4: {'config.json': 2739, 'configuration_ultrabert.py': 2209, 'model.safetensors': 755137636, 'modeling_ultrabert.py': 30581, 'tokenizer.json': 3523731, 'tokenizer_config.json': 814}}

@contextmanager
def staged_dataset_tree(source, files, *, label, input_root='/kaggle/input', temp_root='/tmp'):
    source = Path(source)
    input_root = Path(input_root)
    temp_root = Path(temp_root)
    assert source.is_dir() and source.resolve().is_relative_to(input_root.resolve())
    assert label and label.replace('-', '').replace('_', '').isalnum()
    assert files and all('/' not in name and '..' not in name and size > 0
                         for name, size in files.items())
    total = sum(files.values())
    for name, size in files.items():
        item = source / name
        assert item.is_file() and item.stat().st_size == size, (label, name, size)
    free = shutil.disk_usage(temp_root).free
    assert free > total + 2 * 2**30, ('Insufficient /tmp disk for explicit staging', label, total, free)
    destination = Path(tempfile.mkdtemp(prefix='ntl-stage-' + label + '-', dir=temp_root))
    assert destination.resolve().is_relative_to(temp_root.resolve())
    start = time.monotonic()
    try:
        for name, size in files.items():
            target = destination / name
            with (source / name).open('rb') as reader, target.open('wb') as writer:
                while block := reader.read(16 * 1024 * 1024):
                    writer.write(block)
            assert target.stat().st_size == size
            print(json.dumps({'stage': label, 'file': name, 'bytes': size,
                              'elapsed_seconds': round(time.monotonic() - start, 2)}), flush=True)
        yield destination
    finally:
        assert destination.resolve().is_relative_to(temp_root.resolve())
        shutil.rmtree(destination)
        print(json.dumps({'stage': label, 'event': 'STAGING_CLEANED'}), flush=True)


In [ ]:
%%writefile /kaggle/temp/ntl-gsq-v27-runtime/ensemble.py
"""Sequential GPU stages and one final CPU blend; no model is loaded here."""
import argparse
import json
import os
from pathlib import Path
import subprocess
import sys

import numpy as np
import pandas as pd
from scipy.special import softmax

from core_math import LABELS, topic_groups, finalize
from stage_dataset_weights import staged_dataset_tree


def run(root, logs, stage, script, arguments, gpus='', extra_env=None):
    env = dict(os.environ, HF_HUB_OFFLINE='1', TRANSFORMERS_OFFLINE='1',
               TOKENIZERS_PARALLELISM='false', CUDA_VISIBLE_DEVICES=gpus)
    env.update(extra_env or {})
    with (logs / (stage + '.log')).open('w', encoding='utf-8', buffering=1) as stream:
        code = subprocess.call([sys.executable, '-u', str(root / script), *map(str, arguments)],
                               stdout=stream, stderr=subprocess.STDOUT, cwd=root, env=env)
    if code:
        raise RuntimeError(f'{stage} failed with exit code {code}; see its stage log')


def aligned(path, ids, field):
    with np.load(path, allow_pickle=False) as data:
        saved_ids, labels = data['ids'].astype(str).tolist(), data['labels'].astype(str).tolist()
        assert len(saved_ids) == len(set(saved_ids)) == len(ids) and set(saved_ids) == set(ids)
        assert len(labels) == len(set(labels)) == 18 and set(labels) == set(LABELS)
        index = {v: i for i, v in enumerate(saved_ids)}
        value = data[field][[index[i] for i in ids]][:, [labels.index(l) for l in LABELS]]
    assert value.shape == (len(ids), 18) and np.isfinite(value).all()
    return value


def blend_all_rows(core, complementary, atlas, atlas_rows, gsq, gsq_share):
    """The three probability mixtures occur before any class balancing."""
    prior = .85 * softmax(core, axis=1) + .15 * softmax(complementary, axis=1)
    prior /= prior.sum(1, keepdims=True)
    prior[atlas_rows] = .9 * prior[atlas_rows] + .1 * atlas
    prior /= prior.sum(1, keepdims=True)
    mixed = prior if gsq_share == 0 else (1.0 - gsq_share) * prior + gsq_share * gsq
    assert np.isfinite(mixed).all() and np.allclose(mixed.sum(1), 1)
    return mixed


def main():
    parser = argparse.ArgumentParser()
    parser.add_argument('--config', type=Path, required=True)
    parser.add_argument('--out', type=Path, required=True)
    parser.add_argument('--work', type=Path, default=Path('/kaggle/temp/ntl-gsq-v27-work'))
    args = parser.parse_args()
    config = json.loads(args.config.read_text())
    assert config['RELEASE_READY'] is True and all(config['gates'].values()), 'Release binding gate is closed'
    assert config['complementary_share'] == .15, 'Fixed complementary share differs'
    assert config['gsq_share'] == 0.12, 'Unexpected GSQ share'
    assert config['scored_reference_script_version'] == 363008892
    assert config['token_attention_q35_enabled'] is False
    assert all(spec['name'] != 'modernbert' for spec in config['models'])
    root = Path(__file__).resolve().parent
    work = args.work
    work.mkdir(parents=True, exist_ok=True)
    logs = Path('/kaggle/working/inference-logs')
    logs.mkdir(parents=True, exist_ok=True)
    frame = pd.read_csv(config['paths']['test_csv'], dtype={'id': str})
    assert frame.columns.tolist() == ['id', 'text'] and len(frame) > 0 and frame.id.is_unique
    frame['text'] = frame.text.fillna('').astype(str)
    ids = frame.id.tolist()
    query = work / 'query.csv'
    frame.to_csv(query, index=False)
    groups = topic_groups(frame.text.tolist())
    group_path = work / 'Gte.npy'
    np.save(group_path, groups, allow_pickle=False)
    features = work / 'core-features'
    features.mkdir(exist_ok=True)
    # Each child exits and releases its complete GPU context before the next stage.
    # ---- ours, V1: the two 8B core members (q8v5, ll8v5) run at the same time, one per T4 (each fits one GPU;
    # with both visible, device_map 'auto' split each over the two and one GPU waited); deb follows q8v5 on GPU 0.
    # Same computation per text. A member that fails on one GPU is run again the original way, on both.
    import threading
    names = [spec['name'] for spec in config['models']]
    lanes = {'0': [names.index('q8v5'), names.index('deb')], '1': [names.index('ll8v5')]}
    one_gpu = work / 'config-one-gpu.json'
    one_gpu.write_text(json.dumps(dict(config, device_max_memory={'0': '14GiB'})))
    def member(i, gpus, cfg):
        run(root, logs, 'core-' + names[i], 'core_worker.py',
            ['--config', cfg, '--query', query, '--member', i, '--out', features / names[i]], gpus)
    failed = []
    def lane(gpus, members):
        for i in members:
            try:
                member(i, gpus, one_gpu)
            except Exception as e:
                failed.append(i)
                print(f'core-{names[i]} failed on GPU {gpus} ({e}); it runs again on both GPUs', flush=True)
    for i in range(len(names)):
        if all(i not in m for m in lanes.values()):
            member(i, '0,1', args.config)
    threads = [threading.Thread(target=lane, args=item) for item in lanes.items()]
    for t in threads:
        t.start()
    for t in threads:
        t.join()
    for i in sorted(failed):
        member(i, '0,1', args.config)
    (logs / 'core-parallel.json').write_text(json.dumps({'lanes': {g: [names[i] for i in m] for g, m in lanes.items()},
                                                         'rerun_on_both': [names[i] for i in failed]}))
    run(root, logs, 'core-readouts-cpu', 'core_readout.py',
        ['--config', args.config, '--query', query, '--groups', group_path,
         '--features', features, '--out', work / 'core.npz'])
    core = aligned(work / 'core.npz', ids, 'logits')
    run(root, logs, 'complementary', 'legacy/comp_cached.py', [], '0,1', {
        'NTL_QUERY': str(query), 'NTL_PREFIT_ROOT': config['paths']['legacy_prefit'],
        'NTL_STYLE_ROOT': config['paths']['complement_style'],
        'NTL660_PREFIT_ROOT': config['paths']['ntl660_prefit'],
        'NTL_COMP_OUT': str(work / 'complementary.npy'),
        'PYTHONPATH': str(root),
    })
    comp_meta = json.loads((work / 'complementary.npy.json').read_text())
    assert comp_meta['ids'] == ids and comp_meta['labels'] == LABELS
    comp = np.load(work / 'complementary.npy', allow_pickle=False)
    assert comp.shape == core.shape and np.isfinite(comp).all()
    prior = .85 * softmax(core, axis=1) + .15 * softmax(comp, axis=1)
    prior /= prior.sum(1, keepdims=True)
    order = np.sort(prior, axis=1)
    atlas_rows = np.flatnonzero(order[:, -1] - order[:, -2] < .3) if len(ids) >= 50 else np.arange(len(ids))
    atlas = np.empty((0, 18))
    if len(atlas_rows):
        atlas_query = work / 'atlas-query.csv'
        frame.iloc[atlas_rows].to_csv(atlas_query, index=False)
        run(root, logs, 'atlas', 'legacy/atlas_worker.py', ['--query', atlas_query, '--out', work / 'atlas.npz'], '0,1')
        atlas = aligned(work / 'atlas.npz', [ids[i] for i in atlas_rows], 'probabilities')
        assert (atlas >= 0).all() and np.allclose(atlas.sum(1), 1, atol=1e-4)
    # Load new-transformers dependencies only in the own GSQ process tree.
    target = work / 'gsq-python'
    subprocess.run([sys.executable, '-m', 'pip', 'install', '--no-index', '--no-deps', '--disable-pip-version-check',
                    '--target', str(target), *config['paths']['qwen_wheels']], check=True)
    if config['gsq_share'] > 0:
        parquet = work / 'query.parquet'
        frame.to_parquet(parquet, index=False)
        paths = config['paths']
        # One sequential /tmp copy is shared read-only by the two isolated GPU workers.
        with staged_dataset_tree(paths['gsq_base'], config['gsq_base_inventory'], label='gsq3-base') as base:
            run(root, logs, 'gsq-two-t4', 'gsq/infer.py',
                ['--base', base, '--final', paths['gsq_final'], '--metadata', paths['gsq_metadata'],
                 '--complete', paths['gsq_complete'], '--input', parquet, '--groups', group_path,
                 '--output', work / 'gsq.npz', '--work-dir', work / 'gsq-workers', '--batch-size', 1],
                '', dict(PYTHONPATH=str(target)))
        with np.load(work / 'gsq.npz', allow_pickle=False) as own:
            assert np.array_equal(own['row_index'], np.arange(len(frame)))
            assert np.array_equal(own['group_code'], groups)
            labels = own['labels'].tolist()
            assert len(labels) == len(set(labels)) == 18 and set(labels) == set(LABELS)
            assert config['gsq_head'] == 'plain', 'A centered mixture needs separate evidence and an explicit policy'
            gsq = softmax(own['plain_logits'][:, [labels.index(l) for l in LABELS]], axis=1)
            assert np.isfinite(gsq).all() and np.allclose(gsq.sum(1), 1)
    else:
        gsq = np.empty((0,18),dtype=np.float64)
    mixed = blend_all_rows(core, comp, atlas, atlas_rows, gsq, config['gsq_share'])
    native = config['native_embedding']
    run(root, logs, 'native-q8-fivefold', 'native_embedding_worker.py',
        ['--base',native['base'],'--assets',native['assets'],'--query',query,
         '--out',work/'native-q8.npz','--work',work], '0,1', dict(PYTHONPATH=str(target)))
    np.savez(work / 'pre-balance.npz', ids=np.asarray(ids), labels=np.asarray(LABELS),
             core=core, complementary=comp, atlas_rows=atlas_rows, atlas=atlas, gsq=gsq, probabilities=mixed)
    official = pd.read_csv(config['paths']['train_csv'], dtype={'id': str})
    assert len(official) == 2699 and official.id.is_unique
    official_texts = official.text.fillna('').astype(str).tolist()
    official_labels = official.model.map({label: i for i, label in enumerate(LABELS)})
    assert official_labels.notna().all()
    from native_fusion import finish_native
    logits = finish_native(mixed,work/'native-q8.npz',native['assets'],ids,LABELS,groups,
        frame.text.values,official_texts,official_labels.to_numpy(dtype=np.int64),native['share'])
    np.savez(work / 'final-logits.npz', ids=np.asarray(ids), labels=np.asarray(LABELS), logits=logits)
    output = pd.DataFrame({'id': ids, 'model': np.asarray(LABELS)[logits.argmax(1)]})
    assert len(output) == len(frame) and output.id.is_unique and output.model.isin(LABELS).all()
    output.to_parquet(args.out, index=False)
    # Inspect the final on-disk writer, after all gates, balances and typography.
    saved = pd.read_parquet(args.out)
    assert saved.equals(output) and saved.id.tolist() == ids
    (logs / 'execution-contract.json').write_text(json.dumps({
        'mode': config['execution_mode'], 'rows': len(frame),
        'experiment': 'native40_q03', 'open_pool_dataset_version': 13, 'svc4_weight': 12.0,
        'stages': [spec['name'] for spec in config['models']] + ['core-readouts-cpu', 'complementary', 'atlas'] + (['gsq-two-t4'] if config['gsq_share']>0 else []) + ['native-q8-fivefold'],
        'gsq_share': config['gsq_share'], 'complementary_share': .15, 'native_embedding_share':native['share'],
        'native_source_script_version_id':365653660, 'native_folds':5, 'native_full_document_windows':True,
        'q35_token_attention_enabled': False, 'deberta_token_attention_enabled': True,
        'sinkhorn_strength': .30, 'sinkhorn_target_mix': 0.5,
        'typography_markers': 37, 'final_writer': 'ensemble.py:output.to_parquet',
    }, indent=2))


if __name__ == '__main__':
    main()


## Input preparation and execution

Large immutable backbone shards are copied to local temporary storage before repeated inference. Each GPU child exits before the next stage, releasing its model and CUDA context. Fixed fitted states are restored from the shared artifact dataset. The exact public sample preview skips all model forwards. The final writer checks passage IDs, class labels and the saved parquet.


In [ ]:
import shutil
ULTRA_ROOT = Path('/kaggle/input/datasets/pjmathematician/ultrabert-380m-ntl')
ULTRA_FILES = ['config.json', 'model.safetensors', 'tokenizer.json', 'tokenizer_config.json']
for fold in range(5):
    original_fold = ULTRA_ROOT / f'fold{fold}'
    local_fold = RUNTIME / 'legacy/inline_support/ultra' / f'fold{fold}'
    local_fold.mkdir(parents=True, exist_ok=True)
    for name in ULTRA_FILES:
        original_file = original_fold / name
        assert original_file.is_file(), original_file
        local_file = local_fold / name
        if local_file.is_symlink():
            assert local_file.resolve() == original_file.resolve(), local_file
        else:
            assert not local_file.exists(), local_file
            local_file.symlink_to(original_file)
    for name in ['configuration_ultrabert.py', 'modeling_ultrabert.py']:
        shutil.copyfile(RUNTIME / 'legacy/inline_support/ultra' / name, local_fold / name)


In [ ]:
NUMERIC_CACHE_PARTS = {'input_root': '/kaggle/input/datasets/lightningv08/ntl-olmo2-native24-final-inference-assets', 'parts': [{'name': 'ntl660-prefit.part000.bin', 'bytes': 25165824}, {'name': 'ntl660-prefit.part001.bin', 'bytes': 25165824}, {'name': 'ntl660-prefit.part002.bin', 'bytes': 25165824}, {'name': 'ntl660-prefit.part003.bin', 'bytes': 25165824}, {'name': 'ntl660-prefit.part004.bin', 'bytes': 25165824}, {'name': 'ntl660-prefit.part005.bin', 'bytes': 25165824}, {'name': 'ntl660-prefit.part006.bin', 'bytes': 25165824}, {'name': 'ntl660-prefit.part007.bin', 'bytes': 25165824}, {'name': 'ntl660-prefit.part008.bin', 'bytes': 25165824}, {'name': 'ntl660-prefit.part009.bin', 'bytes': 25165824}, {'name': 'ntl660-prefit.part010.bin', 'bytes': 25165824}, {'name': 'ntl660-prefit.part011.bin', 'bytes': 963274}], 'archive_bytes': 277787338, 'cache_path': '/kaggle/temp/ntl660-prefit'}
CACHE_LABELS = ['claude-fable-5', 'claude-opus-4-6', 'claude-opus-4-8', 'claude-opus-5', 'claude-sonnet-5', 'gemini-3.1-pro-preview', 'gemini-3.6-flash', 'gemini-3.7-flash', 'gemini-3.8-flash', 'glm-5.3', 'glm-5.3-flash', 'gpt-5.5', 'gpt-5.6-luna', 'gpt-5.6-sol', 'gpt-5.6-terra', 'gpt-6-astra', 'kimi-k3', 'qwen3.8-max']
if not preview:
    import shutil
    import stat
    import zipfile
    from pathlib import PurePosixPath
    parts_root = Path(NUMERIC_CACHE_PARTS['input_root'])
    cache_target = Path(NUMERIC_CACHE_PARTS['cache_path'])
    assembled_archive = Path('/kaggle/temp/ntl660-prefit.zip')
    assert cache_target == Path('/kaggle/temp/ntl660-prefit')
    assert CONFIG['paths']['ntl660_prefit'] == str(cache_target)
    assert not cache_target.exists(), 'Refusing to overwrite a prior materialized numeric cache'
    assert [part['name'] for part in NUMERIC_CACHE_PARTS['parts']] == [f'ntl660-prefit.part{i:03d}.bin' for i in range(12)]
    assembled_bytes = 0
    created_cache = False
    try:
        with assembled_archive.open('wb') as joined:
            for part in NUMERIC_CACHE_PARTS['parts']:
                source_part = parts_root / part['name']
                assert source_part.is_file() and source_part.stat().st_size == part['bytes']
                copied_bytes = 0
                with source_part.open('rb') as source:
                    while block := source.read(16 * 1024 * 1024):
                        joined.write(block)
                        copied_bytes += len(block)
                assert copied_bytes == part['bytes']
                assembled_bytes += copied_bytes
        assert assembled_bytes == assembled_archive.stat().st_size == NUMERIC_CACHE_PARTS['archive_bytes'] == 277787338
        with zipfile.ZipFile(assembled_archive) as archive:
            entries = archive.infolist()
            assert len(entries) == 51 and len({entry.filename for entry in entries}) == 51
            for entry in entries:
                relative = PurePosixPath(entry.filename)
                assert entry.filename and '\\' not in entry.filename and not relative.is_absolute()
                assert '..' not in relative.parts and '.' not in relative.parts
                assert relative.as_posix() == entry.filename and not entry.is_dir()
                assert not stat.S_ISLNK(entry.external_attr >> 16)
                assert entry.filename == 'COMPLETE' or relative.suffix in {'.json', '.npz', '.joblib', '.md'}
            cache_manifest = json.loads(archive.read('manifest.json'))
            assert cache_manifest['status'] == 'COMPLETE'
            assert cache_manifest['source']['script_version_id'] == 363008892
            assert cache_manifest['labels'] == CACHE_LABELS
            assert cache_manifest['core']['svc4']['rows'] == 123828
            assert cache_manifest['core']['matched_clean_rows_after_dedup'] == 24443
            assert len(cache_manifest['artifacts']) == 48
            declared_files = {item['path']: item['bytes'] for item in cache_manifest['artifacts']}
            assert len(declared_files) == 48
            assert set(declared_files) | {'manifest.json', 'COMPLETE', 'README.md'} == {entry.filename for entry in entries}
            for entry in entries:
                if entry.filename in declared_files:
                    assert entry.file_size == declared_files[entry.filename]
            assert archive.read('COMPLETE') == b'ntl660-numeric-v1\n'
            cache_target.mkdir(parents=True, exist_ok=False)
            created_cache = True
            archive.extractall(cache_target)
        assert (cache_target/'COMPLETE').is_file()
        for name, size in declared_files.items():
            assert (cache_target/name).is_file() and (cache_target/name).stat().st_size == size
        print(json.dumps({'mode': 'lossless_numeric_cache_parts', 'parts': 12, 'files': 51,
                          'archive_bytes': assembled_bytes, 'manifest_status': 'COMPLETE',
                          'source_script_version_id': 363008892, 'runtime_source_in_parts': False}), flush=True)
    except Exception:
        if created_cache:
            assert cache_target == Path('/kaggle/temp/ntl660-prefit')
            shutil.rmtree(cache_target)
        raise
    finally:
        assembled_archive.unlink(missing_ok=True)


In [ ]:
if not preview:
    import zipfile
    from pathlib import PurePosixPath
    source = Path('/kaggle/input/datasets/lightningv08/ntl-olmo2-native24-final-inference-assets')
    target = Path('/kaggle/temp/ntl666-prefit')
    target.parent.mkdir(parents=True, exist_ok=True)
    assert not target.exists()
    parts = json.loads((source / 'ntl666-cache-parts.json').read_text())
    assert parts['format'] == 'ntl666-cache-parts-v1'
    assert [p['name'] for p in parts['parts']] == [f'ntl666-prefit.part{i:03d}.bin' for i in range(len(parts['parts']))]
    archive_path = Path('/kaggle/temp/ntl666-prefit.zip')
    try:
        with archive_path.open('wb') as out:
            for part in parts['parts']:
                part_path = source / part['name']
                assert part_path.stat().st_size == part['bytes']
                with part_path.open('rb') as stream:
                    while block := stream.read(16 * 1024 * 1024):
                        out.write(block)
        assert archive_path.stat().st_size == parts['archive_bytes']
        with zipfile.ZipFile(archive_path) as archive:
            entries = archive.infolist()
            assert len(entries) == parts['files'] and len({e.filename for e in entries}) == len(entries)
            for entry in entries:
                path = PurePosixPath(entry.filename)
                assert not path.is_absolute() and '..' not in path.parts and chr(92) not in entry.filename
                assert path.parts[0] == 'ntl666-prefit' and not entry.is_dir()
                assert path.suffix in {'.json', '.npz', '.joblib', ''}
            assert archive.read('ntl666-prefit/COMPLETE') == b'ntl666-v14-data-numeric-v1\n'
            provenance = json.loads(archive.read('ntl666-prefit/provenance.json'))
            assert provenance['status'] == 'COMPLETE' and provenance['pool_rows'] == 190588
            assert provenance['dataset_version'] == 13
            archive.extractall(target.parent)
    finally:
        archive_path.unlink(missing_ok=True)
    CONFIG['paths']['ntl666_prefit'] = str(target)
    CONFIG['gsq_share'] = 0.05
    (RUNTIME / 'config.json').write_text(json.dumps(CONFIG))



## Native embedding expert and calibrated fusion

Five independent trained fold states share a single staged backbone. Each native fold remains active in the final average.


In [ ]:
%%writefile /kaggle/temp/ntl-gsq-v27-runtime/native_embedding_worker.py
"""CHEN Han's trained eight-depth Qwen3-Embedding expert, with covering windows.

The full backbone is split over two T4s. Task LoRA/head values stay FP32;
the original BF16 backbone values are loaded as FP16. Only one fold's
trainable state is resident at a time; the backbone is loaded once.
"""
import argparse
import gc
import json
import os
from pathlib import Path
import shutil
import time

os.environ.setdefault('CUBLAS_WORKSPACE_CONFIG', ':4096:8')
os.environ.setdefault('TOKENIZERS_PARALLELISM', 'false')
import numpy as np
import pandas as pd
import torch
from torch import nn
from torch.nn import functional as F
from safetensors.torch import load_file
from transformers import AutoModel, AutoTokenizer

INSTRUCTION = 'Instruct: Identify the language model that generated the text based on its writing style.\nQuery: '
DEPTHS = (3, 6, 9, 12, 18, 24, 30, 36)
SUFFIXES = ('q_proj', 'k_proj', 'v_proj', 'o_proj', 'gate_proj', 'up_proj', 'down_proj')


class TaskLoRA(nn.Module):
    def __init__(self, base):
        super().__init__()
        self.base_layer = base
        device = base.weight.device
        self.a = nn.Linear(base.in_features, 16, bias=False, device=device, dtype=torch.float32)
        self.b = nn.Linear(16, base.out_features, bias=False, device=device, dtype=torch.float32)

    def forward(self, x):
        result = self.base_layer(x)
        delta = self.b(self.a(x.float())) * 2.0
        return (result + delta).to(result.dtype)


class NativeQwen:
    def __init__(self, base, assets, work):
        torch.set_num_threads(2)
        torch.use_deterministic_algorithms(True)
        torch.backends.cudnn.benchmark = False
        torch.backends.cudnn.deterministic = True
        torch.backends.cudnn.allow_tf32 = False
        for key in ('allow_tf32', 'allow_fp16_reduced_precision_reduction', 'allow_bf16_reduced_precision_reduction'):
            setattr(torch.backends.cuda.matmul, key, False)
        assert torch.cuda.device_count() == 2, 'The native expert uses both T4 devices'
        self.assets = Path(assets)
        self.labels = json.loads((self.assets/'manifest.json').read_text())['classes']
        origin = Path(base)
        index = json.loads((origin/'model.safetensors.index.json').read_text())
        shards = sorted(set(index['weight_map'].values()))
        names = ['config.json','model.safetensors.index.json','tokenizer.json','tokenizer_config.json',
                 'merges.txt','vocab.json'] + shards
        # Use the original training tokenizer, not a same-named Kaggle revision.
        sources = {name:(origin/name if name in shards else self.assets/'base'/name) for name in names}
        for name in names:
            assert sources[name].is_file(), str(sources[name])
        for fold in range(5):
            assert (self.assets/f'fold{fold}/trainables.safetensors').is_file()
        self.staged = Path(work)/'native-q8-base'
        self.staged.mkdir(parents=True, exist_ok=True)
        for name in names:
            destination = self.staged/name
            if not destination.exists():
                shutil.copyfile(sources[name], destination)
            assert destination.stat().st_size == sources[name].stat().st_size
        cfg = json.loads((self.staged/'config.json').read_text())
        assert cfg['num_hidden_layers'] == 36 and cfg['hidden_size'] == 4096
        self.tokenizer = AutoTokenizer.from_pretrained(self.staged, local_files_only=True)
        self.tokenizer.padding_side = 'right'
        if self.tokenizer.pad_token_id is None:
            self.tokenizer.pad_token = self.tokenizer.eos_token
        placement = {'embed_tokens':0, 'norm':1, 'rotary_emb':1}
        placement.update({f'layers.{i}':0 if i<16 else 1 for i in range(36)})
        self.model, info = AutoModel.from_pretrained(self.staged, local_files_only=True,
            torch_dtype=torch.float16, device_map=placement, attn_implementation='sdpa',
            output_loading_info=True)
        assert not any(info.get(k) for k in ('missing_keys','mismatched_keys','error_msgs')), info
        assert not info.get('unexpected_keys'), info
        self.model.config.use_cache = False
        self.modules = {}
        for name, layer in list(self.model.named_modules()):
            if isinstance(layer, nn.Linear) and name.rsplit('.',1)[-1] in SUFFIXES:
                wrapped = TaskLoRA(layer)
                parent, attr = name.rsplit('.',1)
                setattr(self.model.get_submodule(parent), attr, wrapped)
                self.modules[name] = wrapped
        assert len(self.modules) == 252, len(self.modules)
        self.norms = nn.ModuleList([nn.LayerNorm(4096, device='cuda:0', dtype=torch.float32) for _ in DEPTHS])
        self.plain = nn.Linear(32768,18,device='cuda:0',dtype=torch.float32)
        self.centered = nn.Linear(32768,18,device='cuda:0',dtype=torch.float32)
        self.model.requires_grad_(False).eval()
        for module in (self.norms,self.plain,self.centered): module.requires_grad_(False).eval()
        self.current = None
        self.expected = {}
        for name, module in self.modules.items():
            for suffix, parameter in [('lora_A.default.weight',module.a.weight),('lora_B.default.weight',module.b.weight)]:
                self.expected['encoder.backbone.base_model.model.'+name+'.'+suffix] = parameter
        for prefix, module in [('norms',self.norms),('plain',self.plain),('centered',self.centered)]:
            self.expected.update({prefix+'.'+name:parameter for name,parameter in module.named_parameters()})
        assert len(self.expected) == 524

    @torch.inference_mode()
    def load_fold(self, fold):
        if self.current == fold: return
        state = load_file(str(self.assets/f'fold{fold}/trainables.safetensors'))
        assert set(state) == set(self.expected), (set(state)-set(self.expected), set(self.expected)-set(state))
        for name, parameter in self.expected.items():
            assert state[name].shape == parameter.shape and state[name].dtype == torch.float32, name
            assert torch.isfinite(state[name]).all(), name
            parameter.copy_(state[name].to(parameter.device))
        del state
        gc.collect()
        self.current = fold

    def windows(self, texts):
        prefix = self.tokenizer(INSTRUCTION,add_special_tokens=False)['input_ids']
        terminal = self.tokenizer('',add_special_tokens=True)['input_ids']
        assert terminal == [151643], 'The training tokenizer appends its native end-of-text token'
        capacity = 512-len(prefix)-len(terminal)
        assert capacity>0
        full = self.tokenizer([INSTRUCTION+str(t) for t in texts],truncation=False,add_special_tokens=False)['input_ids']
        windows=[]; owners=[]; weights=[]; lengths=[]
        for row,tokens in enumerate(full):
            assert len(tokens)>0
            first_capacity=512-len(terminal)
            windows.append(tokens[:first_capacity]+terminal);owners.append(row)
            weights.append(max(1,min(len(tokens),first_capacity)-len(prefix)))
            position=first_capacity
            while position<len(tokens):
                content=tokens[position:position+capacity]
                windows.append(prefix+content+terminal);owners.append(row);weights.append(len(content))
                position+=len(content)
            lengths.append(len(tokens)+len(terminal))
            assert position>=len(tokens)
        return windows,np.asarray(owners),np.asarray(weights,dtype=np.float64),lengths

    @torch.inference_mode()
    def predict(self, texts, folds=range(5), batch_size=4):
        windows,owners,weights,lengths=self.windows(texts)
        # A token sequence provides a deterministic tie-break independent of row order.
        order=sorted(range(len(windows)),key=lambda i:(len(windows[i]),windows[i]))
        outputs={'plain':[],'centered':[],'centered_bias':[]}
        timings=[]
        for fold in folds:
            self.load_fold(fold)
            began=time.monotonic()
            values={name:np.zeros((len(texts),18),dtype=np.float64) for name in ('plain','centered')}
            for start in range(0,len(order),batch_size):
                chosen=order[start:start+batch_size]
                batch=self.tokenizer.pad({'input_ids':[windows[i] for i in chosen]},padding=True,return_tensors='pt').to('cuda:0')
                output=self.model(**batch,output_hidden_states=True,use_cache=False,return_dict=True)
                assert len(output.hidden_states)==37
                positions=batch['attention_mask'].sum(1)-1
                parts=[]
                for depth,norm in zip(DEPTHS,self.norms):
                    hidden=output.last_hidden_state if depth==36 else output.hidden_states[depth]
                    pooled=hidden[torch.arange(len(chosen),device=hidden.device),positions.to(hidden.device)].float()
                    if depth==36: pooled=F.normalize(pooled,dim=-1)
                    parts.append(norm(pooled.to('cuda:0')))
                features=torch.cat(parts,dim=-1)
                for name,head in [('plain',self.plain),('centered',self.centered)]:
                    scores=head(features).cpu().numpy().astype(np.float64)
                    np.add.at(values[name],owners[chosen],scores*weights[chosen,None])
                del output,features,parts,batch
            denominator=np.bincount(owners,weights=weights,minlength=len(texts))[:,None]
            for name in values: outputs[name].append(values[name]/denominator)
            outputs['centered_bias'].append(self.centered.bias.cpu().numpy().copy())
            torch.cuda.synchronize(0);torch.cuda.synchronize(1)
            timings.append(dict(fold=int(fold),seconds=time.monotonic()-began))
            print(json.dumps(dict(completed_fold=int(fold),documents=len(texts),windows=len(windows),seconds=timings[-1]['seconds'])),flush=True)
        for key in outputs:
            outputs[key]=np.stack(outputs[key]);assert np.isfinite(outputs[key]).all(),key
        return outputs,dict(documents=len(texts),windows=len(windows),discarded_tokens=0,
            formatted_token_lengths=lengths,fold_timings=timings,
            peak_reserved=[torch.cuda.max_memory_reserved(i) for i in range(2)])


def main():
    parser=argparse.ArgumentParser()
    for name in ('base','assets','query','out','work'): parser.add_argument('--'+name,type=Path,required=True)
    args=parser.parse_args()
    frame=pd.read_csv(args.query,dtype={'id':str});assert frame.id.is_unique and len(frame)
    began=time.monotonic()
    model=NativeQwen(args.base,args.assets,args.work)
    values,report=model.predict(frame.text.fillna('').astype(str).tolist())
    np.savez(args.out,ids=frame.id.to_numpy(dtype=str),labels=np.asarray(model.labels),**values)
    report['seconds_including_load_and_staging']=time.monotonic()-began
    args.out.with_suffix('.json').write_text(json.dumps(report,indent=2))
    print(json.dumps(report),flush=True)


if __name__=='__main__': main()


In [ ]:
%%writefile /kaggle/temp/ntl-gsq-v27-runtime/native_fusion.py
"""Calibrate the five native folds and combine each with a fixed parent."""
import json
from pathlib import Path
import numpy as np
from scipy.special import log_softmax,softmax


def calibrate_native(plain,centered,bias,groups,selections):
    plain,centered,bias=(np.asarray(a,dtype=np.float64) for a in (plain,centered,bias))
    groups=np.asarray(groups)
    assert plain.shape==centered.shape and plain.ndim==3 and plain.shape[0]==5 and plain.shape[-1]==18
    assert bias.shape==(5,18) and groups.shape==(plain.shape[1],) and groups.dtype.kind in 'iu'
    assert all(np.isfinite(x).all() for x in (plain,centered,bias))
    _,inverse=np.unique(groups,return_inverse=True)
    count=np.bincount(inverse);eligible=(count[inverse]>=2)&(count[inverse]<=18)
    result=[]
    for fold,selection in enumerate(selections):
        state=selection['models']['qwen8']
        assert not state['outer_labels_used'] and state['centered_head_trained']
        assert state['plain_scale']>0 and state['centered_scale']>0
        means=np.zeros((len(count),18),dtype=np.float64)
        np.add.at(means,inverse,centered[fold]);means/=count[:,None]
        relative=centered[fold]-means[inverse]+bias[fold]
        a=log_softmax(plain[fold]*state['plain_scale'],axis=1)
        b=log_softmax(relative*state['centered_scale'],axis=1)
        weight=float(state['selected_centered_weight']);assert 0<=weight<=.5
        fraction=np.where(eligible,weight,0)[:,None]
        result.append(log_softmax((1-fraction)*a+fraction*b,axis=1))
    return np.stack(result)


def native_mixture(parent,native_logp,share):
    parent=np.asarray(parent,dtype=np.float64);native_logp=np.asarray(native_logp,dtype=np.float64)
    assert parent.ndim==2 and parent.shape[1]==18 and native_logp.shape==(5,*parent.shape)
    assert np.isfinite(parent).all() and np.isfinite(native_logp).all() and (parent>=0).all()
    assert np.allclose(parent.sum(1),1) and 0<=share<1
    if share==0:return np.broadcast_to(parent,(5,*parent.shape)).copy()
    return softmax((1-share)*np.log(np.maximum(parent,1e-15))[None]+share*native_logp,axis=-1)


def finish_native(parent,archive,assets,ids,labels,groups,texts,train_texts,train_labels,share):
    from core_math import finalize
    with np.load(archive,allow_pickle=False) as z:
        assert z['ids'].tolist()==ids and len(set(ids))==len(ids)
        saved_labels=z['labels'].tolist();assert set(saved_labels)==set(labels) and len(saved_labels)==18
        order=[saved_labels.index(label) for label in labels]
        plain=z['plain'][:,:,order];centered=z['centered'][:,:,order];bias=z['centered_bias'][:,order]
    selections=[json.loads((Path(assets)/f'calibration/fold{fold}.json').read_text()) for fold in range(5)]
    lp=calibrate_native(plain,centered,bias,groups,selections)
    mixed=native_mixture(parent,lp,share)
    # Exactly one legacy balancing/typography call per fold, then fold averaging.
    probabilities=np.mean([softmax(finalize(p,texts,groups,train_texts=train_texts,train_labels=train_labels),axis=1)
                           for p in mixed],axis=0)
    assert probabilities.shape==parent.shape and np.isfinite(probabilities).all()
    assert np.allclose(probabilities.sum(1),1)
    return np.log(np.maximum(probabilities,1e-300))


In [ ]:
# The scored ensemble reads the competition test directly.


In [ ]:
CONFIG['gsq_share'] = 0.12
CONFIG['artifact_hub_version'] = 22
CONFIG['native_embedding'] = {'base': '/kaggle/input/models/qwen-lm/qwen-3-embedding/transformers/8b/1', 'assets': '/kaggle/input/datasets/lightningv08/ntl-olmo2-native24-final-inference-assets/ntl-native-q8-multidepth-fivefold-0679', 'share': 0.36}
(RUNTIME/'config.json').write_text(json.dumps(CONFIG),encoding='utf-8')
if not preview:
    assert CONFIG['RELEASE_READY'] is True and all(CONFIG['gates'].values()), 'Release binding gate is closed'
    import subprocess
    subprocess.run([sys.executable,'-u',str(RUNTIME/'ensemble.py'),'--config',str(RUNTIME/'config.json'),
                    '--out',str(OUTPUT)],check=True)
    actual = pd.read_csv(DATA,dtype={'id':str})
    result = pd.read_parquet(OUTPUT)
    assert result.columns.tolist() == ['id','model'] and result.id.tolist() == actual.id.tolist()
    assert result.model.notna().all()
display(pd.read_parquet(OUTPUT).head())


## Likelihood under GLM-5.3-Flash and GLM-5.3

Rank passages by the sum of the ensemble probabilities for the two GLM classes. Score the same ranked subset with both GLMs, capped at 20% of passages and 130,000 estimated tokens. The fitted likelihood classifier uses the first 160 tokens per scored passage, as in training. Its log-probabilities adjust the ensemble logits on those passages. The stage report records coverage, prediction changes and model timings.


**This version (ours):** passages are ranked by the probability mass of glm-5.3, glm-5.3-flash and claude-fable-5 multiplied by one minus the top probability, so confident passages that cannot change are skipped; the cap is 30% of passages instead of 20%. The 130,000-token budget, the fitted classifier and its weight are unchanged, so the stage takes the same time.

In [ ]:
if not preview:
    command = [sys.executable, str(V1_DIR/'v1_stage.py'), '--test-csv',str(DATA),
        '--final-logits','/kaggle/temp/ntl-gsq-v27-work/final-logits.npz','--out',str(OUTPUT),
        '--report',str(V1_OUT/'report.json'),'--bin',str(V1_BIN),'--pipeline',str(V1_DIR/'run_pipeline.py'),
        '--flash-segs',json.dumps(V1_FLASH),'--glm53-segs',json.dumps(V1_GLM53),
        '--model',str(V1_DIR/'ll_model.json'),'--work','/kaggle/temp/v1-work','--outdir',str(V1_OUT),
        '--seq-batch','8','--ubatch-div','1',
        # ours: the GLM / Claude corner, close calls first, coverage cap 20% -> 30% (token budget unchanged)
        '--select-classes','glm-5.3,glm-5.3-flash,claude-fable-5','--select-mode','uncertain','--max-cover','0.30']
    subprocess.run(command,env=V1_ENV,check=True)
    final = pd.read_parquet(OUTPUT)
    assert final.columns.tolist()==['id','model'] and final.id.tolist()==pd.read_csv(DATA,dtype={'id':str}).id.tolist()
display(pd.read_parquet(OUTPUT).head())
